# Unlearning Project — Full Pre-LangChain A/B Experiment (API-Hardened v1.2)

**Protocol:** `prelangchain_ab_v1_2_api_hardened`  
**Provider-adapter revision:** `api_hardened_2026_07_22_r1`  
**Unit of analysis:** one target paragraph per request  
**Safe default:** no paid API calls  
**Few-shot examples:** excluded; deferred to the later LangChain phase

This notebook is a staged, auditable experiment for selecting the definition, prompt structure, context window, and classification workflow before introducing LangChain. OpenAI, Anthropic, and Gemini remain isolated in separate smoke-test cells, result directories, execution cells, and circuit breakers.

The central experiment compares:

- **D1 — old broad definition**;
- **D2 — current strict definition**;
- **D3 — provisional adaptive-reconfiguration definition**, designed to test whether the EPA failures arise partly from a definition that excludes durable post-failure changes in roles, protocols, coordination, and technical routines.

D3 is an exploratory research treatment, not a replacement for later human adjudication.

> **Mandatory before a real run:** restart the kernel and execute from the first import cell through the relevant provider smoke-test cell. v1.2 uses a new protocol namespace, new adapter names, an adapter-revision gate, and an OpenAI parameter firewall so stale v1/v1.1 functions cannot be used silently.

## Experimental sequence

| Phase | Research question | Conditions |
|---|---|---|
| 0 | Are the benchmark, context blocks, definitions, schemas, and provider SDKs valid? | integrity gates and smoke tests |
| 1 | Which definition and prompt structure work best? | P1 direct; P2 simple definition; P3 evidence checklist × D1/D2/D3 |
| 2 | How much context is useful? | C1 target; C2 metadata; C3 ±1; C4 ±2 on the two Phase-1 finalists |
| 3 | Is a hierarchical workflow better? | W1 one-stage joint vs W2 binary-first then target/agency |
| 4 | Are the finalists stable? | repeated runs with seeds 17, 43, 101, 211, 307 |
| 5 | How should production review work? | fixed three-provider tiered review policy |

Selection is lexicographic: schema/evidence validity and specificity constraints first, then worst-document recall, F1, MCC, balanced accuracy, stability, and finally cost/latency.

## Reproducibility principles

- Historical labels are never overwritten.
- Definition-aligned human gold columns are used when complete; otherwise results are explicitly marked provisional against the historical labels.
- Inputs, definitions, schemas, prompts, model configurations, requests, raw responses, and reports are hashed.
- A seed controls local scheduling, preprocessing, retry jitter, and provider seed fields only when verified. Temperature 0 and a seed do **not** guarantee deterministic hosted-model output.
- Every successful request is resumable by deterministic run key.
- Raw JSONL is append-only. Summary tables are regenerated from raw logs.
- Provider configuration errors stop only that provider's cell.
- A neighboring paragraph cannot independently make the target positive.
- No narrative rationale field is requested; only structured evidence and decisions are returned.

## API-hardening notes for v1.2

The real OpenAI run showed that an older or stale execution path was still transmitting `temperature`. v1.2 makes this impossible through several independent controls:

1. The OpenAI request builder never adds `temperature`, `seed`, `top_p`, `top_k`, frequency penalties, or presence penalties.
2. A runtime parameter firewall strips known forbidden sampling fields immediately before the SDK call and rejects unknown fields locally.
3. The OpenAI adapter and builder have new v1.2 function names and an adapter-revision marker.
4. Provider dispatch refuses to call an adapter whose revision does not match the active notebook.
5. A kernel-protocol guard blocks paid execution after switching from an older protocol without restarting the kernel.
6. The smoke-test signature includes the model configuration, prompts, schema, adapter code, request-builder code, parameter-preflight report, and adapter revision.
7. v1.2 writes to a new run namespace, so failed v1/v1.1 logs and smoke-test records cannot be reused.

The provider contracts are deliberately different:

- **OpenAI GPT-5.6 Terra:** low reasoning is sent; temperature and native seed are not sent.
- **Claude Haiku 4.5:** temperature 0 is sent; extended thinking and seed are not sent.
- **Gemini 3.1 Flash-Lite:** temperature 0, low thinking, and the registered native seed are sent.

The `jupyter_client` `datetime.utcnow()` deprecation warning comes from Jupyter internals, not from the experiment, and does not affect predictions. A narrowly targeted warning filter is included to prevent it from obscuring provider diagnostics.

## 0. Optional dependency installation

In [1]:
# Run once in a fresh environment, then RESTART THE KERNEL before any API call.
# The notebook feature-checks each SDK locally before its one-request smoke test.
%pip install -U pandas numpy openpyxl pyarrow pydantic scikit-learn scipy \
     statsmodels krippendorff python-dotenv tqdm nbformat openai anthropic google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 M

## 1. Imports

In [3]:
# ================================================================
# RESTORE / RESUME FROM GOOGLE DRIVE
# Run before the notebook's main parameter cell.
# ================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import json
import os

DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Unlearning_Project/"
    "prelangchain_ab_v1_2_workspace"
).resolve()

PROTOCOL = "prelangchain_ab_v1_2_api_hardened"

LATEST_CHECKPOINT = (
    DRIVE_PROJECT_ROOT
    / "backups"
    / PROTOCOL
    / "LATEST_CHECKPOINT.json"
)

if not LATEST_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Checkpoint metadata was not found: {LATEST_CHECKPOINT}"
    )

checkpoint = json.loads(
    LATEST_CHECKPOINT.read_text(encoding="utf-8")
)

input_workbook = checkpoint.get(
    "input_workbook_on_drive"
)

if not input_workbook:
    raise RuntimeError(
        "The checkpoint does not contain a Drive input-workbook path."
    )

if not Path(input_workbook).is_file():
    raise FileNotFoundError(
        f"Backed-up workbook is missing: {input_workbook}"
    )


# Make the persistent Drive workspace the notebook's PROJECT_ROOT.
os.environ["UNLEARNING_PROJECT_ROOT"] = str(
    DRIVE_PROJECT_ROOT
)

os.environ["UNLEARNING_INPUT_WORKBOOK"] = str(
    input_workbook
)

# Preserve the same replication namespace so completed run keys resume.
os.environ["UNLEARNING_REPLICATE_ID"] = str(
    checkpoint.get("run_replicate_id", "r1")
)

# Continue with the next scientific phase.
os.environ["UNLEARNING_ACTIVE_PHASES"] = "phase3"

# Use the same discovery seed unless your preregistration says otherwise.
active_seeds = checkpoint.get("active_seeds") or [17]
os.environ["UNLEARNING_SEEDS"] = ",".join(
    str(seed) for seed in active_seeds
)

os.environ["UNLEARNING_STABILITY_SEEDS"] = "17,43,101,211,307"

# Real benchmark and real providers.
os.environ["UNLEARNING_USE_SYNTHETIC"] = "false"
os.environ["UNLEARNING_USE_MOCK_PROVIDER"] = "false"

# Do not enable paid calls until setup and Phase 2 restoration are verified.
os.environ["UNLEARNING_RUN_API_CALLS"] = "true"
os.environ["UNLEARNING_API_CONFIRMATION"] = "RUN_PRELANGCHAIN_AB_V1_2"

print("Restored project root:", os.environ["UNLEARNING_PROJECT_ROOT"])
print("Restored workbook:", os.environ["UNLEARNING_INPUT_WORKBOOK"])
print("Restored replicate ID:", os.environ["UNLEARNING_REPLICATE_ID"])
print("Next active phase:", os.environ["UNLEARNING_ACTIVE_PHASES"])
print("Checkpoint time:", checkpoint["checkpoint_created_at_utc"])
print("Phase 1 selection present:",
      checkpoint["phase1_selection_file_exists"])
print("Phase 2 selection present:",
      checkpoint["phase2_selection_file_exists"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Restored project root: /content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace
Restored workbook: /content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/data/Unlearning_Codebook_Local_Context_Test_Set.xlsx
Restored replicate ID: r1
Next active phase: phase3
Checkpoint time: 2026-07-22T23:55:30.218618+00:00
Phase 1 selection present: True
Phase 2 selection present: False


In [ ]:
from pathlib import Path
import json
import pandas as pd

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUNS_ROOT:", RUNS_ROOT)
print("CONFIG_ROOT:", CONFIG_ROOT)

selection_path = Path(CONFIG_ROOT) / "phase2_selection.json"

print("\nCurrent selection file exists:", selection_path.exists())
print("Selection path:", selection_path)

# Find all Phase 2 files anywhere below the restored run directory.
phase2_files = sorted(
    path
    for path in Path(RUNS_ROOT).rglob("*")
    if path.is_file()
    and "phase2" in path.as_posix().lower()
)

print("\nPhase 2 files found:", len(phase2_files))

for path in phase2_files:
    print(
        path.relative_to(Path(RUNS_ROOT)),
        f"({path.stat().st_size:,} bytes)"
    )

In [18]:
from collections import defaultdict

latest_by_run_key = {}
source_by_run_key = {}

for path in phase2_files:
    if path.suffix.lower() != ".jsonl":
        continue

    try:
        with path.open("r", encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue

                try:
                    record = json.loads(line)
                except json.JSONDecodeError:
                    print(
                        f"WARNING: malformed JSON ignored: "
                        f"{path} line {line_number}"
                    )
                    continue

                run_key = record.get("run_key")
                status = record.get("status")

                # Ignore request logs and other JSONL records that are not results.
                if not run_key or status is None:
                    continue

                # Keep the latest record for each deterministic run key.
                latest_by_run_key[run_key] = record
                source_by_run_key[run_key] = str(path)

    except Exception as exc:
        print(
            f"WARNING: could not inspect {path}: "
            f"{type(exc).__name__}: {exc}"
        )

restored_phase2_results = pd.DataFrame(
    latest_by_run_key.values()
)

if restored_phase2_results.empty:
    print(
        "\nNo Phase 2 result records were found. "
        "Check whether PROJECT_ROOT and RUNS_ROOT point to the Drive workspace."
    )
else:
    print(
        "\nUnique Phase 2 run keys:",
        len(restored_phase2_results),
    )

    display(
        restored_phase2_results.groupby(
            ["provider", "status"],
            dropna=False,
        )
        .size()
        .rename("records")
        .reset_index()
    )

    successful_phase2 = restored_phase2_results[
        restored_phase2_results["status"].eq("ok")
    ].copy()

    print(
        "\nSuccessful unique Phase 2 calls:",
        len(successful_phase2),
    )

    group_columns = [
        column
        for column in [
            "provider",
            "condition_id",
            "context_id",
        ]
        if column in successful_phase2.columns
    ]

    if group_columns:
        display(
            successful_phase2.groupby(
                group_columns,
                dropna=False,
            )
            .agg(
                successful_calls=("run_key", "nunique"),
                unique_rows=("row_id", "nunique"),
            )
            .reset_index()
            .sort_values(group_columns)
        )


Unique Phase 2 run keys: 703


,provider,status,records
0,anthropic,ok,252
1,gemini,ok,199
2,openai,ok,252



Successful unique Phase 2 calls: 703


,provider,condition_id,context_id,successful_calls,unique_rows
0,anthropic,P1_DIRECT__C2_metadata,C2_metadata,42,42
1,anthropic,P1_DIRECT__C3_plusminus1,C3_plusminus1,42,42
2,anthropic,P1_DIRECT__C4_plusminus2,C4_plusminus2,42,42
3,anthropic,P3_D2_CURRENT__C2_metadata,C2_metadata,42,42
4,anthropic,P3_D2_CURRENT__C3_plusminus1,C3_plusminus1,42,42
5,anthropic,P3_D2_CURRENT__C4_plusminus2,C4_plusminus2,42,42
6,gemini,P1_DIRECT__C2_metadata,C2_metadata,33,33
7,gemini,P1_DIRECT__C3_plusminus1,C3_plusminus1,33,33
8,gemini,P1_DIRECT__C4_plusminus2,C4_plusminus2,34,34
9,gemini,P3_D2_CURRENT__C2_metadata,C2_metadata,33,33


In [2]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass, asdict, field, replace
from datetime import datetime, timezone
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Callable, Iterable, Literal, Mapping, Optional, Sequence, Type
import contextlib
import hashlib
import importlib.metadata
import inspect
import itertools
import json
import math
import os
import platform
import random
import re
import shutil
import sys
import time
import traceback
import unicodedata
import uuid
import warnings

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from pydantic import BaseModel, ConfigDict, Field
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    confusion_matrix, f1_score, matthews_corrcoef,
    precision_score, recall_score,
)

try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    from statsmodels.stats.multitest import multipletests
except ImportError:
    sm = smf = multipletests = None

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 220)
warnings.filterwarnings('default')
# Harmless warning emitted by some Jupyter versions; it is unrelated to API calls
# and has no effect on experiment timestamps, which are timezone-aware below.
warnings.filterwarnings(
    'ignore',
    message=r'datetime\.datetime\.utcnow\(\) is deprecated.*',
    category=DeprecationWarning,
    module=r'jupyter_client\.session',
)

## 2. Parameters

In [10]:
# PAPERMILL / NOTEBOOK PARAMETERS
PROTOCOL_VERSION = 'prelangchain_ab_v1_2_api_hardened'
ADAPTER_REVISION = 'api_hardened_2026_07_22_r1'

# Detect a notebook-protocol change in an already-running kernel. Reusing a kernel
# after v1/v1.1 is the most plausible route by which a stale OpenAI adapter can
# continue sending `temperature` even when the visible v1.2 builder does not.
_PREVIOUS_PROTOCOL_IN_KERNEL = globals().get('_UNLEARNING_ACTIVE_PROTOCOL_VERSION')
KERNEL_PROTOCOL_CHANGED = bool(
    _PREVIOUS_PROTOCOL_IN_KERNEL
    and _PREVIOUS_PROTOCOL_IN_KERNEL != PROTOCOL_VERSION
)
_UNLEARNING_ACTIVE_PROTOCOL_VERSION = PROTOCOL_VERSION
if KERNEL_PROTOCOL_CHANGED:
    warnings.warn(
        'A different Unlearning protocol was already loaded in this kernel. '
        'Restart the kernel before enabling paid API calls.',
        RuntimeWarning,
    )

PROJECT_ROOT = Path(os.getenv('UNLEARNING_PROJECT_ROOT', Path.cwd())).expanduser().resolve()
DEFAULT_WORKBOOK_NAME = 'Unlearning_Codebook_Local_Context_Test_Set.xlsx'


def resolve_input_workbook() -> Path:
    """Resolve a workbook without silently selecting an unrelated spreadsheet."""
    explicit = os.getenv('UNLEARNING_INPUT_WORKBOOK', '').strip()
    candidates: list[Path] = []
    if explicit:
        path = Path(explicit).expanduser()
        candidates.append(path if path.is_absolute() else PROJECT_ROOT / path)
    candidates.extend([
        PROJECT_ROOT / 'data' / DEFAULT_WORKBOOK_NAME,
        PROJECT_ROOT / DEFAULT_WORKBOOK_NAME,
        Path.cwd() / 'data' / DEFAULT_WORKBOOK_NAME,
        Path.cwd() / DEFAULT_WORKBOOK_NAME,
        Path('/content') / DEFAULT_WORKBOOK_NAME,
        Path('/mnt/data') / DEFAULT_WORKBOOK_NAME,
    ])
    deduplicated: list[Path] = []
    seen: set[str] = set()
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        key = str(resolved)
        if key not in seen:
            seen.add(key)
            deduplicated.append(resolved)
    for candidate in deduplicated:
        if candidate.is_file():
            return candidate
    return deduplicated[0]


INPUT_WORKBOOK = resolve_input_workbook()
INPUT_SHEET = os.getenv('UNLEARNING_INPUT_SHEET', 'GPT Test')

RUNS_ROOT = PROJECT_ROOT / 'runs' / PROTOCOL_VERSION
ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts' / PROTOCOL_VERSION
REPORTS_ROOT = PROJECT_ROOT / 'reports' / PROTOCOL_VERSION
CONFIG_ROOT = PROJECT_ROOT / 'configs' / PROTOCOL_VERSION

# Deliberately safe defaults. Enable one scientific phase at a time.
RUN_API_CALLS = os.getenv('UNLEARNING_RUN_API_CALLS', 'true').lower() == 'true'
API_RUN_CONFIRMATION = os.getenv('UNLEARNING_API_CONFIRMATION', 'RUN_PRELANGCHAIN_AB_V1_2')
REQUIRED_API_CONFIRMATION = 'RUN_PRELANGCHAIN_AB_V1_2'
USE_SYNTHETIC_DATA = os.getenv('UNLEARNING_USE_SYNTHETIC', 'false').lower() == 'true'
USE_MOCK_PROVIDER = os.getenv('UNLEARNING_USE_MOCK_PROVIDER', 'false').lower() == 'true'
ALLOW_PROVISIONAL_SELECTION = os.getenv(
    'UNLEARNING_ALLOW_PROVISIONAL_SELECTION', 'true'
).lower() == 'true'

RUN_REPLICATE_ID = os.getenv('UNLEARNING_REPLICATE_ID', 'r1')
ACTIVE_PHASES = tuple(x.strip() for x in os.getenv(
    'UNLEARNING_ACTIVE_PHASES', 'phase1'
).split(',') if x.strip())
ACTIVE_SEEDS = tuple(int(x) for x in os.getenv('UNLEARNING_SEEDS', '17').split(',') if x.strip())
STABILITY_SEEDS = tuple(int(x) for x in os.getenv(
    'UNLEARNING_STABILITY_SEEDS', '17,43,101,211,307'
).split(',') if x.strip())
ENABLED_PROVIDERS = tuple(x.strip().lower() for x in os.getenv(
    'UNLEARNING_ENABLED_PROVIDERS', 'openai,anthropic,gemini'
).split(',') if x.strip())

DISCOVERY_SEED = 17
MAX_OUTPUT_TOKENS = int(os.getenv('UNLEARNING_MAX_OUTPUT_TOKENS', '1800'))
MAX_STAGE2_OUTPUT_TOKENS = int(os.getenv('UNLEARNING_MAX_STAGE2_OUTPUT_TOKENS', '800'))
MAX_NEW_CALLS_PER_PROVIDER_CELL = int(os.getenv('UNLEARNING_MAX_NEW_CALLS_PER_PROVIDER_CELL', '350'))
MAX_RETRY_ATTEMPTS = int(os.getenv('UNLEARNING_MAX_RETRY_ATTEMPTS', '10'))
BASE_RETRY_SECONDS = float(os.getenv('UNLEARNING_BASE_RETRY_SECONDS', '15'))
REQUEST_SPACING_SECONDS = float(os.getenv('UNLEARNING_REQUEST_SPACING_SECONDS', '0.1'))
MAX_CONSECUTIVE_ERRORS = int(os.getenv('UNLEARNING_MAX_CONSECUTIVE_ERRORS', '3'))
MAX_IDENTICAL_ERROR_FINGERPRINTS = int(os.getenv('UNLEARNING_MAX_IDENTICAL_ERRORS', '2'))
STOP_PROVIDER_ON_CONFIGURATION_ERROR = True
STOP_PROVIDER_ON_QUOTA_ERROR = False
RAISE_PROVIDER_CELL_EXCEPTIONS = False
REQUIRE_PROVIDER_SMOKE_PASS = os.getenv(
    'UNLEARNING_REQUIRE_PROVIDER_SMOKE_PASS', 'true'
).lower() == 'true'
FORCE_PROVIDER_SMOKE_TESTS = os.getenv(
    'UNLEARNING_FORCE_PROVIDER_SMOKE_TESTS', 'false'
).lower() == 'true'
STRICT_PROVIDER_PARAMETER_FIREWALL = True
REQUIRE_SDK_PREFLIGHT = True

# Provider-native decoding seeds are capability-specific. OpenAI and Anthropic
# never receive a seed in this protocol; Gemini does, because its GenerationConfig
# documents a native decoding seed.
ENABLE_GEMINI_NATIVE_SEED = os.getenv(
    'UNLEARNING_ENABLE_GEMINI_NATIVE_SEED', 'true'
).lower() == 'true'

EXPECTED_BENCHMARK_ROWS = 42
EXPECTED_GOLD_YES = 32
EXPECTED_GOLD_NO = 10
TARGET_CONTEXT_MATCH_THRESHOLD = 0.97
NEAR_DUPLICATE_JACCARD_THRESHOLD = 0.90
SPECIFICITY_FLOOR = float(os.getenv('UNLEARNING_SPECIFICITY_FLOOR', '0.80'))
MIN_SCHEMA_VALID_RATE = float(os.getenv('UNLEARNING_MIN_SCHEMA_VALID_RATE', '0.99'))
MIN_EVIDENCE_VALID_RATE = float(os.getenv('UNLEARNING_MIN_EVIDENCE_VALID_RATE', '0.98'))
N_BOOTSTRAP = int(os.getenv('UNLEARNING_N_BOOTSTRAP', '2000'))
RUN_GEE_MODELS = os.getenv('UNLEARNING_RUN_GEE', 'true').lower() == 'true'

PHASE1_SELECTION_OVERRIDE = tuple(x.strip() for x in os.getenv(
    'UNLEARNING_PHASE1_SELECTION_OVERRIDE', ''
).split(',') if x.strip())
PHASE2_SELECTION_OVERRIDE = os.getenv('UNLEARNING_PHASE2_SELECTION_OVERRIDE', '').strip()
PHASE3_WORKFLOW_OVERRIDE = os.getenv('UNLEARNING_PHASE3_WORKFLOW_OVERRIDE', '').strip()

In [11]:
phase1_path = CONFIG_ROOT / "phase1_selection.json"
phase2_path = CONFIG_ROOT / "phase2_selection.json"

print("Phase 1 selection:", phase1_path.exists(), phase1_path)
print("Phase 2 selection:", phase2_path.exists(), phase2_path)

if phase1_path.exists():
    print(json.loads(phase1_path.read_text(encoding="utf-8")))

if phase2_path.exists():
    print(json.loads(phase2_path.read_text(encoding="utf-8")))

Phase 1 selection: True /content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/configs/prelangchain_ab_v1_2_api_hardened/phase1_selection.json
Phase 2 selection: False /content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/configs/prelangchain_ab_v1_2_api_hardened/phase2_selection.json
{'selected_condition_ids': ['P1_DIRECT', 'P3_D2_CURRENT'], 'selection_basis': 'preregistered constrained ranking against common-final gold', 'provisional_gold': True, 'created_at_utc': '2026-07-22T21:40:58.573684+00:00', 'phase1_ranking_sha256': '52f8a5ecf0e60ed480f97c1f9bbf06612247be7051f39555879f2fb46bf3f107'}


In [12]:
for path in [RUNS_ROOT, ARTIFACTS_ROOT, REPORTS_ROOT, CONFIG_ROOT, INPUT_WORKBOOK.parent]:
    path.mkdir(parents=True, exist_ok=True)

CONFIG_SUMMARY = {
    'protocol_version': PROTOCOL_VERSION,
    'adapter_revision': ADAPTER_REVISION,
    'project_root': str(PROJECT_ROOT),
    'input_workbook': str(INPUT_WORKBOOK),
    'input_workbook_exists': INPUT_WORKBOOK.is_file(),
    'input_sheet': INPUT_SHEET,
    'run_api_calls': RUN_API_CALLS,
    'api_confirmation_valid': API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION,
    'kernel_protocol_changed': KERNEL_PROTOCOL_CHANGED,
    'use_synthetic_data': USE_SYNTHETIC_DATA,
    'use_mock_provider': USE_MOCK_PROVIDER,
    'active_phases': ACTIVE_PHASES,
    'active_seeds': ACTIVE_SEEDS,
    'stability_seeds': STABILITY_SEEDS,
    'enabled_providers': ENABLED_PROVIDERS,
    'gemini_native_seed_enabled': ENABLE_GEMINI_NATIVE_SEED,
    'replicate_id': RUN_REPLICATE_ID,
    'smoke_pass_required': REQUIRE_PROVIDER_SMOKE_PASS,
    'sdk_preflight_required': REQUIRE_SDK_PREFLIGHT,
    'strict_parameter_firewall': STRICT_PROVIDER_PARAMETER_FIREWALL,
    'max_new_calls_per_provider_cell': MAX_NEW_CALLS_PER_PROVIDER_CELL,
    'max_consecutive_errors': MAX_CONSECUTIVE_ERRORS,
    'max_identical_errors': MAX_IDENTICAL_ERROR_FINGERPRINTS,
}
display(pd.DataFrame([CONFIG_SUMMARY]).T.rename(columns={0: 'value'}))

,value
protocol_version,prelangchain_ab_v1_2_api_hardened
adapter_revision,api_hardened_2026_07_22_r1
project_root,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace
input_workbook,/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/data/Unlearning_Codebook_Local_Context_Test_Set.xlsx
input_workbook_exists,True
input_sheet,GPT Test
run_api_calls,False
api_confirmation_valid,False
kernel_protocol_changed,False
use_synthetic_data,False


### Optional Colab secret import

This cell reads Colab secrets into environment variables without displaying secret values. Outside Colab it is a no-op. Existing environment variables take precedence.

In [13]:
SECRET_ENV_NAMES = ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'GOOGLE_API_KEY')


def load_colab_secrets() -> dict[str, bool]:
    status = {name: bool(os.getenv(name)) for name in SECRET_ENV_NAMES}
    try:
        from google.colab import userdata  # type: ignore
    except (ImportError, ModuleNotFoundError):
        return status

    for name in SECRET_ENV_NAMES:
        if os.getenv(name):
            continue
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
        status[name] = bool(os.getenv(name))
    return status


COLAB_SECRET_STATUS = load_colab_secrets()
display(pd.DataFrame([
    {'environment_variable': name, 'available': available}
    for name, available in COLAB_SECRET_STATUS.items()
]))

,environment_variable,available
0,OPENAI_API_KEY,True
1,ANTHROPIC_API_KEY,True
2,GEMINI_API_KEY,True
3,GOOGLE_API_KEY,False


## 3. Deterministic local state, hashing, and environment capture

In [14]:
def set_global_seed(seed: int) -> dict[str, Any]:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    status = {'python': seed, 'numpy': seed, 'torch': False, 'tensorflow': False}
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        with contextlib.suppress(Exception):
            torch.use_deterministic_algorithms(True)
        status['torch'] = True
    except ImportError:
        pass
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
        status['tensorflow'] = True
    except ImportError:
        pass
    return status

SEED_STATUS = set_global_seed(DISCOVERY_SEED)

def canonical_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), default=str)

def sha256_text(value: Any) -> str:
    return hashlib.sha256(str(value).encode('utf-8')).hexdigest()

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> Optional[str]:
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def stable_int_seed(*parts: Any) -> int:
    digest = hashlib.sha256(canonical_json(parts).encode('utf-8')).hexdigest()
    return int(digest[:16], 16) % (2**32 - 1)

def package_version(name: str) -> Optional[str]:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

TRACKED_PACKAGES = [
    'pandas', 'numpy', 'openpyxl', 'pyarrow', 'pydantic', 'scikit-learn',
    'scipy', 'statsmodels', 'krippendorff', 'openai', 'anthropic',
    'google-genai', 'python-dotenv', 'tqdm', 'nbformat',
]
ENVIRONMENT_SNAPSHOT = {
    'captured_at_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'executable': sys.executable,
    'platform': platform.platform(),
    'seed_status': SEED_STATUS,
    'packages': {name: package_version(name) for name in TRACKED_PACKAGES},
}
(CONFIG_ROOT / 'environment_snapshot.json').write_text(
    json.dumps(ENVIRONMENT_SNAPSHOT, indent=2, default=str), encoding='utf-8'
)
display(pd.DataFrame([
    {'package': key, 'version': value}
    for key, value in ENVIRONMENT_SNAPSHOT['packages'].items()
]))

,package,version
0,pandas,3.0.5
1,numpy,2.5.1
2,openpyxl,3.1.5
3,pyarrow,25.0.0
4,pydantic,2.13.4
5,scikit-learn,1.9.0
6,scipy,1.18.0
7,statsmodels,0.14.6
8,krippendorff,0.8.2
9,openai,2.48.0


### API execution guard

In [20]:
def assert_api_execution_allowed(provider: str) -> None:
    provider = provider.lower()
    if provider not in MODEL_CONFIGS:
        raise KeyError(f'Unknown provider: {provider}')
    if provider not in ENABLED_PROVIDERS:
        raise RuntimeError(f'{provider}: provider is disabled by UNLEARNING_ENABLED_PROVIDERS.')
    if USE_MOCK_PROVIDER:
        return
    if KERNEL_PROTOCOL_CHANGED:
        raise ProviderConfigurationError(
            'Paid execution is blocked because this kernel previously loaded a different '
            'protocol. Restart the kernel, then run v1.2 from the import cell.'
        )
    if globals().get('_UNLEARNING_ACTIVE_PROTOCOL_VERSION') != PROTOCOL_VERSION:
        raise ProviderConfigurationError(
            'Active kernel protocol does not match this notebook. Restart and rerun setup.'
        )
    if not RUN_API_CALLS:
        raise RuntimeError(f'{provider}: paid API calls are disabled.')
    if API_RUN_CONFIRMATION != REQUIRED_API_CONFIRMATION:
        raise RuntimeError(
            f'{provider}: set UNLEARNING_API_CONFIRMATION='
            f'{REQUIRED_API_CONFIRMATION!r} after reviewing the smoke test.'
        )
    env_name = {
        'openai': 'OPENAI_API_KEY',
        'anthropic': 'ANTHROPIC_API_KEY',
        'gemini': 'GEMINI_API_KEY or GOOGLE_API_KEY',
    }[provider]
    if provider == 'gemini':
        present = bool(os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY'))
    else:
        present = bool(os.getenv(env_name))
    if not present:
        raise RuntimeError(f'{provider}: missing {env_name}.')


print({
    'RUN_API_CALLS': RUN_API_CALLS,
    'confirmation_valid': API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION,
    'kernel_protocol_changed': KERNEL_PROTOCOL_CHANGED,
    'mock_provider': USE_MOCK_PROVIDER,
    'enabled_providers': ENABLED_PROVIDERS,
    'key_presence_only': {
        'openai': bool(os.getenv('OPENAI_API_KEY')),
        'anthropic': bool(os.getenv('ANTHROPIC_API_KEY')),
        'gemini': bool(os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')),
    },
})

{'RUN_API_CALLS': False, 'confirmation_valid': False, 'kernel_protocol_changed': False, 'mock_provider': False, 'enabled_providers': ('openai', 'anthropic', 'gemini'), 'key_presence_only': {'openai': True, 'anthropic': True, 'gemini': True}}


## 4. Fixed provider/model protocol

In [11]:
@dataclass(frozen=True)
class ProviderModelConfig:
    provider: Literal['openai', 'anthropic', 'gemini']
    model_id: str
    temperature_requested: float
    temperature_sent: Optional[float]
    temperature_transport: str
    reasoning_label: str
    reasoning_payload: Optional[dict[str, Any]]
    native_seed_supported: bool
    native_seed_enabled: bool
    native_seed_transport: str
    api_transport: str
    structured_output_transport: str
    api_key_env: str
    prompt_caching: str
    reproducibility_mode: str
    max_output_tokens: int = MAX_OUTPUT_TOKENS


MODEL_CONFIGS: dict[str, ProviderModelConfig] = {
    'openai': ProviderModelConfig(
        provider='openai',
        model_id='gpt-5.6-terra',
        temperature_requested=0.0,
        temperature_sent=None,
        temperature_transport=(
            'temperature is unsupported by GPT-5.6 Terra in this Responses configuration; '
            'the field is forbidden by the local request firewall and is never transmitted'
        ),
        reasoning_label='reasoning.effort=low',
        reasoning_payload={'effort': 'low'},
        native_seed_supported=False,
        native_seed_enabled=False,
        native_seed_transport='no documented Responses decoding-seed field; experiment seed is local/repeat metadata only',
        api_transport='OpenAI Responses API: client.responses.parse',
        structured_output_transport='Pydantic text_format through Responses parse helper',
        api_key_env='OPENAI_API_KEY',
        prompt_caching='hashed prompt_cache_key (63 chars); exact static prefix first',
        reproducibility_mode='fixed prompt/schema/model configuration + raw logs + repeated identical calls',
    ),
    'anthropic': ProviderModelConfig(
        provider='anthropic',
        model_id='claude-haiku-4-5-20251001',
        temperature_requested=0.0,
        temperature_sent=0.0,
        temperature_transport='temperature=0.0 sent; extended thinking omitted',
        reasoning_label='lowest setting: extended thinking disabled',
        reasoning_payload=None,
        native_seed_supported=False,
        native_seed_enabled=False,
        native_seed_transport='Claude Messages API exposes no decoding seed in this protocol',
        api_transport='Anthropic Messages API: client.messages.parse',
        structured_output_transport='Pydantic output_format convenience parameter',
        api_key_env='ANTHROPIC_API_KEY',
        prompt_caching='explicit ephemeral cache breakpoint on the static system block',
        reproducibility_mode='pinned dated model ID + temperature 0 + raw logs + repeated identical calls',
    ),
    'gemini': ProviderModelConfig(
        provider='gemini',
        model_id='gemini-3.1-flash-lite',
        temperature_requested=0.0,
        temperature_sent=0.0,
        temperature_transport='temperature=0.0 sent',
        reasoning_label='thinking_level=low',
        reasoning_payload={'thinking_level': 'low'},
        native_seed_supported=True,
        native_seed_enabled=ENABLE_GEMINI_NATIVE_SEED,
        native_seed_transport='GenerationConfig.seed receives the registered experiment seed',
        api_transport='Google Gen AI SDK: models.generate_content',
        structured_output_transport='response_mime_type=application/json + response_json_schema',
        api_key_env='GEMINI_API_KEY or GOOGLE_API_KEY',
        prompt_caching='implicit prefix caching; cached token usage logged when reported',
        reproducibility_mode='stable model ID + temperature 0 + native decoding seed + raw logs + repeats',
    ),
}


def model_config_hash(config: ProviderModelConfig) -> str:
    return sha256_text(canonical_json(asdict(config)))


MODEL_PROTOCOL = pd.DataFrame([
    {**asdict(config), 'config_sha256': model_config_hash(config)}
    for config in MODEL_CONFIGS.values()
])
MODEL_PROTOCOL.to_csv(CONFIG_ROOT / 'model_protocol.csv', index=False)
display(MODEL_PROTOCOL)

,provider,model_id,temperature_requested,temperature_sent,temperature_transport,reasoning_label,reasoning_payload,native_seed_supported,native_seed_enabled,native_seed_transport,api_transport,structured_output_transport,api_key_env,prompt_caching,reproducibility_mode,max_output_tokens,config_sha256
0,openai,gpt-5.6-terra,0.0,NaN,temperature is unsupported by GPT-5.6 Terra in this Responses configuration; the field is forbidden by the local request firewall and is never transmitted,reasoning.effort=low,{'effort': 'low'},False,False,no documented Responses decoding-seed field; experiment seed is local/repeat metadata only,OpenAI Responses API: client.responses.parse,Pydantic text_format through Responses parse helper,OPENAI_API_KEY,hashed prompt_cache_key (63 chars); exact static prefix first,fixed prompt/schema/model configuration + raw logs + repeated identical calls,1800,9f79900c8bed0b7477c913b17364bb219bc9284c1c9b1d9099c7ce1f2cc084bc
1,anthropic,claude-haiku-4-5-20251001,0.0,0.0,temperature=0.0 sent; extended thinking omitted,lowest setting: extended thinking disabled,None,False,False,Claude Messages API exposes no decoding seed in this protocol,Anthropic Messages API: client.messages.parse,Pydantic output_format convenience parameter,ANTHROPIC_API_KEY,explicit ephemeral cache breakpoint on the static system block,pinned dated model ID + temperature 0 + raw logs + repeated identical calls,1800,8090d745697d7da125cd149fd7001c7978c6b4cafdc2156f22e73eec3857c1a2
2,gemini,gemini-3.1-flash-lite,0.0,0.0,temperature=0.0 sent,thinking_level=low,{'thinking_level': 'low'},True,True,GenerationConfig.seed receives the registered experiment seed,Google Gen AI SDK: models.generate_content,response_mime_type=application/json + response_json_schema,GEMINI_API_KEY or GOOGLE_API_KEY,implicit prefix caching; cached token usage logged when reported,stable model ID + temperature 0 + native decoding seed + raw logs + repeats,1800,1d141db9a3fbe2d79e7d076b6e712cc6af3f7d7478e85098f1aba9ff08c4deaa


### Reproducibility and provider parity

The providers do not expose identical decoding controls, so “same settings” means **pre-registered provider-specific contracts**, not forcing unsupported parameters into each API.

- OpenAI receives `reasoning.effort="low"`; `temperature` and `seed` are forbidden at transport time.
- Claude receives `temperature=0`; extended thinking and seed are omitted.
- Gemini receives `temperature=0`, `thinking_level="low"`, and the registered decoding seed.

For OpenAI and Claude, the integer seed controls local scheduling, retry jitter, run IDs, analysis resampling, and repeat registration—it is not misrepresented as a provider decoding seed. Reproducibility is assessed empirically through repeated identical calls and complete provenance rather than claimed from temperature alone.

In [12]:
API_DOCUMENTATION_AUDIT = pd.DataFrame([
    {
        'provider': 'openai',
        'model_id': MODEL_CONFIGS['openai'].model_id,
        'documentation_checked_utc': '2026-07-22',
        'model_status': 'official GPT-5.6 Terra model page; Responses and structured outputs supported',
        'validated_controls': 'reasoning.effort=low; no temperature; no provider seed',
        'structured_output': 'client.responses.parse(..., text_format=PydanticModel)',
        'documentation': 'https://developers.openai.com/api/docs/models/gpt-5.6-terra',
    },
    {
        'provider': 'anthropic',
        'model_id': MODEL_CONFIGS['anthropic'].model_id,
        'documentation_checked_utc': '2026-07-22',
        'model_status': 'official dated Haiku 4.5 snapshot',
        'validated_controls': 'temperature=0; extended thinking disabled; no provider seed',
        'structured_output': 'messages.parse(..., output_format=PydanticModel)',
        'documentation': 'https://platform.claude.com/docs/en/build-with-claude/structured-outputs',
    },
    {
        'provider': 'gemini',
        'model_id': MODEL_CONFIGS['gemini'].model_id,
        'documentation_checked_utc': '2026-07-22',
        'model_status': 'official stable Gemini 3.1 Flash-Lite ID',
        'validated_controls': 'temperature=0; thinking_level=low; GenerationConfig.seed',
        'structured_output': 'application/json + response_json_schema',
        'documentation': 'https://ai.google.dev/gemini-api/docs/models/gemini-3.1-flash-lite',
    },
])
API_DOCUMENTATION_AUDIT.to_csv(CONFIG_ROOT / 'api_documentation_audit.csv', index=False)

display(API_DOCUMENTATION_AUDIT)

display(pd.DataFrame([
    {
        'provider': p,
        'temperature_requested': c.temperature_requested,
        'temperature_sent': c.temperature_sent,
        'native_seed_supported': c.native_seed_supported,
        'native_seed_enabled': c.native_seed_enabled,
        'reasoning': c.reasoning_label,
        'reproducibility_mode': c.reproducibility_mode,
    }
    for p, c in MODEL_CONFIGS.items()
]))

,provider,model_id,documentation_checked_utc,model_status,validated_controls,structured_output,documentation
0,openai,gpt-5.6-terra,2026-07-22,official GPT-5.6 Terra model page; Responses and structured outputs supported,reasoning.effort=low; no temperature; no provider seed,"client.responses.parse(..., text_format=PydanticModel)",https://developers.openai.com/api/docs/models/gpt-5.6-terra
1,anthropic,claude-haiku-4-5-20251001,2026-07-22,official dated Haiku 4.5 snapshot,temperature=0; extended thinking disabled; no provider seed,"messages.parse(..., output_format=PydanticModel)",https://platform.claude.com/docs/en/build-with-claude/structured-outputs
2,gemini,gemini-3.1-flash-lite,2026-07-22,official stable Gemini 3.1 Flash-Lite ID,temperature=0; thinking_level=low; GenerationConfig.seed,application/json + response_json_schema,https://ai.google.dev/gemini-api/docs/models/gemini-3.1-flash-lite


,provider,temperature_requested,temperature_sent,native_seed_supported,native_seed_enabled,reasoning,reproducibility_mode
0,openai,0.0,NaN,False,False,reasoning.effort=low,fixed prompt/schema/model configuration + raw logs + repeated identical calls
1,anthropic,0.0,0.0,False,False,lowest setting: extended thinking disabled,pinned dated model ID + temperature 0 + raw logs + repeated identical calls
2,gemini,0.0,0.0,True,True,thinking_level=low,stable model ID + temperature 0 + native decoding seed + raw logs + repeats


## 5. Registered definition treatments

In [13]:
@dataclass(frozen=True)
class DefinitionSpec:
    definition_id: str
    name: str
    status: str
    operational_text: str
    scientific_role: str

DEFINITIONS: dict[str, DefinitionSpec] = {
    'D0_none': DefinitionSpec(
        'D0_none', 'No supplied definition', 'control', '',
        'Direct-prompt baseline; ordinary model understanding.'
    ),
    'D1_old_broad': DefinitionSpec(
        'D1_old_broad', 'Old broad operational definition', 'historical treatment',
        """BROAD HISTORICAL OPERATIONALIZATION\n\nOrganizational unlearning is evidenced when a government organization reconsiders, discards, realigns, or merges established knowledge, assumptions, routines, roles, policies, capabilities, resources, or organizational arrangements after experience or failure.\n\nQualifying modes include epistemic reconsidering, normative discarding, technical realignment, and integrative merging. A passage may be positive even when it describes strategic integration, revised coordination, changed roles, technical adaptation, or additive institutional change rather than literal removal. Diagnosis alone is insufficient, but explicit words such as abandon or replace are not required.""",
        'Recover the broader construct used in early annotations.'
    ),
    'D2_current_strict': DefinitionSpec(
        'D2_current_strict', 'Current strict operational definition', 'current treatment',
        """CURRENT STRICT OPERATIONALIZATION\n\nOrganizational unlearning is present only when the passage identifies an earlier assumption, policy, practice, role arrangement, capability, resource logic, routine, or system as inadequate and indicates a meaningful departure from it through rejection, replacement, removal, abandonment, suspension, redistribution of authority, movement away from business as usual, or fundamental rethinking.\n\nDiagnosis, lessons-learned language, better implementation, extra training, extra staff, added equipment, more resources, and additive capacity are not sufficient unless the earlier approach being displaced is identifiable. Ordinary learning, reform, innovation, and improvement without discontinuity are No.""",
        'Create a sharp boundary between learning/reform and unlearning.'
    ),
    'D3_provisional_adaptive': DefinitionSpec(
        'D3_provisional_adaptive', 'Provisional adaptive-reconfiguration definition',
        'exploratory candidate; requires later human adjudication',
        """PROVISIONAL ADAPTIVE-RECONFIGURATION OPERATIONALIZATION\n\nOrganizational unlearning is present through either route.\n\nROUTE A — SUBTRACTIVE DISCONTINUITY: an identifiable earlier assumption, policy, practice, role arrangement, capability, resource logic, routine, or system is judged inadequate and is rejected, replaced, removed, abandoned, suspended, fundamentally rethought, or has authority redistributed.\n\nROUTE B — ADAPTIVE RECONFIGURATION: after a demonstrated failure, duplication, ambiguity, mismatch, or operational breakdown, the organization durably reconfigures formal roles, responsibility boundaries, coordination protocols, memoranda, standard operating procedures, decision rights, information routines, or technical operating routines for future events. The target must describe the institutionalized reconfiguration, not merely the problem.\n\nTraining, user guides, extra staff, equipment, laboratories, funding, or resources alone remain No. They count only when embedded in and subordinate to a qualifying change in the governing role, protocol, standard, authority structure, or routine. Temporary workarounds, implementation of an unchanged approach, diagnosis alone, and generic plans to improve remain No.""",
        'Test whether EPA recall can improve without treating all additive capacity as unlearning.'
    ),
}

DEFINITION_REGISTRY = pd.DataFrame([
    {**asdict(spec), 'sha256': sha256_text(spec.operational_text)}
    for spec in DEFINITIONS.values()
])
DEFINITION_REGISTRY.to_csv(CONFIG_ROOT / 'definition_registry.csv', index=False)
for spec in DEFINITIONS.values():
    (CONFIG_ROOT / f'{spec.definition_id}.txt').write_text(spec.operational_text, encoding='utf-8')
display(DEFINITION_REGISTRY[['definition_id','name','status','scientific_role','sha256']])

,definition_id,name,status,scientific_role,sha256
0,D0_none,No supplied definition,control,Direct-prompt baseline; ordinary model understanding.,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
1,D1_old_broad,Old broad operational definition,historical treatment,Recover the broader construct used in early annotations.,7531f2c94a782ca6a2642e01fca30d3bd631f94fb2a4e8fad4a90e82dfad513e
2,D2_current_strict,Current strict operational definition,current treatment,Create a sharp boundary between learning/reform and unlearning.,258f6556333fc5521c03d9aea3d43626ee22e7480b522f5e4c6729122e9e85ad
3,D3_provisional_adaptive,Provisional adaptive-reconfiguration definition,exploratory candidate; requires later human adjudication,Test whether EPA recall can improve without treating all additive capacity as unlearning.,99af0841b61738d090c0ecccff0f9c77d35f487944d743c52fa3dc11fcf595d5


### D3 interpretation guard

D3 is deliberately narrower than the original broad codebook. It can rescue cases involving formal liaison positions, changed responsibility boundaries, interagency protocols, revised SOPs, redistributed authority, or institutionalized technical routines after failure. It still excludes staff, training, equipment, laboratories, funding, and capacity additions by themselves. The notebook reports both EPA gains and non-EPA specificity costs.

## 6. Strict structured-output schemas — no narrative rationale

In [14]:
EvidenceLocation = Literal[
    'target', 'previous_1', 'previous_2', 'next_1', 'next_2', 'metadata', 'absent'
]
EvidenceElementName = Literal[
    'prior_state', 'inadequacy_or_failure', 'departure_or_reconfiguration', 'other'
]
TargetType = Literal[
    'leadership', 'laws_plans_policies', 'capabilities',
    'funds_resources', 'misc_organizational', 'none'
]
UnlearningMode = Literal[
    'subtractive_discontinuity', 'adaptive_reconfiguration',
    'epistemic_reconsidering', 'technical_realignment',
    'integrative_merging', 'none', 'unclear'
]
ChangeType = Literal[
    'reject', 'replace', 'remove', 'abandon', 'suspend',
    'fundamentally_rethink', 'redistribute_authority',
    'revise_protocol_or_standard', 'integrate_roles_or_plans',
    'add_capacity_only', 'diagnosis_only', 'no_change', 'unclear'
]

class StrictOutputModel(BaseModel):
    model_config = ConfigDict(extra='forbid')

class EvidenceQuote(StrictOutputModel):
    element: EvidenceElementName
    quote: str | None = Field(description='Exact short excerpt from supplied text, or null.')
    source_scope: EvidenceLocation

class ChecklistElement(StrictOutputModel):
    identified: bool
    quote: str | None = Field(description='Exact short excerpt from supplied text, or null.')
    source_scope: EvidenceLocation

class SimpleJointOutput(StrictOutputModel):
    unlearning_present: bool
    unlearning_mode: UnlearningMode
    change_type: ChangeType
    evidence: list[EvidenceQuote]
    target_type: TargetType
    agency: str | None
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_review: bool

class ChecklistJointOutput(StrictOutputModel):
    prior_state: ChecklistElement
    inadequacy_or_failure: ChecklistElement
    departure_or_reconfiguration: ChecklistElement
    unlearning_mode: UnlearningMode
    change_type: ChangeType
    all_required_elements_present: bool
    missing_elements: list[Literal[
        'prior_state', 'inadequacy_or_failure', 'departure_or_reconfiguration'
    ]]
    unlearning_present: bool
    target_type: TargetType
    agency: str | None
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_review: bool

class SimpleBinaryOutput(StrictOutputModel):
    unlearning_present: bool
    unlearning_mode: UnlearningMode
    change_type: ChangeType
    evidence: list[EvidenceQuote]
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_review: bool

class ChecklistBinaryOutput(StrictOutputModel):
    prior_state: ChecklistElement
    inadequacy_or_failure: ChecklistElement
    departure_or_reconfiguration: ChecklistElement
    unlearning_mode: UnlearningMode
    change_type: ChangeType
    all_required_elements_present: bool
    missing_elements: list[Literal[
        'prior_state', 'inadequacy_or_failure', 'departure_or_reconfiguration'
    ]]
    unlearning_present: bool
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_review: bool

class Stage2TargetOutput(StrictOutputModel):
    target_type: TargetType
    agency: str | None
    target_evidence_quote: str | None
    target_evidence_scope: EvidenceLocation
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_review: bool

SCHEMA_MODELS: dict[str, Type[BaseModel]] = {
    'simple_joint': SimpleJointOutput,
    'checklist_joint': ChecklistJointOutput,
    'simple_binary': SimpleBinaryOutput,
    'checklist_binary': ChecklistBinaryOutput,
    'stage2_target': Stage2TargetOutput,
}
SCHEMA_AUDIT = pd.DataFrame([
    {
        'schema_id': schema_id,
        'fields': ', '.join(model.model_fields),
        'has_rationale_field': 'rationale' in model.model_fields,
        'sha256': sha256_text(canonical_json(model.model_json_schema())),
    }
    for schema_id, model in SCHEMA_MODELS.items()
])
assert not SCHEMA_AUDIT['has_rationale_field'].any()
display(SCHEMA_AUDIT)

,schema_id,fields,has_rationale_field,sha256
0,simple_joint,"unlearning_present, unlearning_mode, change_type, evidence, target_type, agency, confidence, needs_human_review",False,b6fae6ca706adbcd29328a1015fe3b7e1788e2e54977bfe56b5b790981eb481e
1,checklist_joint,"prior_state, inadequacy_or_failure, departure_or_reconfiguration, unlearning_mode, change_type, all_required_elements_present, missing_elements, unlearning_present, target_type, agency, confidence, needs_human_review",False,160ef51b848bbba94ee5996a63b20821792de7b7319a848da6741e11e321f8b6
2,simple_binary,"unlearning_present, unlearning_mode, change_type, evidence, confidence, needs_human_review",False,247094025bbf94277cb5a6a9a2c817fdfc99013e4f7cf6a1adae2f2ed4951137
3,checklist_binary,"prior_state, inadequacy_or_failure, departure_or_reconfiguration, unlearning_mode, change_type, all_required_elements_present, missing_elements, unlearning_present, confidence, needs_human_review",False,509b21ba3a52fffc80899c327afc9f42e73bc2ce91a695ea01463c6cb5ba17bc
4,stage2_target,"target_type, agency, target_evidence_quote, target_evidence_scope, confidence, needs_human_review",False,9a6c02f76d61ee3d74f8687ae16f157a02d9bd113cc3d7ec9e181ddb616c6509


## 7. Benchmark loading, context parsing, and freeze

In [15]:
COLUMN_ALIASES = {
    'Number': 'row_id', 'Row ID': 'row_id',
    'Text Content': 'target_text', 'Paragraph Text': 'target_text',
    'Document': 'document_title', 'Document Title': 'document_title',
    'PDF Page': 'pdf_page', 'Page(s)': 'pdf_page',
    'Section Heading': 'section_heading', 'Section/Location': 'section_heading',
    'Paragraph Order': 'paragraph_order',
    'Local Context (±2 source paragraphs)': 'context_raw',
    'Unlearning': 'gold_historical', 'Unlearning?': 'gold_historical',
    'Original Unlearning': 'gold_historical',
    'Target': 'gold_target', 'Original Target': 'gold_target',
    'Government Agency': 'gold_agency', 'Original Government Agency': 'gold_agency',
    'Gold Old Definition': 'gold_old_definition',
    'Gold Current Definition': 'gold_current_definition',
    'Gold Final Definition': 'gold_final_definition',
}
REQUIRED_CANONICAL_COLUMNS = {
    'row_id', 'target_text', 'document_title', 'section_heading',
    'context_raw', 'gold_historical'
}
OPTIONAL_GOLD_COLUMNS = [
    'gold_old_definition', 'gold_current_definition', 'gold_final_definition',
    'gold_target', 'gold_agency'
]

def synthetic_benchmark() -> pd.DataFrame:
    records = [
        dict(row_id='S001', target_text='The agency replaced the layered response model with a push model after the prior approach delayed assistance.', document_title='Synthetic GAO', pdf_page=1, section_heading='Response doctrine', paragraph_order=1, context_raw='[PREVIOUS 2]\n\n[PREVIOUS 1]\nThe layered approach delayed aid.\n\n[TARGET]\nThe agency replaced the layered response model with a push model after the prior approach delayed assistance.\n\n[NEXT 1]\nThe new doctrine became standard.\n\n[NEXT 2]\n', gold_historical='Yes', gold_target='laws_plans_policies', gold_agency='Synthetic agency'),
        dict(row_id='S002', target_text='The office added two mobile laboratories and additional staff.', document_title='Synthetic EPA', pdf_page=2, section_heading='Capacity', paragraph_order=2, context_raw='[PREVIOUS 2]\n\n[PREVIOUS 1]\nAnalytical capacity was insufficient.\n\n[TARGET]\nThe office added two mobile laboratories and additional staff.\n\n[NEXT 1]\nMonitoring resumed.\n\n[NEXT 2]\n', gold_historical='Yes', gold_target='capabilities', gold_agency='Synthetic EPA'),
        dict(row_id='S003', target_text='Region 6 established a formal liaison, defined agency responsibilities, and replaced informal coordination with an interagency protocol.', document_title='Synthetic EPA', pdf_page=3, section_heading='Coordination', paragraph_order=3, context_raw='[PREVIOUS 2]\n\n[PREVIOUS 1]\nDuplicated work followed unclear roles.\n\n[TARGET]\nRegion 6 established a formal liaison, defined agency responsibilities, and replaced informal coordination with an interagency protocol.\n\n[NEXT 1]\nThe protocol governs future incidents.\n\n[NEXT 2]\n', gold_historical='Yes', gold_target='misc_organizational', gold_agency='Region 6'),
        dict(row_id='S004', target_text='The report documented communication failures during the storm.', document_title='Synthetic EPA', pdf_page=4, section_heading='Findings', paragraph_order=4, context_raw='[PREVIOUS 2]\n\n[PREVIOUS 1]\n\n[TARGET]\nThe report documented communication failures during the storm.\n\n[NEXT 1]\n\n[NEXT 2]\n', gold_historical='No', gold_target='none', gold_agency=None),
        dict(row_id='S005', target_text='Officials provided refresher training on the existing procedure.', document_title='Synthetic Post-Katrina', pdf_page=5, section_heading='Training', paragraph_order=5, context_raw='[PREVIOUS 2]\n\n[PREVIOUS 1]\nSome staff misunderstood the procedure.\n\n[TARGET]\nOfficials provided refresher training on the existing procedure.\n\n[NEXT 1]\nThe procedure itself was unchanged.\n\n[NEXT 2]\n', gold_historical='No', gold_target='none', gold_agency=None),
        dict(row_id='S006', target_text='The framework redistributed authority to local incident commanders and eliminated the former centralized approval requirement.', document_title='Synthetic Post-Katrina', pdf_page=6, section_heading='Authority', paragraph_order=6, context_raw='[PREVIOUS 2]\n\n[PREVIOUS 1]\nCentral approval caused delay.\n\n[TARGET]\nThe framework redistributed authority to local incident commanders and eliminated the former centralized approval requirement.\n\n[NEXT 1]\n\n[NEXT 2]\n', gold_historical='Yes', gold_target='leadership', gold_agency='Synthetic framework'),
    ]
    return pd.DataFrame(records)

In [16]:
def normalize_space(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()


def normalize_for_match(value: Any) -> str:
    text = unicodedata.normalize('NFKC', normalize_space(value))
    translation = str.maketrans({'“':'"','”':'"','‘':"'",'’':"'",'–':'-','—':'-','\u00a0':' '})
    return normalize_space(text.translate(translation)).casefold()


def normalize_label(value: Any) -> Optional[str]:
    text = normalize_space(value).casefold()
    if not text:
        return None
    if text in {'yes','y','true','1','unlearning','present'} or text.startswith('yes'):
        return 'Yes'
    if text in {'no','n','false','0','not unlearning','absent'} or text.startswith('no'):
        return 'No'
    return None


def label_to_int(value: Any) -> Optional[int]:
    return {'Yes':1, 'No':0}.get(normalize_label(value))


CONTEXT_MARKERS = ['PREVIOUS 2','PREVIOUS 1','TARGET','NEXT 1','NEXT 2']
CONTEXT_PATTERN = re.compile(r'\[(PREVIOUS 2|PREVIOUS 1|TARGET|NEXT 1|NEXT 2)\]\s*', flags=re.I)


def parse_context_blocks(raw: Any) -> dict[str, Any]:
    """Parse available context blocks while preserving boundary-row diagnostics."""
    text = '' if raw is None else str(raw)
    matches = list(CONTEXT_PATTERN.finditer(text))
    labels = [match.group(1).upper() for match in matches]
    counts = Counter(labels)
    blocks = {marker.lower().replace(' ','_'): '' for marker in CONTEXT_MARKERS}
    for index, match in enumerate(matches):
        start = match.end()
        stop = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        key = match.group(1).lower().replace(' ', '_')
        # Preserve the first occurrence and flag duplicates rather than silently overwriting.
        if not blocks[key]:
            blocks[key] = text[start:stop].strip()
    first_marker_start = matches[0].start() if matches else len(text)
    blocks.update({
        'marker_count': len(matches),
        'target_marker_count': counts.get('TARGET', 0),
        'duplicate_marker_count': sum(max(0, count - 1) for count in counts.values()),
        'available_neighbor_count': sum(bool(blocks[m.lower().replace(' ', '_')]) for m in CONTEXT_MARKERS if m != 'TARGET'),
        'marker_sequence_json': canonical_json(labels),
        'unparsed_preamble': text[:first_marker_start].strip(),
    })
    return blocks


def context_target_similarity(target: Any, parsed_target: Any) -> float:
    left, right = normalize_for_match(target), normalize_for_match(parsed_target)
    return SequenceMatcher(None, left, right).ratio() if (left or right) else 1.0


def canonicalize_benchmark(raw: pd.DataFrame) -> pd.DataFrame:
    data = raw.copy().dropna(axis=1, how='all')
    data = data.loc[:, ~data.columns.astype(str).str.match(r'^Unnamed')]
    data = data.rename(columns={k:v for k,v in COLUMN_ALIASES.items() if k in data.columns})
    missing = REQUIRED_CANONICAL_COLUMNS - set(data.columns)
    if missing:
        raise ValueError(f'Missing benchmark columns: {sorted(missing)}')
    for column in OPTIONAL_GOLD_COLUMNS:
        if column not in data.columns:
            data[column] = None
    for column in ['pdf_page','paragraph_order']:
        if column not in data.columns:
            data[column] = None
    data['row_id'] = data['row_id'].astype(str)
    for column in ['target_text','document_title','section_heading','context_raw']:
        data[column] = data[column].map(normalize_space)
    for column in ['gold_historical','gold_old_definition','gold_current_definition','gold_final_definition']:
        data[column] = data[column].map(normalize_label)
    parsed = data['context_raw'].map(parse_context_blocks).apply(pd.Series)
    parsed = parsed.rename(columns={
        'previous_2':'context_previous_2','previous_1':'context_previous_1',
        'target':'context_target','next_1':'context_next_1','next_2':'context_next_2'
    })
    data = pd.concat([data.reset_index(drop=True), parsed.reset_index(drop=True)], axis=1)
    data['context_target_similarity'] = data.apply(
        lambda row: context_target_similarity(row['target_text'], row['context_target']), axis=1
    )
    data['normalized_target_sha256'] = data['target_text'].map(
        lambda value: sha256_text(normalize_for_match(value))
    )
    return data


def load_benchmark() -> pd.DataFrame:
    if USE_SYNTHETIC_DATA:
        print('USING SYNTHETIC ENGINEERING FIXTURE — NOT SCIENTIFIC DATA')
        return canonicalize_benchmark(synthetic_benchmark())
    if not INPUT_WORKBOOK.exists():
        raise FileNotFoundError(
            f'Benchmark not found: {INPUT_WORKBOOK}. Upload {DEFAULT_WORKBOOK_NAME} '
            'or set UNLEARNING_INPUT_WORKBOOK.'
        )
    print('Loading benchmark:', INPUT_WORKBOOK)
    return canonicalize_benchmark(pd.read_excel(INPUT_WORKBOOK, sheet_name=INPUT_SHEET))


BENCHMARK_LOAD_ERROR = None
try:
    BENCHMARK = load_benchmark()
except Exception as exc:
    BENCHMARK = None
    BENCHMARK_LOAD_ERROR = f'{type(exc).__name__}: {exc}'
    print('BENCHMARK LOAD FAILED:', BENCHMARK_LOAD_ERROR)

Loading benchmark: /content/data/Unlearning_Codebook_Local_Context_Test_Set.xlsx


In [19]:
def token_set(value: Any) -> set[str]:
    return set(re.findall(r'[a-z0-9]+', normalize_for_match(value)))


def token_jaccard(left: Any, right: Any) -> float:
    a, b = token_set(left), token_set(right)
    return len(a & b) / len(a | b) if (a | b) else 1.0


def near_duplicate_pairs(data: pd.DataFrame, threshold: float = NEAR_DUPLICATE_JACCARD_THRESHOLD) -> pd.DataFrame:
    records = data[['row_id','target_text','document_title']].to_dict('records')
    rows = []
    for i in range(len(records)):
        for j in range(i + 1, len(records)):
            score = token_jaccard(records[i]['target_text'], records[j]['target_text'])
            if score >= threshold:
                rows.append({
                    'row_id_a': records[i]['row_id'], 'row_id_b': records[j]['row_id'],
                    'document_a': records[i]['document_title'], 'document_b': records[j]['document_title'],
                    'token_jaccard': score, 'text_a': records[i]['target_text'], 'text_b': records[j]['target_text'],
                })
    return pd.DataFrame(rows).sort_values('token_jaccard', ascending=False) if rows else pd.DataFrame()


def audit_benchmark(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    exact = data[data.duplicated('normalized_target_sha256', keep=False)]
    near = near_duplicate_pairs(data)
    all_five = data['marker_count'].eq(5) & data['duplicate_marker_count'].eq(0)
    checks = [
        {'check':'unique row_id','passed':data['row_id'].is_unique,'observed':data['row_id'].nunique(),'expected':len(data),'severity':'error'},
        {'check':'nonempty target','passed':data['target_text'].str.len().gt(0).all(),'observed':data['target_text'].str.len().gt(0).sum(),'expected':len(data),'severity':'error'},
        {'check':'exactly one TARGET marker','passed':data['target_marker_count'].eq(1).all(),'observed':data['target_marker_count'].eq(1).sum(),'expected':len(data),'severity':'error'},
        {'check':'no duplicate context markers','passed':data['duplicate_marker_count'].eq(0).all(),'observed':data['duplicate_marker_count'].eq(0).sum(),'expected':len(data),'severity':'error'},
        {'check':'target-context match','passed':data['context_target_similarity'].ge(TARGET_CONTEXT_MATCH_THRESHOLD).all(),'observed':data['context_target_similarity'].min(),'expected':f'>={TARGET_CONTEXT_MATCH_THRESHOLD}','severity':'error'},
        # Boundary paragraphs can legitimately lack PREVIOUS 2 or NEXT 2. This is
        # reported but no longer blocks target-only, metadata, or available-context runs.
        {'check':'all five context marker slots available','passed':all_five.all(),'observed':int(all_five.sum()),'expected':len(data),'severity':'warning'},
        {'check':'no exact target duplicates','passed':exact.empty,'observed':len(exact),'expected':0,'severity':'error'},
        {'check':'expected row count','passed':USE_SYNTHETIC_DATA or len(data)==EXPECTED_BENCHMARK_ROWS,'observed':len(data),'expected':EXPECTED_BENCHMARK_ROWS,'severity':'warning'},
        {'check':'historical labels complete','passed':data['gold_historical'].notna().all(),'observed':data['gold_historical'].notna().sum(),'expected':len(data),'severity':'error'},
    ]
    return pd.DataFrame(checks), near


if BENCHMARK is not None:
    BENCHMARK_AUDIT, NEAR_DUPLICATES = audit_benchmark(BENCHMARK)
    CONTEXT_AVAILABILITY = BENCHMARK[[
        'row_id','document_title','marker_count','target_marker_count',
        'available_neighbor_count','duplicate_marker_count','context_target_similarity'
    ]].copy()
    display(BENCHMARK_AUDIT)
    display(BENCHMARK.groupby(['document_title','gold_historical'], dropna=False).size().rename('rows').reset_index())
    incomplete_context = CONTEXT_AVAILABILITY[CONTEXT_AVAILABILITY['marker_count'].lt(5)]
    if not incomplete_context.empty:
        print('Boundary/incomplete context rows (allowed when TARGET is valid):')
        display(incomplete_context)
    if not NEAR_DUPLICATES.empty:
        display(NEAR_DUPLICATES)
else:
    BENCHMARK_AUDIT, NEAR_DUPLICATES, CONTEXT_AVAILABILITY = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()


def assert_benchmark_ready(data: Optional[pd.DataFrame]) -> pd.DataFrame:
    if data is None:
        raise RuntimeError(f'Benchmark unavailable: {BENCHMARK_LOAD_ERROR}')
    failed = BENCHMARK_AUDIT[(~BENCHMARK_AUDIT['passed']) & BENCHMARK_AUDIT['severity'].eq('error')]
    if not failed.empty:
        raise RuntimeError('Benchmark integrity checks failed:\n' + failed.to_string(index=False))
    return data

NameError: name 'BENCHMARK' is not defined

In [18]:
DEFINITION_GOLD_COLUMN = {
    'D0_none':'gold_historical',
    'D1_old_broad':'gold_old_definition',
    'D2_current_strict':'gold_current_definition',
    'D3_provisional_adaptive':'gold_final_definition',
}
GOLD_AVAILABILITY = pd.DataFrame([
    {
        'definition_id': definition_id,
        'preferred_gold_column': column,
        'complete': bool(BENCHMARK is not None and column in BENCHMARK and BENCHMARK[column].notna().all()),
        'non_null_rows': int(BENCHMARK[column].notna().sum()) if BENCHMARK is not None and column in BENCHMARK else 0,
    }
    for definition_id, column in DEFINITION_GOLD_COLUMN.items()
])
display(GOLD_AVAILABILITY)

DATASET_HASH_COLUMNS = [
    'row_id','target_text','document_title','pdf_page','section_heading','paragraph_order',
    'context_previous_2','context_previous_1','context_target','context_next_1','context_next_2',
    'gold_historical','gold_old_definition','gold_current_definition','gold_final_definition',
    'gold_target','gold_agency'
]
DATASET_SHA256 = None
if BENCHMARK is not None:
    canonical = BENCHMARK[DATASET_HASH_COLUMNS].fillna('').astype(str).to_csv(index=False, lineterminator='\n')
    DATASET_SHA256 = sha256_text(canonical)
    BENCHMARK.to_csv(ARTIFACTS_ROOT / 'benchmark_canonical.csv', index=False)
    with contextlib.suppress(Exception):
        BENCHMARK.to_parquet(ARTIFACTS_ROOT / 'benchmark_canonical.parquet', index=False)
    manifest = {
        'dataset_sha256': DATASET_SHA256,
        'source_path': str(INPUT_WORKBOOK),
        'source_file_sha256': sha256_file(INPUT_WORKBOOK),
        'sheet': INPUT_SHEET,
        'rows': len(BENCHMARK),
        'columns_hashed': DATASET_HASH_COLUMNS,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    (ARTIFACTS_ROOT / 'dataset_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print('DATASET SHA-256:', DATASET_SHA256)

,definition_id,preferred_gold_column,complete,non_null_rows
0,D0_none,gold_historical,True,42
1,D1_old_broad,gold_old_definition,False,0
2,D2_current_strict,gold_current_definition,False,0
3,D3_provisional_adaptive,gold_final_definition,False,0


DATASET SHA-256: ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0


## 8. Experimental condition registry

In [19]:
PromptStyle = Literal['P1_direct','P2_simple_definition','P3_evidence_checklist']
ContextLevel = Literal['C1_target','C2_metadata','C3_plusminus1','C4_plusminus2']
Workflow = Literal['W1_one_stage','W2_binary_first']
Stage = Literal['joint','binary','target']

@dataclass(frozen=True)
class ConditionSpec:
    condition_id: str
    phase: str
    definition_id: str
    prompt_style: PromptStyle
    context_id: ContextLevel
    workflow: Workflow
    description: str

PHASE1_CONDITIONS = [
    ConditionSpec('P1_DIRECT','phase1','D0_none','P1_direct','C1_target','W1_one_stage','Direct task; no supplied definition.'),
    ConditionSpec('P2_D1_OLD','phase1','D1_old_broad','P2_simple_definition','C1_target','W1_one_stage','Old definition + simple structured classification.'),
    ConditionSpec('P2_D2_CURRENT','phase1','D2_current_strict','P2_simple_definition','C1_target','W1_one_stage','Current definition + simple structured classification.'),
    ConditionSpec('P2_D3_ADAPTIVE','phase1','D3_provisional_adaptive','P2_simple_definition','C1_target','W1_one_stage','Adaptive candidate + simple structured classification.'),
    ConditionSpec('P3_D1_OLD','phase1','D1_old_broad','P3_evidence_checklist','C1_target','W1_one_stage','Old definition + explicit evidence checklist.'),
    ConditionSpec('P3_D2_CURRENT','phase1','D2_current_strict','P3_evidence_checklist','C1_target','W1_one_stage','Current definition + explicit evidence checklist.'),
    ConditionSpec('P3_D3_ADAPTIVE','phase1','D3_provisional_adaptive','P3_evidence_checklist','C1_target','W1_one_stage','Adaptive candidate + explicit evidence checklist.'),
]
PHASE1_REGISTRY = pd.DataFrame([asdict(x) for x in PHASE1_CONDITIONS])
display(PHASE1_REGISTRY)

,condition_id,phase,definition_id,prompt_style,context_id,workflow,description
0,P1_DIRECT,phase1,D0_none,P1_direct,C1_target,W1_one_stage,Direct task; no supplied definition.
1,P2_D1_OLD,phase1,D1_old_broad,P2_simple_definition,C1_target,W1_one_stage,Old definition + simple structured classification.
2,P2_D2_CURRENT,phase1,D2_current_strict,P2_simple_definition,C1_target,W1_one_stage,Current definition + simple structured classification.
3,P2_D3_ADAPTIVE,phase1,D3_provisional_adaptive,P2_simple_definition,C1_target,W1_one_stage,Adaptive candidate + simple structured classification.
4,P3_D1_OLD,phase1,D1_old_broad,P3_evidence_checklist,C1_target,W1_one_stage,Old definition + explicit evidence checklist.
5,P3_D2_CURRENT,phase1,D2_current_strict,P3_evidence_checklist,C1_target,W1_one_stage,Current definition + explicit evidence checklist.
6,P3_D3_ADAPTIVE,phase1,D3_provisional_adaptive,P3_evidence_checklist,C1_target,W1_one_stage,Adaptive candidate + explicit evidence checklist.


## 9. Prompt construction and leakage audit

In [20]:
TARGET_CODEBOOK_TEXT = """
TARGET CATEGORY — assign exactly one only when unlearning_present is true.
- leadership: prior leadership, command, authority, accountability, or decision-right arrangement is displaced.
- laws_plans_policies: prior law, policy, doctrine, plan, formal rule, standard, or operating policy is displaced.
- capabilities: prior technical system, procedure, information process, organizational capability, operational method, or professional routine is displaced.
- funds_resources: prior funding rule, budget logic, resource-distribution practice, procurement arrangement, or material-resource policy is displaced.
- misc_organizational: qualifying organizational arrangement not represented above; use sparingly.
- none: required when unlearning_present is false.
""".strip()

EVIDENCE_RULES_TEXT = """
STRUCTURED EVIDENCE RULES
- Return only registered fields. Do not provide narrative explanation or hidden chain-of-thought.
- Every quote must be an exact short excerpt from one supplied source block or null.
- Declare the correct source_scope.
- Do not invent a prior state, failure, change, agency, or target.
- Context may resolve antecedents or background, but a neighbor cannot independently make TARGET positive.
- A positive decision must include TARGET evidence of the decisive departure or reconfiguration.
- For No, set target_type='none' and agency=null in joint output.
- Set needs_human_review=true for ambiguous boundaries, mixed mechanisms, or uncertain target/agency.
""".strip()

DIRECT_TASK_TEXT = """
Classify whether the TARGET contains organizational unlearning in a government organization using ordinary understanding. Extract exact textual evidence before deciding.
""".strip()
SIMPLE_TASK_TEXT = """
Classify whether TARGET satisfies the supplied operational definition. Return the decision, change mechanism, structured evidence, target, agency, confidence, and review flag.
""".strip()
CHECKLIST_TASK_TEXT = """
Apply the definition with an explicit evidence checklist. Identify separately: (1) prior state; (2) inadequacy/failure; (3) departure or durable reconfiguration in TARGET. Then decide and assign mechanism, target, agency, confidence, and review flag.
""".strip()
STAGE2_TASK_TEXT = """
Stage 1 already made a positive binary judgment. Do not reconsider it. Assign target category and agency, grounded in one exact excerpt.
""".strip()

def schema_id_for(prompt_style: PromptStyle, stage: Stage) -> str:
    checklist = prompt_style == 'P3_evidence_checklist'
    if stage == 'joint':
        return 'checklist_joint' if checklist else 'simple_joint'
    if stage == 'binary':
        return 'checklist_binary' if checklist else 'simple_binary'
    return 'stage2_target'

def source_blocks_for_context(row: pd.Series, context_id: ContextLevel) -> dict[str,str]:
    blocks = {'target': normalize_space(row['target_text'])}
    if context_id in {'C2_metadata','C3_plusminus1','C4_plusminus2'}:
        blocks['metadata'] = (
            f"Document title: {normalize_space(row['document_title'])}\n"
            f"Section heading: {normalize_space(row['section_heading'])}\n"
            f"PDF page: {normalize_space(row.get('pdf_page'))}"
        )
    if context_id in {'C3_plusminus1','C4_plusminus2'}:
        blocks['previous_1'] = normalize_space(row['context_previous_1'])
        blocks['next_1'] = normalize_space(row['context_next_1'])
    if context_id == 'C4_plusminus2':
        blocks['previous_2'] = normalize_space(row['context_previous_2'])
        blocks['next_2'] = normalize_space(row['context_next_2'])
    return blocks

def render_variable_input(row: pd.Series, context_id: ContextLevel) -> str:
    blocks = source_blocks_for_context(row, context_id)
    sections = [f"ROW ID: {row['row_id']}"]
    if blocks.get('metadata'):
        sections.append('<METADATA>\n'+blocks['metadata']+'\n</METADATA>')
    if blocks.get('previous_2'):
        sections.append('<PREVIOUS_2>\n'+blocks['previous_2']+'\n</PREVIOUS_2>')
    if blocks.get('previous_1'):
        sections.append('<PREVIOUS_1>\n'+blocks['previous_1']+'\n</PREVIOUS_1>')
    sections.append('<TARGET>\n'+blocks['target']+'\n</TARGET>')
    if blocks.get('next_1'):
        sections.append('<NEXT_1>\n'+blocks['next_1']+'\n</NEXT_1>')
    if blocks.get('next_2'):
        sections.append('<NEXT_2>\n'+blocks['next_2']+'\n</NEXT_2>')
    return '\n\n'.join(sections)

@dataclass(frozen=True)
class PromptPackage:
    system_prompt: str
    user_prompt: str
    schema_id: str
    schema_model: Type[BaseModel]
    schema_sha256: str
    prompt_sha256: str
    cache_key: str
    source_blocks: dict[str,str]

def build_system_prompt(condition: ConditionSpec, stage: Stage) -> str:
    sections = [
        'You are an independent qualitative-coding model for a reproducible benchmark on organizational unlearning in government agencies.',
        'Classify one TARGET paragraph per request. Never infer or reproduce a human label. Return only registered structured output.',
    ]
    if condition.prompt_style == 'P1_direct':
        sections.append(DIRECT_TASK_TEXT)
    else:
        sections.append('OPERATIONAL DEFINITION\n\n'+DEFINITIONS[condition.definition_id].operational_text)
        sections.append(CHECKLIST_TASK_TEXT if condition.prompt_style == 'P3_evidence_checklist' else SIMPLE_TASK_TEXT)
    if stage in {'joint','target'}:
        sections.append(TARGET_CODEBOOK_TEXT)
    sections.append(EVIDENCE_RULES_TEXT)
    if stage == 'binary':
        sections.append('This is Stage 1. Return binary/evidence fields only; do not assign target or agency.')
    elif stage == 'target':
        sections.append(STAGE2_TASK_TEXT)
    sections.append(f'SCHEMA ID: {schema_id_for(condition.prompt_style, stage)}')
    sections.append(f'PROTOCOL VERSION: {PROTOCOL_VERSION}')
    return '\n\n'.join(sections)

def build_prompt_package(row: pd.Series, condition: ConditionSpec, stage: Stage='joint', stage1_output: Optional[dict[str,Any]]=None) -> PromptPackage:
    schema_id = schema_id_for(condition.prompt_style, stage)
    schema_model = SCHEMA_MODELS[schema_id]
    system_prompt = build_system_prompt(condition, stage)
    user_prompt = render_variable_input(row, condition.context_id)
    if stage == 'target':
        if stage1_output is None:
            raise ValueError('Stage 2 requires Stage-1 output.')
        user_prompt += '\n\n<STAGE_1_STRUCTURED_RESULT>\n'+json.dumps(stage1_output, ensure_ascii=False, sort_keys=True)+'\n</STAGE_1_STRUCTURED_RESULT>'
    schema_hash = sha256_text(canonical_json(schema_model.model_json_schema()))
    prompt_hash = sha256_text(canonical_json({'system':system_prompt,'user':user_prompt,'schema':schema_hash}))
    return PromptPackage(
        system_prompt, user_prompt, schema_id, schema_model, schema_hash, prompt_hash,
        f'{PROTOCOL_VERSION}:{condition.definition_id}:{condition.prompt_style}:{stage}:{schema_id}',
        source_blocks_for_context(row, condition.context_id),
    )

In [21]:
FORBIDDEN_PROMPT_TERMS = [
    'gold_historical','gold_old_definition','gold_current_definition','gold_final_definition',
    'gold_target','gold_agency','original rationale','labeled examples'
]

def audit_prompt_package(package: PromptPackage, row: pd.Series, condition: ConditionSpec) -> dict[str,Any]:
    combined = (package.system_prompt+'\n'+package.user_prompt).casefold()
    target = normalize_for_match(row['target_text'])
    user = normalize_for_match(package.user_prompt)
    return {
        'condition_id':condition.condition_id,'definition_id':condition.definition_id,
        'prompt_style':condition.prompt_style,'context_id':condition.context_id,
        'schema_id':package.schema_id,'target_occurrences_normalized':user.count(target),
        'contains_forbidden_term':any(term.casefold() in combined for term in FORBIDDEN_PROMPT_TERMS),
        'contains_definition':bool(DEFINITIONS[condition.definition_id].operational_text and DEFINITIONS[condition.definition_id].operational_text in package.system_prompt),
        'contains_checklist':'explicit evidence checklist' in package.system_prompt,
        'prompt_sha256':package.prompt_sha256,'system_chars':len(package.system_prompt),'user_chars':len(package.user_prompt),
    }

PROMPT_AUDIT = pd.DataFrame()
if BENCHMARK is not None:
    PROMPT_AUDIT = pd.DataFrame([
        audit_prompt_package(build_prompt_package(BENCHMARK.iloc[0], c), BENCHMARK.iloc[0], c)
        for c in PHASE1_CONDITIONS
    ])
    assert PROMPT_AUDIT['target_occurrences_normalized'].eq(1).all()
    assert not PROMPT_AUDIT['contains_forbidden_term'].any()
    display(PROMPT_AUDIT)

,condition_id,definition_id,prompt_style,context_id,schema_id,target_occurrences_normalized,contains_forbidden_term,contains_definition,contains_checklist,prompt_sha256,system_chars,user_chars
0,P1_DIRECT,D0_none,P1_direct,C1_target,simple_joint,1,False,False,False,299499495898c0a53e4ee6b8bb1e16c79e6e9ec4f83691ffb17b431144e9aebe,1951,405
1,P2_D1_OLD,D1_old_broad,P2_simple_definition,C1_target,simple_joint,1,False,True,False,384a054cde59870385af856c8d124d579fdb853c3803ea5ce5743924576a6b7d,2691,405
2,P2_D2_CURRENT,D2_current_strict,P2_simple_definition,C1_target,simple_joint,1,False,True,False,5dc8f427bc9ccc4248a0266947cc5102413d02ab316f6b7d61256b658ed53f9f,2722,405
3,P2_D3_ADAPTIVE,D3_provisional_adaptive,P2_simple_definition,C1_target,simple_joint,1,False,True,False,936350627df93f2b7c5014a138d0934fde15b01de7a3db50a92260f967a71388,3211,405
4,P3_D1_OLD,D1_old_broad,P3_evidence_checklist,C1_target,checklist_joint,1,False,True,True,ed3f7345212078b69580314f87ce54685dadd4dafcbaace61a53f8625151f217,2769,405
5,P3_D2_CURRENT,D2_current_strict,P3_evidence_checklist,C1_target,checklist_joint,1,False,True,True,56d9e5f9387d230ded0de0506b2b4816daa5a31458c82577daf3e5a44f5d95f3,2800,405
6,P3_D3_ADAPTIVE,D3_provisional_adaptive,P3_evidence_checklist,C1_target,checklist_joint,1,False,True,True,909a19f8c68a7bacf35a6078fedde62d6d76dcfcc2ebcf25e43b8bdf21884f73,3289,405


## 10. Balanced scheduling, append-only logs, and deterministic run keys

In [22]:
@dataclass(frozen=True)
class TaskSpec:
    phase: str
    condition_id: str
    row_id: str
    seed: int
    repeat_id: str
    stage: Stage
    sequence: int

def conditions_by_id(conditions: Sequence[ConditionSpec]) -> dict[str,ConditionSpec]:
    mapping = {condition.condition_id: condition for condition in conditions}
    if len(mapping) != len(conditions):
        raise ValueError('Condition IDs must be unique within a phase.')
    return mapping

def blocked_condition_order(row_ids: Sequence[str], condition_ids: Sequence[str], seed: int) -> list[tuple[str,str,int]]:
    rng = random.Random(stable_int_seed(PROTOCOL_VERSION, seed, 'schedule'))
    shuffled_rows = list(row_ids)
    rng.shuffle(shuffled_rows)
    rows = []
    sequence = 0
    for row_index, row_id in enumerate(shuffled_rows):
        local_conditions = list(condition_ids)
        local_rng = random.Random(stable_int_seed(PROTOCOL_VERSION, seed, row_id, 'condition_order'))
        local_rng.shuffle(local_conditions)
        # Rotate to reduce provider-time confounding beyond random shuffling.
        offset = row_index % max(1, len(local_conditions))
        local_conditions = local_conditions[offset:] + local_conditions[:offset]
        for condition_id in local_conditions:
            rows.append((str(row_id), condition_id, sequence))
            sequence += 1
    return rows

def make_task_schedule(data: pd.DataFrame, conditions: Sequence[ConditionSpec], seeds: Sequence[int], phase: str, stage: Stage='joint') -> list[TaskSpec]:
    tasks = []
    for seed in seeds:
        ordered = blocked_condition_order(data['row_id'].astype(str).tolist(), [c.condition_id for c in conditions], seed)
        for row_id, condition_id, sequence in ordered:
            tasks.append(TaskSpec(
                phase=phase, condition_id=condition_id, row_id=row_id, seed=seed,
                repeat_id=f'seed_{seed}_{RUN_REPLICATE_ID}', stage=stage, sequence=sequence,
            ))
    return tasks

def provider_phase_directory(phase: str, provider: str) -> Path:
    path = RUNS_ROOT / phase / provider
    path.mkdir(parents=True, exist_ok=True)
    return path

def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix+'.tmp')
    temp.write_text(text, encoding='utf-8')
    temp.replace(path)

def atomic_write_dataframe_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix+'.tmp')
    frame.to_csv(temp, index=False)
    temp.replace(path)

def append_jsonl(path: Path, record: dict[str,Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False, default=str)+'\n')

def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    records = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Malformed JSONL at {path}:{line_number}: {exc}') from exc
    return pd.DataFrame(records)

def latest_records_by_run_key(path: Path) -> dict[str,dict[str,Any]]:
    latest = {}
    if not path.exists():
        return latest
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                record = json.loads(line)
                if record.get('run_key'):
                    latest[record['run_key']] = record
    return latest

def successful_run_keys(path: Path) -> set[str]:
    return {key for key, record in latest_records_by_run_key(path).items() if record.get('status') == 'ok'}

def make_run_key(provider_config: ProviderModelConfig, task: TaskSpec, condition: ConditionSpec, package: PromptPackage) -> str:
    payload = {
        'protocol_version':PROTOCOL_VERSION,'dataset_sha256':DATASET_SHA256,
        'provider':provider_config.provider,'model':provider_config.model_id,
        'model_config_sha256':model_config_hash(provider_config),'phase':task.phase,
        'condition_id':task.condition_id,'definition_id':condition.definition_id,
        'definition_sha256':sha256_text(DEFINITIONS[condition.definition_id].operational_text),
        'prompt_style':condition.prompt_style,'context_id':condition.context_id,
        'workflow':condition.workflow,'stage':task.stage,'row_id':task.row_id,
        'seed':task.seed,'repeat_id':task.repeat_id,'schema_sha256':package.schema_sha256,
        'prompt_sha256':package.prompt_sha256,
    }
    return sha256_text(canonical_json(payload))

## 11. Provider result contract, retries, and deterministic mock

In [23]:
@dataclass
class ProviderCallResult:
    provider: str
    requested_model: str
    returned_model: Optional[str]
    request_id: Optional[str]
    raw_response_text: str
    parsed_output: dict[str,Any]
    input_tokens: Optional[int]
    output_tokens: Optional[int]
    total_tokens: Optional[int]
    reasoning_tokens: Optional[int]
    cached_input_tokens: Optional[int]
    cache_creation_input_tokens: Optional[int]
    stop_reason: Optional[str]
    latency_seconds: float
    accepted_parameters: dict[str,Any]
    provider_metadata: dict[str,Any] = field(default_factory=dict)


class ProviderProtocolError(RuntimeError):
    pass


class ProviderConfigurationError(ProviderProtocolError):
    pass


class ProviderTransientError(ProviderProtocolError):
    pass


def safe_attr(obj: Any, path: str, default: Any=None) -> Any:
    current = obj
    for part in path.split('.'):
        if current is None:
            return default
        if isinstance(current, dict):
            current = current.get(part, default)
        elif part.isdigit() and isinstance(current, (list,tuple)):
            index = int(part)
            current = current[index] if index < len(current) else default
        else:
            current = getattr(current, part, default)
    return current


def exception_status_code(exc: Exception) -> Optional[int]:
    for path in ['status_code','response.status_code','code']:
        value = safe_attr(exc, path)
        with contextlib.suppress(TypeError,ValueError):
            if value is not None:
                return int(value)
    return None


def error_classification(exc: Exception) -> str:
    """Classify before retrying; configuration and quota failures are never fanned out."""
    status = exception_status_code(exc)
    text = (type(exc).__name__ + ' ' + str(exc)).casefold()

    if isinstance(exc, ProviderConfigurationError):
        return 'configuration'
    if isinstance(exc, (TypeError, ImportError, ModuleNotFoundError)):
        return 'configuration'
    if any(token in text for token in [
        'unexpected keyword argument', 'unsupported parameter', 'not supported with this model',
        'string too long', 'model not found', 'unknown model', 'invalid api key',
        'authentication', 'permission denied', 'invalid_request_error',
    ]):
        return 'configuration'
    if status in {400,401,403,404,405,422}:
        return 'configuration'

    if any(token in text for token in [
        'quota exceeded', 'resource_exhausted', 'insufficient_quota',
        'free-tier quota', 'free tier quota', 'daily quota', 'billing limit',
    ]):
        return 'quota'

    if status in {408,409,425,429} or (status is not None and status >= 500):
        return 'transient'
    if any(token in text for token in [
        'timeout', 'connection', 'temporar', 'rate limit', 'overloaded',
        'service unavailable', 'server error',
    ]):
        return 'transient'

    if type(exc).__name__ in {'ValidationError','JSONDecodeError'}:
        return 'validation'
    if isinstance(exc, ProviderProtocolError):
        return 'protocol'
    return 'unknown'


def error_fingerprint(exc: Exception, classification: Optional[str]=None) -> str:
    normalized_message = re.sub(r'\b\d+(?:\.\d+)?\b', '<N>', normalize_space(str(exc)).casefold())
    payload = {
        'classification': classification or error_classification(exc),
        'type': type(exc).__name__,
        'status_code': exception_status_code(exc),
        'message': normalized_message[:1500],
    }
    return sha256_text(canonical_json(payload))


def provider_circuit_breaker_reason(
    classification: str,
    identical_error_count: int,
    consecutive_error_count: int,
) -> Optional[str]:
    if classification == 'configuration' and STOP_PROVIDER_ON_CONFIGURATION_ERROR:
        return 'configuration error'
    if classification == 'quota' and STOP_PROVIDER_ON_QUOTA_ERROR:
        return 'quota error'
    if identical_error_count >= MAX_IDENTICAL_ERROR_FINGERPRINTS:
        return f'repeated identical error ({identical_error_count})'
    if consecutive_error_count >= MAX_CONSECUTIVE_ERRORS:
        return f'consecutive error limit ({consecutive_error_count})'
    return None


def call_with_retry(function: Callable[[],ProviderCallResult], seed: int) -> ProviderCallResult:
    rng = random.Random(stable_int_seed(PROTOCOL_VERSION, seed, 'retry'))
    for attempt in range(1, MAX_RETRY_ATTEMPTS + 1):
        try:
            return function()
        except Exception as exc:
            classification = error_classification(exc)
            if classification != 'transient' or attempt == MAX_RETRY_ATTEMPTS:
                raise
            delay = BASE_RETRY_SECONDS * 2**(attempt - 1) + rng.uniform(0, BASE_RETRY_SECONDS)
            print(f'Transient error {attempt}/{MAX_RETRY_ATTEMPTS}; retry in {delay:.2f}s: {exc}')
            time.sleep(delay)
    raise RuntimeError('Retry loop ended unexpectedly.')

In [24]:
def extract_tagged_target(user_prompt: str) -> str:
    match = re.search(r'<TARGET>\s*(.*?)\s*</TARGET>', user_prompt, flags=re.S)
    return normalize_space(match.group(1)) if match else ''

def first_matching_quote(text: str, patterns: Sequence[str]) -> Optional[str]:
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.I)
        if match:
            return normalize_space(match.group(0))
    return None

def mock_decision(package: PromptPackage, schema_model: Type[BaseModel], seed: int) -> dict[str,Any]:
    target = extract_tagged_target(package.user_prompt)
    lower = target.casefold()
    adaptive = 'ROUTE B — ADAPTIVE RECONFIGURATION' in package.system_prompt
    old = 'BROAD HISTORICAL OPERATIONALIZATION' in package.system_prompt
    subtractive = any(x in lower for x in ['replace','eliminat','abandon','remove','move away','redistribut','no longer','fundamentally'])
    reconfiguration = any(x in lower for x in ['protocol','defined agency responsibilities','formal liaison','standard operating procedure','integrated operational plan'])
    capacity_only = any(x in lower for x in ['additional staff','mobile laborator','refresher training','user guide','equipment']) and not reconfiguration
    diagnosis_only = any(x in lower for x in ['documented','could have been better','experienced problems']) and not (subtractive or reconfiguration)
    positive = subtractive
    mode = 'subtractive_discontinuity' if subtractive else 'none'
    change_type = 'replace' if 'replace' in lower else ('redistribute_authority' if 'redistribut' in lower else ('remove' if 'eliminat' in lower else 'no_change'))
    if adaptive and reconfiguration and not capacity_only:
        positive, mode, change_type = True, 'adaptive_reconfiguration', 'revise_protocol_or_standard'
    elif old and (reconfiguration or capacity_only):
        positive, mode = True, 'technical_realignment'
        change_type = 'revise_protocol_or_standard' if reconfiguration else 'add_capacity_only'
    elif capacity_only:
        positive, change_type = False, 'add_capacity_only'
    elif diagnosis_only:
        positive, change_type = False, 'diagnosis_only'
    departure = first_matching_quote(target,[r'replaced[^.]*',r'redistributed[^.]*',r'eliminated[^.]*',r'established[^.]*protocol[^.]*',r'added[^.]*',r'documented[^.]*'])
    prior = first_matching_quote(package.user_prompt,[r'layered approach delayed aid',r'central approval caused delay',r'duplicated work followed unclear roles',r'analytical capacity was insufficient',r'Some staff misunderstood the procedure'])
    target_type: TargetType = 'none'
    if positive:
        if any(x in lower for x in ['authority','commander','leadership']): target_type='leadership'
        elif any(x in lower for x in ['doctrine','policy','framework']): target_type='laws_plans_policies'
        elif any(x in lower for x in ['protocol','procedure','laborator']): target_type='capabilities'
        else: target_type='misc_organizational'
    confidence = 0.91 if subtractive else (0.78 if positive else 0.82)
    review = bool(reconfiguration or capacity_only)
    if schema_model in {SimpleJointOutput,SimpleBinaryOutput}:
        evidence=[]
        if prior: evidence.append({'element':'prior_state','quote':prior,'source_scope':'previous_1'})
        if departure: evidence.append({'element':'departure_or_reconfiguration','quote':departure,'source_scope':'target'})
        payload={'unlearning_present':positive,'unlearning_mode':mode,'change_type':change_type,'evidence':evidence,'confidence':confidence,'needs_human_review':review}
        if schema_model is SimpleJointOutput: payload.update({'target_type':target_type,'agency':'Synthetic agency' if positive else None})
    elif schema_model in {ChecklistJointOutput,ChecklistBinaryOutput}:
        payload={
            'prior_state':{'identified':bool(prior),'quote':prior,'source_scope':'previous_1' if prior else 'absent'},
            'inadequacy_or_failure':{'identified':bool(prior),'quote':prior,'source_scope':'previous_1' if prior else 'absent'},
            'departure_or_reconfiguration':{'identified':bool(departure),'quote':departure,'source_scope':'target' if departure else 'absent'},
            'unlearning_mode':mode,'change_type':change_type,
            'all_required_elements_present':bool(prior and departure),
            'missing_elements':[name for name,present in [('prior_state',bool(prior)),('inadequacy_or_failure',bool(prior)),('departure_or_reconfiguration',bool(departure))] if not present],
            'unlearning_present':positive,'confidence':confidence,'needs_human_review':review,
        }
        if schema_model is ChecklistJointOutput: payload.update({'target_type':target_type,'agency':'Synthetic agency' if positive else None})
    elif schema_model is Stage2TargetOutput:
        payload={'target_type':target_type if target_type!='none' else 'misc_organizational','agency':'Synthetic agency','target_evidence_quote':departure,'target_evidence_scope':'target' if departure else 'absent','confidence':confidence,'needs_human_review':review}
    else:
        raise ValueError(f'Unsupported mock schema: {schema_model}')
    return schema_model.model_validate(payload).model_dump()

def call_mock_provider(config: ProviderModelConfig, package: PromptPackage, seed: int) -> ProviderCallResult:
    started=time.perf_counter(); parsed=mock_decision(package,package.schema_model,seed); raw=json.dumps(parsed,ensure_ascii=False)
    input_tokens=len((package.system_prompt+' '+package.user_prompt).split()); output_tokens=len(raw.split())
    return ProviderCallResult(config.provider,config.model_id,'mock-deterministic-v1',f'mock_{uuid.uuid4().hex}',raw,parsed,input_tokens,output_tokens,input_tokens+output_tokens,0,0,0,'mock_complete',time.perf_counter()-started,{'temperature_requested':config.temperature_requested,'temperature_sent':config.temperature_sent,'provider_seed_sent':config.native_seed_enabled,'provider_seed_value':seed if config.native_seed_enabled else None,'mock':True})

## 12. Provider-native structured-output adapters

The adapters below deliberately do not share execution state. Each provider has its own client, request builder, smoke-test cell, result directory, JSONL log, CSV snapshot, error log, and status file. A provider configuration error is recorded and stops that provider only.

Every request passes through a provider-specific allowlist immediately before the SDK call. Known unsupported fields are stripped and recorded; unknown fields cause a local configuration error before network transmission. The OpenAI firewall specifically forbids `temperature` and `seed`.

There is no automatic substitution of models, prompt styles, context levels, or reasoning settings. Provider transport differences that do not alter the scientific prompt—such as OpenAI omitting an unsupported temperature field—are explicitly stored in the model protocol and every result record.

In [25]:
_OPENAI_CLIENT = None
OPENAI_PROMPT_CACHE_KEY_MAX_CHARS = 64
OPENAI_PARSE_ALLOWED_PARAMETERS = frozenset({
    'model', 'input', 'text_format', 'max_output_tokens',
    'prompt_cache_key', 'store', 'reasoning',
})
OPENAI_FORBIDDEN_TRANSPORT_PARAMETERS = frozenset({
    'temperature', 'seed', 'top_p', 'top_k',
    'frequency_penalty', 'presence_penalty', 'logit_bias',
})


def get_openai_client():
    global _OPENAI_CLIENT
    if _OPENAI_CLIENT is None:
        from openai import OpenAI
        _OPENAI_CLIENT = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    return _OPENAI_CLIENT


def openai_prompt_cache_key(package: PromptPackage) -> str:
    # Stable per static definition/prompt/stage prefix. 3 + 60 = 63 characters.
    key_payload = canonical_json({
        'cache_group': package.cache_key,
        'system_prompt_sha256': sha256_text(package.system_prompt),
        'schema_sha256': package.schema_sha256,
    })
    key = 'ul-' + sha256_text(key_payload)[:60]
    if len(key) > OPENAI_PROMPT_CACHE_KEY_MAX_CHARS:
        raise ProviderConfigurationError(
            f'OpenAI prompt cache key length {len(key)} exceeds '
            f'{OPENAI_PROMPT_CACHE_KEY_MAX_CHARS}.'
        )
    return key


def sanitize_openai_request_parameters_v12(
    parameters: Mapping[str, Any],
) -> tuple[dict[str, Any], dict[str, Any]]:
    """Last-mile firewall: forbidden sampling fields can never reach OpenAI."""
    safe = dict(parameters)
    removed: dict[str, Any] = {}
    for key in sorted(OPENAI_FORBIDDEN_TRANSPORT_PARAMETERS):
        if key in safe:
            removed[key] = safe.pop(key)

    unknown = set(safe) - OPENAI_PARSE_ALLOWED_PARAMETERS
    if unknown:
        raise ProviderConfigurationError(
            'OpenAI request contains unregistered transport parameter(s): '
            f'{sorted(unknown)}. Update the protocol rather than sending them silently.'
        )
    if 'temperature' in safe or 'seed' in safe:
        raise ProviderConfigurationError('OpenAI parameter firewall invariant failed.')
    return safe, removed


def build_openai_request_parameters_v12(
    config: ProviderModelConfig,
    package: PromptPackage,
    seed: int,
) -> dict[str, Any]:
    # IMPORTANT: temperature and seed are intentionally absent. `seed` remains in
    # the function signature because it is part of the common runner contract and
    # is logged as repeat metadata, not transmitted to OpenAI.
    raw_parameters: dict[str, Any] = {
        'model': config.model_id,
        'input': [
            {
                'role': 'developer',
                'content': [{'type': 'input_text', 'text': package.system_prompt}],
            },
            {
                'role': 'user',
                'content': [{'type': 'input_text', 'text': package.user_prompt}],
            },
        ],
        'text_format': package.schema_model,
        'max_output_tokens': (
            MAX_STAGE2_OUTPUT_TOKENS
            if package.schema_id == 'stage2_target'
            else config.max_output_tokens
        ),
        'prompt_cache_key': openai_prompt_cache_key(package),
        'store': False,
    }
    if config.reasoning_payload is not None:
        raw_parameters['reasoning'] = dict(config.reasoning_payload)
    safe, removed = sanitize_openai_request_parameters_v12(raw_parameters)
    if removed:
        raise ProviderConfigurationError(
            f'OpenAI builder unexpectedly produced forbidden fields: {sorted(removed)}'
        )
    return safe


def call_openai_provider_v12(
    config: ProviderModelConfig,
    package: PromptPackage,
    seed: int,
) -> ProviderCallResult:
    assert_api_execution_allowed('openai')
    client = get_openai_client()
    built_parameters = build_openai_request_parameters_v12(config, package, seed)

    # Apply the firewall a second time at the exact network boundary. This protects
    # against accidental wrapper mutation and makes the invariant auditable.
    request_parameters, removed_at_dispatch = sanitize_openai_request_parameters_v12(
        built_parameters
    )

    started = time.perf_counter()
    try:
        response = client.responses.parse(**request_parameters)
    except Exception as exc:
        message = str(exc).casefold()
        if 'temperature' in message or "param': 'temperature'" in message:
            raise ProviderConfigurationError(
                'OpenAI reported an unsupported temperature parameter even though v1.2 '
                f'transmitted only these keys: {sorted(request_parameters)}. This almost '
                'certainly indicates a stale notebook cell/kernel or an external wrapper. '
                'Restart the kernel and run the v1.2 setup and OpenAI smoke-test cells.'
            ) from exc
        raise
    latency = time.perf_counter() - started

    parsed = response.output_parsed
    if parsed is None:
        raise ProviderProtocolError(
            'OpenAI returned no output_parsed object. '
            f"status={safe_attr(response, 'status')!r}; "
            f"incomplete_details={safe_attr(response, 'incomplete_details')!r}"
        )
    parsed_dict = parsed.model_dump() if isinstance(parsed, BaseModel) else dict(parsed)
    validated = package.schema_model.model_validate(parsed_dict).model_dump()

    input_tokens = safe_attr(response, 'usage.input_tokens')
    output_tokens = safe_attr(response, 'usage.output_tokens')
    total_tokens = safe_attr(response, 'usage.total_tokens')
    reasoning_tokens = safe_attr(response, 'usage.output_tokens_details.reasoning_tokens', 0)
    cached_tokens = safe_attr(response, 'usage.input_tokens_details.cached_tokens', 0)
    cache_write_tokens = safe_attr(response, 'usage.input_tokens_details.cache_write_tokens', 0)
    raw_text = safe_attr(response, 'output_text') or canonical_json(validated)

    return ProviderCallResult(
        provider='openai',
        requested_model=config.model_id,
        returned_model=safe_attr(response, 'model'),
        request_id=safe_attr(response, 'id'),
        raw_response_text=raw_text,
        parsed_output=validated,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=total_tokens,
        reasoning_tokens=reasoning_tokens,
        cached_input_tokens=cached_tokens,
        cache_creation_input_tokens=cache_write_tokens,
        stop_reason=safe_attr(response, 'status'),
        latency_seconds=latency,
        accepted_parameters={
            'adapter_revision': ADAPTER_REVISION,
            'temperature_requested': config.temperature_requested,
            'temperature_sent': None,
            'temperature_transport': config.temperature_transport,
            'reasoning': config.reasoning_payload,
            'max_output_tokens': request_parameters['max_output_tokens'],
            'prompt_cache_key': request_parameters['prompt_cache_key'],
            'prompt_cache_key_length': len(request_parameters['prompt_cache_key']),
            'store': request_parameters['store'],
            'provider_seed_supported': False,
            'provider_seed_sent': False,
            'provider_seed_value': None,
            'experiment_seed_local_only': seed,
            'transport_parameter_keys': sorted(request_parameters),
            'parameter_firewall_removed': removed_at_dispatch,
        },
        provider_metadata={
            'sdk_version': package_version('openai'),
            'incomplete_details': safe_attr(response, 'incomplete_details'),
            'usage': safe_attr(response, 'usage'),
        },
    )


# Backward-readable aliases inside this notebook; live dispatch uses the v1.2 names.
build_openai_request_parameters = build_openai_request_parameters_v12
call_openai_provider = call_openai_provider_v12
build_openai_request_parameters_v12.__adapter_revision__ = ADAPTER_REVISION
call_openai_provider_v12.__adapter_revision__ = ADAPTER_REVISION

In [26]:
_ANTHROPIC_CLIENT = None
ANTHROPIC_PARSE_ALLOWED_PARAMETERS = frozenset({
    'model', 'max_tokens', 'temperature', 'system', 'messages', 'output_format',
})
ANTHROPIC_FORBIDDEN_TOP_LEVEL_PARAMETERS = frozenset({
    'cache_control', 'seed', 'thinking', 'top_p', 'top_k',
})


def get_anthropic_client():
    global _ANTHROPIC_CLIENT
    if _ANTHROPIC_CLIENT is None:
        from anthropic import Anthropic
        _ANTHROPIC_CLIENT = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
    return _ANTHROPIC_CLIENT


def sanitize_anthropic_request_parameters_v12(
    parameters: Mapping[str, Any],
) -> tuple[dict[str, Any], dict[str, Any]]:
    safe = dict(parameters)
    removed: dict[str, Any] = {}
    for key in sorted(ANTHROPIC_FORBIDDEN_TOP_LEVEL_PARAMETERS):
        if key in safe:
            removed[key] = safe.pop(key)
    unknown = set(safe) - ANTHROPIC_PARSE_ALLOWED_PARAMETERS
    if unknown:
        raise ProviderConfigurationError(
            f'Anthropic request contains unregistered parameter(s): {sorted(unknown)}'
        )
    if safe.get('temperature') != 0.0:
        raise ProviderConfigurationError(
            f"Anthropic protocol requires temperature=0.0; observed {safe.get('temperature')!r}."
        )
    return safe, removed


def build_anthropic_request_parameters_v12(
    config: ProviderModelConfig,
    package: PromptPackage,
    seed: int,
) -> dict[str, Any]:
    if config.temperature_sent is None:
        raise ProviderConfigurationError('Anthropic temperature_sent cannot be None in this protocol.')
    raw_parameters: dict[str, Any] = {
        'model': config.model_id,
        'max_tokens': (
            MAX_STAGE2_OUTPUT_TOKENS
            if package.schema_id == 'stage2_target'
            else config.max_output_tokens
        ),
        'temperature': config.temperature_sent,
        'system': [
            {
                'type': 'text',
                'text': package.system_prompt,
                'cache_control': {'type': 'ephemeral'},
            }
        ],
        'messages': [{'role': 'user', 'content': package.user_prompt}],
        'output_format': package.schema_model,
    }
    # Extended thinking is deliberately disabled: no `thinking` field is sent.
    safe, removed = sanitize_anthropic_request_parameters_v12(raw_parameters)
    if removed:
        raise ProviderConfigurationError(
            f'Anthropic builder unexpectedly produced forbidden fields: {sorted(removed)}'
        )
    return safe


def call_anthropic_provider_v12(
    config: ProviderModelConfig,
    package: PromptPackage,
    seed: int,
) -> ProviderCallResult:
    assert_api_execution_allowed('anthropic')
    client = get_anthropic_client()
    built_parameters = build_anthropic_request_parameters_v12(config, package, seed)
    request_parameters, removed_at_dispatch = sanitize_anthropic_request_parameters_v12(
        built_parameters
    )

    if not callable(getattr(client.messages, 'parse', None)):
        raise ProviderConfigurationError(
            'Installed anthropic SDK does not expose client.messages.parse(). '
            'Upgrade with `%pip install -U anthropic`, restart the kernel, and rerun the smoke test.'
        )

    started = time.perf_counter()
    response = client.messages.parse(**request_parameters)
    latency = time.perf_counter() - started
    parsed = safe_attr(response, 'parsed_output')
    if parsed is None:
        raise ProviderProtocolError(
            'Anthropic returned no parsed_output object. '
            f"stop_reason={safe_attr(response, 'stop_reason')!r}; "
            f"content_types={[safe_attr(x, 'type') for x in (safe_attr(response, 'content', []) or [])]!r}"
        )
    parsed_dict = parsed.model_dump() if isinstance(parsed, BaseModel) else dict(parsed)
    validated = package.schema_model.model_validate(parsed_dict).model_dump()

    content_texts = [
        safe_attr(block, 'text', '')
        for block in (safe_attr(response, 'content', []) or [])
        if safe_attr(block, 'type') == 'text'
    ]
    raw_text = '\n'.join(text for text in content_texts if text) or canonical_json(validated)
    input_tokens = safe_attr(response, 'usage.input_tokens')
    output_tokens = safe_attr(response, 'usage.output_tokens')
    cache_read_tokens = safe_attr(response, 'usage.cache_read_input_tokens', 0) or 0
    cache_creation_tokens = safe_attr(response, 'usage.cache_creation_input_tokens', 0) or 0
    total_tokens = (
        (input_tokens or 0) + (output_tokens or 0) + cache_read_tokens + cache_creation_tokens
        if any(x is not None for x in [input_tokens, output_tokens])
        else None
    )

    return ProviderCallResult(
        provider='anthropic',
        requested_model=config.model_id,
        returned_model=safe_attr(response, 'model'),
        request_id=safe_attr(response, 'id'),
        raw_response_text=raw_text,
        parsed_output=validated,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=total_tokens,
        reasoning_tokens=0,
        cached_input_tokens=cache_read_tokens,
        cache_creation_input_tokens=cache_creation_tokens,
        stop_reason=safe_attr(response, 'stop_reason'),
        latency_seconds=latency,
        accepted_parameters={
            'adapter_revision': ADAPTER_REVISION,
            'temperature_requested': config.temperature_requested,
            'temperature_sent': request_parameters['temperature'],
            'thinking_sent': False,
            'max_tokens': request_parameters['max_tokens'],
            'cache_control_location': 'system[0].cache_control',
            'cache_control': {'type': 'ephemeral'},
            'provider_seed_supported': False,
            'provider_seed_sent': False,
            'provider_seed_value': None,
            'experiment_seed_local_only': seed,
            'transport_parameter_keys': sorted(request_parameters),
            'parameter_firewall_removed': removed_at_dispatch,
        },
        provider_metadata={
            'sdk_version': package_version('anthropic'),
            'usage': safe_attr(response, 'usage'),
            'stop_sequence': safe_attr(response, 'stop_sequence'),
        },
    )


build_anthropic_request_parameters = build_anthropic_request_parameters_v12
call_anthropic_provider = call_anthropic_provider_v12
build_anthropic_request_parameters_v12.__adapter_revision__ = ADAPTER_REVISION
call_anthropic_provider_v12.__adapter_revision__ = ADAPTER_REVISION

In [27]:
_GEMINI_CLIENT = None


def get_gemini_client():
    global _GEMINI_CLIENT
    if _GEMINI_CLIENT is None:
        from google import genai
        api_key = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')
        _GEMINI_CLIENT = genai.Client(api_key=api_key)
    return _GEMINI_CLIENT


def build_gemini_request_manifest_v12(
    config: ProviderModelConfig,
    package: PromptPackage,
    seed: int,
) -> dict[str, Any]:
    if config.temperature_sent is None:
        raise ProviderConfigurationError('Gemini temperature_sent cannot be None in this protocol.')
    if config.native_seed_enabled and not config.native_seed_supported:
        raise ProviderConfigurationError('Gemini native seed is enabled but marked unsupported.')
    return {
        'model': config.model_id,
        'temperature': config.temperature_sent,
        'max_output_tokens': (
            MAX_STAGE2_OUTPUT_TOKENS
            if package.schema_id == 'stage2_target'
            else config.max_output_tokens
        ),
        'thinking_level': (config.reasoning_payload or {}).get('thinking_level', 'low'),
        'response_mime_type': 'application/json',
        'response_json_schema_sha256': package.schema_sha256,
        'seed': seed if config.native_seed_enabled else None,
        'system_instruction_sha256': sha256_text(package.system_prompt),
    }


def build_gemini_generation_config_v12(
    config: ProviderModelConfig,
    package: PromptPackage,
    seed: int,
):
    from google.genai import types

    manifest = build_gemini_request_manifest_v12(config, package, seed)
    try:
        thinking_config = types.ThinkingConfig(
            thinking_level=manifest['thinking_level']
        )
        kwargs: dict[str, Any] = {
            'system_instruction': package.system_prompt,
            'temperature': manifest['temperature'],
            'max_output_tokens': manifest['max_output_tokens'],
            'thinking_config': thinking_config,
            'response_mime_type': manifest['response_mime_type'],
            'response_json_schema': package.schema_model.model_json_schema(),
        }
        if config.native_seed_enabled:
            kwargs['seed'] = seed
        generate_config = types.GenerateContentConfig(**kwargs)
    except Exception as exc:
        raise ProviderConfigurationError(
            'Installed google-genai SDK could not construct the registered Gemini 3.1 '
            'Flash-Lite configuration (temperature, low thinking, JSON schema, and seed). '
            'Upgrade with `%pip install -U google-genai`, restart the kernel, and rerun '
            f'the Gemini smoke test. Original error: {type(exc).__name__}: {exc}'
        ) from exc
    return generate_config, manifest


def call_gemini_provider_v12(
    config: ProviderModelConfig,
    package: PromptPackage,
    seed: int,
) -> ProviderCallResult:
    assert_api_execution_allowed('gemini')
    client = get_gemini_client()
    generate_config, manifest = build_gemini_generation_config_v12(config, package, seed)

    started = time.perf_counter()
    response = client.models.generate_content(
        model=config.model_id,
        contents=package.user_prompt,
        config=generate_config,
    )
    latency = time.perf_counter() - started
    raw_text = safe_attr(response, 'text')
    if not raw_text:
        raise ProviderProtocolError(
            'Gemini returned empty response.text. '
            f"finish_reason={safe_attr(response, 'candidates.0.finish_reason')!r}; "
            f"prompt_feedback={safe_attr(response, 'prompt_feedback')!r}"
        )
    try:
        parsed_dict = json.loads(raw_text)
    except json.JSONDecodeError as exc:
        raise ProviderProtocolError(f'Gemini response was not JSON: {exc}') from exc
    validated = package.schema_model.model_validate(parsed_dict).model_dump()
    usage = safe_attr(response, 'usage_metadata')
    input_tokens = safe_attr(usage, 'prompt_token_count')
    output_tokens = safe_attr(usage, 'candidates_token_count')
    total_tokens = safe_attr(usage, 'total_token_count')
    cached_tokens = safe_attr(usage, 'cached_content_token_count', 0)
    reasoning_tokens = safe_attr(usage, 'thoughts_token_count', 0)
    finish_reason = safe_attr(response, 'candidates.0.finish_reason')

    return ProviderCallResult(
        provider='gemini',
        requested_model=config.model_id,
        returned_model=safe_attr(response, 'model_version') or config.model_id,
        request_id=safe_attr(response, 'response_id'),
        raw_response_text=raw_text,
        parsed_output=validated,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=total_tokens,
        reasoning_tokens=reasoning_tokens,
        cached_input_tokens=cached_tokens,
        cache_creation_input_tokens=0,
        stop_reason=str(finish_reason) if finish_reason is not None else None,
        latency_seconds=latency,
        accepted_parameters={
            'adapter_revision': ADAPTER_REVISION,
            'temperature_requested': config.temperature_requested,
            'temperature_sent': manifest['temperature'],
            'thinking_level': manifest['thinking_level'],
            'max_output_tokens': manifest['max_output_tokens'],
            'response_mime_type': manifest['response_mime_type'],
            'response_json_schema_sha256': manifest['response_json_schema_sha256'],
            'provider_seed_supported': config.native_seed_supported,
            'provider_seed_sent': config.native_seed_enabled,
            'provider_seed_value': manifest['seed'],
            'experiment_seed_local_only': None if config.native_seed_enabled else seed,
        },
        provider_metadata={
            'sdk_version': package_version('google-genai'),
            'usage_metadata': usage,
            'prompt_feedback': safe_attr(response, 'prompt_feedback'),
        },
    )


build_gemini_request_manifest = build_gemini_request_manifest_v12
call_gemini_provider = call_gemini_provider_v12
build_gemini_request_manifest_v12.__adapter_revision__ = ADAPTER_REVISION
build_gemini_generation_config_v12.__adapter_revision__ = ADAPTER_REVISION
call_gemini_provider_v12.__adapter_revision__ = ADAPTER_REVISION

In [28]:
PROVIDER_CALLS: dict[str, Callable[[ProviderModelConfig,PromptPackage,int],ProviderCallResult]] = {
    'openai': call_openai_provider_v12,
    'anthropic': call_anthropic_provider_v12,
    'gemini': call_gemini_provider_v12,
}
PROVIDER_REQUEST_BUILDERS: dict[str, Callable[..., Any]] = {
    'openai': build_openai_request_parameters_v12,
    'anthropic': build_anthropic_request_parameters_v12,
    'gemini': build_gemini_request_manifest_v12,
}


def assert_adapter_revision(provider: str) -> None:
    adapter = PROVIDER_CALLS[provider]
    observed = getattr(adapter, '__adapter_revision__', None)
    if observed != ADAPTER_REVISION:
        raise ProviderConfigurationError(
            f'{provider}: stale adapter revision {observed!r}; expected {ADAPTER_REVISION!r}. '
            'Restart the kernel and rerun the v1.2 adapter cells.'
        )


def dispatch_provider_call(
    config: ProviderModelConfig,
    package: PromptPackage,
    seed: int,
) -> ProviderCallResult:
    if USE_MOCK_PROVIDER:
        return call_mock_provider(config, package, seed)
    assert_adapter_revision(config.provider)
    return PROVIDER_CALLS[config.provider](config, package, seed)


def build_provider_parameter_preflight() -> pd.DataFrame:
    row = canonicalize_benchmark(synthetic_benchmark()).iloc[2]
    condition = next(c for c in PHASE1_CONDITIONS if c.condition_id == 'P3_D3_ADAPTIVE')
    package = build_prompt_package(row, condition, stage='joint')

    openai_payload = build_openai_request_parameters_v12(
        MODEL_CONFIGS['openai'], package, DISCOVERY_SEED
    )
    injected_openai = dict(openai_payload)
    injected_openai.update({'temperature': 0.0, 'seed': DISCOVERY_SEED})
    sanitized_openai, removed_openai = sanitize_openai_request_parameters_v12(
        injected_openai
    )

    anthropic_payload = build_anthropic_request_parameters_v12(
        MODEL_CONFIGS['anthropic'], package, DISCOVERY_SEED
    )
    anthropic_system = anthropic_payload.get('system', [])

    gemini_manifest = build_gemini_request_manifest_v12(
        MODEL_CONFIGS['gemini'], package, DISCOVERY_SEED
    )

    rows = [
        {
            'provider': 'openai',
            'passed': (
                'temperature' not in openai_payload
                and 'seed' not in openai_payload
                and not (set(openai_payload) - OPENAI_PARSE_ALLOWED_PARAMETERS)
                and set(removed_openai) == {'temperature', 'seed'}
                and 'temperature' not in sanitized_openai
                and 'seed' not in sanitized_openai
            ),
            'transport_keys': sorted(openai_payload),
            'temperature_sent': None,
            'native_seed_sent': False,
            'firewall_probe_removed': sorted(removed_openai),
        },
        {
            'provider': 'anthropic',
            'passed': (
                anthropic_payload.get('temperature') == 0.0
                and 'seed' not in anthropic_payload
                and 'thinking' not in anthropic_payload
                and 'cache_control' not in anthropic_payload
                and isinstance(anthropic_system, list)
                and bool(anthropic_system)
                and anthropic_system[0].get('cache_control') == {'type': 'ephemeral'}
            ),
            'transport_keys': sorted(anthropic_payload),
            'temperature_sent': anthropic_payload.get('temperature'),
            'native_seed_sent': False,
            'firewall_probe_removed': [],
        },
        {
            'provider': 'gemini',
            'passed': (
                gemini_manifest['temperature'] == 0.0
                and gemini_manifest['thinking_level'] == 'low'
                and gemini_manifest['response_mime_type'] == 'application/json'
                and (
                    (not MODEL_CONFIGS['gemini'].native_seed_enabled and gemini_manifest['seed'] is None)
                    or gemini_manifest['seed'] == DISCOVERY_SEED
                )
            ),
            'transport_keys': sorted(gemini_manifest),
            'temperature_sent': gemini_manifest['temperature'],
            'native_seed_sent': MODEL_CONFIGS['gemini'].native_seed_enabled,
            'firewall_probe_removed': [],
        },
    ]
    return pd.DataFrame(rows)


PROVIDER_PARAMETER_PREFLIGHT = build_provider_parameter_preflight()
PROVIDER_PARAMETER_PREFLIGHT.to_csv(
    CONFIG_ROOT / 'provider_parameter_preflight.csv', index=False
)
display(PROVIDER_PARAMETER_PREFLIGHT)
if not PROVIDER_PARAMETER_PREFLIGHT['passed'].all():
    raise ProviderConfigurationError(
        'Provider parameter preflight failed:\n' +
        PROVIDER_PARAMETER_PREFLIGHT.to_string(index=False)
    )

PROVIDER_ADAPTER_AUDIT = pd.DataFrame([
    {
        'provider': provider,
        'model_id': config.model_id,
        'adapter': PROVIDER_CALLS[provider].__name__,
        'adapter_revision': getattr(PROVIDER_CALLS[provider], '__adapter_revision__', None),
        'temperature_requested': config.temperature_requested,
        'temperature_sent': config.temperature_sent,
        'native_seed_supported': config.native_seed_supported,
        'native_seed_enabled': config.native_seed_enabled,
        'reasoning_label': config.reasoning_label,
        'api_transport': config.api_transport,
        'structured_output_transport': config.structured_output_transport,
        'prompt_caching': config.prompt_caching,
        'sdk_version': package_version({
            'openai':'openai', 'anthropic':'anthropic', 'gemini':'google-genai'
        }[provider]),
    }
    for provider, config in MODEL_CONFIGS.items()
])
display(PROVIDER_ADAPTER_AUDIT)

,provider,passed,transport_keys,temperature_sent,native_seed_sent,firewall_probe_removed
0,openai,True,"[input, max_output_tokens, model, prompt_cache_key, reasoning, store, text_format]",NaN,False,"[seed, temperature]"
1,anthropic,True,"[max_tokens, messages, model, output_format, system, temperature]",0.0,False,[]
2,gemini,True,"[max_output_tokens, model, response_json_schema_sha256, response_mime_type, seed, system_instruction_sha256, temperature, thinking_level]",0.0,True,[]


,provider,model_id,adapter,adapter_revision,temperature_requested,temperature_sent,native_seed_supported,native_seed_enabled,reasoning_label,api_transport,structured_output_transport,prompt_caching,sdk_version
0,openai,gpt-5.6-terra,call_openai_provider_v12,api_hardened_2026_07_22_r1,0.0,NaN,False,False,reasoning.effort=low,OpenAI Responses API: client.responses.parse,Pydantic text_format through Responses parse helper,hashed prompt_cache_key (63 chars); exact static prefix first,2.48.0
1,anthropic,claude-haiku-4-5-20251001,call_anthropic_provider_v12,api_hardened_2026_07_22_r1,0.0,0.0,False,False,lowest setting: extended thinking disabled,Anthropic Messages API: client.messages.parse,Pydantic output_format convenience parameter,explicit ephemeral cache breakpoint on the static system block,0.119.0
2,gemini,gemini-3.1-flash-lite,call_gemini_provider_v12,api_hardened_2026_07_22_r1,0.0,0.0,True,True,thinking_level=low,Google Gen AI SDK: models.generate_content,response_mime_type=application/json + response_json_schema,implicit prefix caching; cached token usage logged when reported,2.14.0


## 13. Isolated provider smoke tests

Run these cells one at a time before enabling a full paid phase. Each smoke test performs a **local SDK feature preflight first**, then at most one network request. A successful smoke signature is required before that provider can run a full phase.

The OpenAI smoke record explicitly lists the exact transport keys. It must show no `temperature` and no `seed`. The Claude record must show temperature zero, no thinking field, and a cache breakpoint on the system block. The Gemini record must show temperature zero, low thinking, and the registered native seed.

In [29]:
def callable_fingerprint(function: Callable[..., Any]) -> str:
    try:
        source = inspect.getsource(function)
    except (OSError, TypeError):
        code = getattr(function, '__code__', None)
        source = repr((getattr(code, 'co_code', b''), getattr(code, 'co_consts', ())))
    return sha256_text(source)


def smoke_fixture(provider: str) -> tuple[ProviderModelConfig, PromptPackage]:
    config = MODEL_CONFIGS[provider]
    row = canonicalize_benchmark(synthetic_benchmark()).iloc[2]
    condition = next(c for c in PHASE1_CONDITIONS if c.condition_id == 'P3_D3_ADAPTIVE')
    return config, build_prompt_package(row, condition, stage='joint')


def provider_sdk_preflight(provider: str) -> dict[str, Any]:
    """Validate local SDK capabilities without making a network request."""
    config, package = smoke_fixture(provider)
    if USE_MOCK_PROVIDER:
        return {
            'provider': provider,
            'passed': True,
            'mock': True,
            'sdk_version': None,
            'adapter_revision': ADAPTER_REVISION,
        }

    if provider == 'openai':
        client = get_openai_client()
        if not callable(getattr(client.responses, 'parse', None)):
            raise ProviderConfigurationError(
                'Installed openai SDK has no client.responses.parse(). '
                'Upgrade openai, restart the kernel, and rerun this cell.'
            )
        payload = build_openai_request_parameters_v12(config, package, DISCOVERY_SEED)
        payload, removed = sanitize_openai_request_parameters_v12(payload)
        if removed or 'temperature' in payload or 'seed' in payload:
            raise ProviderConfigurationError('OpenAI local payload firewall failed.')
        details = {'transport_parameter_keys': sorted(payload)}
        sdk_name = 'openai'

    elif provider == 'anthropic':
        client = get_anthropic_client()
        if not callable(getattr(client.messages, 'parse', None)):
            raise ProviderConfigurationError(
                'Installed anthropic SDK has no client.messages.parse(). '
                'Upgrade anthropic, restart the kernel, and rerun this cell.'
            )
        payload = build_anthropic_request_parameters_v12(config, package, DISCOVERY_SEED)
        payload, removed = sanitize_anthropic_request_parameters_v12(payload)
        if removed:
            raise ProviderConfigurationError('Anthropic local payload firewall removed fields unexpectedly.')
        details = {'transport_parameter_keys': sorted(payload)}
        sdk_name = 'anthropic'

    elif provider == 'gemini':
        generation_config, manifest = build_gemini_generation_config_v12(
            config, package, DISCOVERY_SEED
        )
        if config.native_seed_enabled and getattr(generation_config, 'seed', None) != DISCOVERY_SEED:
            raise ProviderConfigurationError(
                'Gemini SDK did not retain the registered native seed in GenerateContentConfig.'
            )
        details = {
            'transport_parameter_keys': sorted(manifest),
            'native_seed_value': manifest['seed'],
        }
        sdk_name = 'google-genai'
    else:
        raise KeyError(provider)

    return {
        'provider': provider,
        'passed': True,
        'mock': False,
        'sdk_version': package_version(sdk_name),
        'adapter_revision': ADAPTER_REVISION,
        **details,
    }


def smoke_signature(provider: str) -> dict[str, Any]:
    config, package = smoke_fixture(provider)
    builder = PROVIDER_REQUEST_BUILDERS[provider]
    provider_preflight = PROVIDER_PARAMETER_PREFLIGHT.set_index('provider').loc[provider].to_dict()
    return {
        'protocol_version': PROTOCOL_VERSION,
        'adapter_revision': ADAPTER_REVISION,
        'provider': provider,
        'model_config_sha256': model_config_hash(config),
        'prompt_sha256': package.prompt_sha256,
        'schema_sha256': package.schema_sha256,
        'adapter_sha256': callable_fingerprint(PROVIDER_CALLS[provider]),
        'request_builder_sha256': callable_fingerprint(builder),
        'parameter_preflight_sha256': sha256_text(canonical_json(provider_preflight)),
    }


def smoke_status_path(provider: str) -> Path:
    return provider_phase_directory('smoke_tests', provider) / 'smoke_test.json'


def load_smoke_status(provider: str) -> dict[str, Any]:
    path = smoke_status_path(provider)
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return {}


def smoke_status_is_current(provider: str, status: Optional[dict[str, Any]]=None) -> tuple[bool, str]:
    status = status or load_smoke_status(provider)
    if not status:
        return False, 'smoke test has not been run for this protocol'
    expected = smoke_signature(provider)
    if not status.get('passed'):
        return False, f"last smoke test failed: {status.get('error_type')}: {status.get('error_message')}"
    for key, value in expected.items():
        if status.get(key) != value:
            return False, f'smoke signature mismatch for {key}'
    return True, 'current smoke test passed'


def run_provider_smoke_test(provider: str) -> dict[str, Any]:
    config, package = smoke_fixture(provider)
    sdk_preflight = provider_sdk_preflight(provider) if REQUIRE_SDK_PREFLIGHT else {
        'provider': provider, 'passed': True, 'skipped': True
    }
    result = dispatch_provider_call(config, package, seed=DISCOVERY_SEED)
    return {
        **smoke_signature(provider),
        'passed': True,
        'started_at_utc': datetime.now(timezone.utc).isoformat(),
        'completed_at_utc': datetime.now(timezone.utc).isoformat(),
        'requested_model': result.requested_model,
        'returned_model': result.returned_model,
        'request_id': result.request_id,
        'schema_id': package.schema_id,
        'schema_valid': True,
        'prediction': result.parsed_output.get('unlearning_present'),
        'input_tokens': result.input_tokens,
        'output_tokens': result.output_tokens,
        'cached_input_tokens': result.cached_input_tokens,
        'cache_creation_input_tokens': result.cache_creation_input_tokens,
        'reasoning_tokens': result.reasoning_tokens,
        'latency_seconds': result.latency_seconds,
        'accepted_parameters': result.accepted_parameters,
        'sdk_preflight': sdk_preflight,
        'mock': USE_MOCK_PROVIDER,
    }


def execute_provider_smoke_test(provider: str) -> pd.DataFrame:
    existing = load_smoke_status(provider)
    current, reason = smoke_status_is_current(provider, existing)
    if current and not FORCE_PROVIDER_SMOKE_TESTS:
        print(f'{provider}: reusing current successful smoke test.')
        return pd.DataFrame([existing])

    started = datetime.now(timezone.utc).isoformat()
    try:
        record = run_provider_smoke_test(provider)
    except Exception as exc:
        record = {
            **smoke_signature(provider),
            'passed': False,
            'started_at_utc': started,
            'completed_at_utc': datetime.now(timezone.utc).isoformat(),
            'error_classification': error_classification(exc),
            'error_type': type(exc).__name__,
            'error_message': str(exc),
            'status_code': exception_status_code(exc),
            'error_fingerprint': error_fingerprint(exc),
            'traceback': traceback.format_exc(),
            'mock': USE_MOCK_PROVIDER,
        }
    atomic_write_text(smoke_status_path(provider), json.dumps(record, indent=2, default=str))
    if not record['passed']:
        print(f"{provider} smoke test failed: {record['error_type']}: {record['error_message']}")
        if RAISE_PROVIDER_CELL_EXCEPTIONS:
            raise RuntimeError(record['error_message'])
    return pd.DataFrame([{key: value for key, value in record.items() if key != 'traceback'}])


def assert_provider_smoke_passed(provider: str) -> None:
    if not REQUIRE_PROVIDER_SMOKE_PASS:
        return
    passed, reason = smoke_status_is_current(provider)
    if not passed:
        raise ProviderConfigurationError(
            f'{provider}: paid phase blocked because {reason}. Run its isolated smoke-test cell first.'
        )


def provider_execution_ready(provider: str) -> bool:
    if provider not in ENABLED_PROVIDERS:
        print(f'{provider}: skipped because provider is disabled.')
        return False
    if not USE_MOCK_PROVIDER and not (
        RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION
    ):
        print(f'{provider}: skipped because API execution is disabled or confirmation is invalid.')
        return False
    if REQUIRE_PROVIDER_SMOKE_PASS:
        passed, reason = smoke_status_is_current(provider)
        if not passed:
            print(f'{provider}: skipped because {reason}.')
            return False
    return True

In [ ]:
# ISOLATED OPENAI SMOKE TEST
OPENAI_SMOKE = pd.DataFrame()
if USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION):
    OPENAI_SMOKE = execute_provider_smoke_test('openai')
    display(OPENAI_SMOKE)
else:
    print('OpenAI smoke test skipped; API execution is disabled.')

,protocol_version,adapter_revision,provider,model_config_sha256,prompt_sha256,schema_sha256,adapter_sha256,request_builder_sha256,parameter_preflight_sha256,passed,started_at_utc,completed_at_utc,requested_model,returned_model,request_id,schema_id,schema_valid,prediction,input_tokens,output_tokens,cached_input_tokens,cache_creation_input_tokens,reasoning_tokens,latency_seconds,accepted_parameters,sdk_preflight,mock
0,prelangchain_ab_v1_2_api_hardened,api_hardened_2026_07_22_r1,openai,9f79900c8bed0b7477c913b17364bb219bc9284c1c9b1d9099c7ce1f2cc084bc,6db2ee701b7b22588e44b8b0a4bde991f7bd9fddacaaa7c3e65a758b1d52b915,160ef51b848bbba94ee5996a63b20821792de7b7319a848da6741e11e321f8b6,7caf2109f1790a75ad2a08f76f6dd3d5f0af4a95bfe8fbeb313fb47d1811bdb4,f07b7adda6057c93ba345c8191fe55ad464253ec338520f025ea5410288f9009,cb9ae3f18f28080221f1803eb345e5081484103355dc2e77140552a71d752637,True,2026-07-22T20:43:58.657440+00:00,2026-07-22T20:43:58.657456+00:00,gpt-5.6-terra,gpt-5.6-terra,resp_087e953332924f4b016a612b8cbc708191af24c8c6b25ef55f,checklist_joint,True,False,1115,138,0,1112,0,2.774257,"{'adapter_revision': 'api_hardened_2026_07_22_r1', 'temperature_requested': 0.0, 'temperature_sent': None, 'temperature_transport': 'temperature is unsupported by GPT-5.6 Terra in this Responses configuration; the fi...","{'provider': 'openai', 'passed': True, 'mock': False, 'sdk_version': '2.47.0', 'adapter_revision': 'api_hardened_2026_07_22_r1', 'transport_parameter_keys': ['input', 'max_output_tokens', 'model', 'prompt_cache_key',...",False


In [ ]:
# ISOLATED ANTHROPIC SMOKE TEST
ANTHROPIC_SMOKE = pd.DataFrame()
if USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION):
    ANTHROPIC_SMOKE = execute_provider_smoke_test('anthropic')
    display(ANTHROPIC_SMOKE)
else:
    print('Anthropic smoke test skipped; API execution is disabled.')

,protocol_version,adapter_revision,provider,model_config_sha256,prompt_sha256,schema_sha256,adapter_sha256,request_builder_sha256,parameter_preflight_sha256,passed,started_at_utc,completed_at_utc,requested_model,returned_model,request_id,schema_id,schema_valid,prediction,input_tokens,output_tokens,cached_input_tokens,cache_creation_input_tokens,reasoning_tokens,latency_seconds,accepted_parameters,sdk_preflight,mock
0,prelangchain_ab_v1_2_api_hardened,api_hardened_2026_07_22_r1,anthropic,8090d745697d7da125cd149fd7001c7978c6b4cafdc2156f22e73eec3857c1a2,6db2ee701b7b22588e44b8b0a4bde991f7bd9fddacaaa7c3e65a758b1d52b915,160ef51b848bbba94ee5996a63b20821792de7b7319a848da6741e11e321f8b6,a1feedcb46c1c0e2977b7413f0af71dee4d03b5f3e15aef0b94d500a938cc6fa,9091f387268b675d96cd646b1a2e28554656d8fd4dceb45d035424ccf7a39814,7c51f4ae1e5ba9022d7990cb8dc3a98a9629b726f12163f190172473b7bfca1c,True,2026-07-22T20:44:04.424770+00:00,2026-07-22T20:44:04.424786+00:00,claude-haiku-4-5-20251001,claude-haiku-4-5-20251001,msg_011CdHkKLaUSgVRcxAGne5CY,checklist_joint,True,True,1904,202,0,0,0,2.375808,"{'adapter_revision': 'api_hardened_2026_07_22_r1', 'temperature_requested': 0.0, 'temperature_sent': 0.0, 'thinking_sent': False, 'max_tokens': 1800, 'cache_control_location': 'system[0].cache_control', 'cache_contro...","{'provider': 'anthropic', 'passed': True, 'mock': False, 'sdk_version': '0.118.0', 'adapter_revision': 'api_hardened_2026_07_22_r1', 'transport_parameter_keys': ['max_tokens', 'messages', 'model', 'output_format', 's...",False


In [ ]:
# ISOLATED GEMINI SMOKE TEST
GEMINI_SMOKE = pd.DataFrame()
if USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION):
    GEMINI_SMOKE = execute_provider_smoke_test('gemini')
    display(GEMINI_SMOKE)
else:
    print('Gemini smoke test skipped; API execution is disabled.')

,protocol_version,adapter_revision,provider,model_config_sha256,prompt_sha256,schema_sha256,adapter_sha256,request_builder_sha256,parameter_preflight_sha256,passed,started_at_utc,completed_at_utc,requested_model,returned_model,request_id,schema_id,schema_valid,prediction,input_tokens,output_tokens,cached_input_tokens,cache_creation_input_tokens,reasoning_tokens,latency_seconds,accepted_parameters,sdk_preflight,mock
0,prelangchain_ab_v1_2_api_hardened,api_hardened_2026_07_22_r1,gemini,1d141db9a3fbe2d79e7d076b6e712cc6af3f7d7478e85098f1aba9ff08c4deaa,6db2ee701b7b22588e44b8b0a4bde991f7bd9fddacaaa7c3e65a758b1d52b915,160ef51b848bbba94ee5996a63b20821792de7b7319a848da6741e11e321f8b6,35bce175a16cff427f30c2ec9b1e46ccf296fb5739354f0f655114175e3bb456,351652c1d9bef697a2b82a33cf6122be6f64b853bdf39f2e5f10e813c92c0db6,f3efdb66d8cdb5180ede0eca939a34bfcbc984581bfdfb8d7ac6fb2b7bd12c3b,True,2026-07-22T20:44:09.368545+00:00,2026-07-22T20:44:09.368560+00:00,gemini-3.1-flash-lite,gemini-3.1-flash-lite,lythauHQOcDpqtsPyOyWoQw,checklist_joint,True,True,691,239,None,0,107,1.652853,"{'adapter_revision': 'api_hardened_2026_07_22_r1', 'temperature_requested': 0.0, 'temperature_sent': 0.0, 'thinking_level': 'low', 'max_output_tokens': 1800, 'response_mime_type': 'application/json', 'response_json_s...","{'provider': 'gemini', 'passed': True, 'mock': False, 'sdk_version': '2.13.0', 'adapter_revision': 'api_hardened_2026_07_22_r1', 'transport_parameter_keys': ['max_output_tokens', 'model', 'response_json_schema_sha256...",False


## 14. Pricing snapshot and cost estimation

In [30]:
# USD per one million tokens. Snapshot date is part of the manifest because
# provider prices are time-varying. Override through a new protocol version
# rather than retroactively editing a completed run.
PRICING_SNAPSHOT_DATE = '2026-07-22'
TOKEN_PRICING = {
    'openai': {
        'uncached_input': 2.50,
        'cache_write_input': 3.125,
        'cached_input': 0.25,
        'output_including_reasoning': 15.00,
    },
    'anthropic': {
        'uncached_input': 1.00,
        'cache_write_input': 1.25,
        'cached_input': 0.10,
        'output_including_reasoning': 5.00,
    },
    'gemini': {
        'uncached_input': 0.25,
        'cache_write_input': 0.25,
        'cached_input': 0.025,
        'output_including_reasoning': 1.50,
    },
}

def estimate_call_cost_usd(
    provider: str,
    input_tokens: Optional[int],
    output_tokens: Optional[int],
    cached_input_tokens: Optional[int] = 0,
    cache_creation_input_tokens: Optional[int] = 0,
) -> Optional[float]:
    if provider not in TOKEN_PRICING or input_tokens is None or output_tokens is None:
        return None
    rates = TOKEN_PRICING[provider]
    cached = max(0, int(cached_input_tokens or 0))
    created = max(0, int(cache_creation_input_tokens or 0))
    total_input = max(0, int(input_tokens or 0))
    # Provider usage semantics differ. Clamp the residual to avoid negative
    # uncached tokens if a provider reports cache writes outside input_tokens.
    uncached = max(0, total_input - cached - created)
    cost = (
        uncached * rates['uncached_input']
        + created * rates['cache_write_input']
        + cached * rates['cached_input']
        + int(output_tokens or 0) * rates['output_including_reasoning']
    ) / 1_000_000
    return float(cost)

PRICING_MANIFEST = {
    'snapshot_date': PRICING_SNAPSHOT_DATE,
    'currency': 'USD',
    'per_tokens': 1_000_000,
    'rates': TOKEN_PRICING,
    'caution': 'Estimates exclude taxes, storage, priority/flex premiums, and provider billing adjustments.',
}
(CONFIG_ROOT / 'pricing_snapshot.json').write_text(
    json.dumps(PRICING_MANIFEST, indent=2), encoding='utf-8'
)
display(pd.DataFrame(TOKEN_PRICING).T)

,uncached_input,cache_write_input,cached_input,output_including_reasoning
openai,2.50,3.125,0.250,15.0
anthropic,1.00,1.250,0.100,5.0
gemini,0.25,0.250,0.025,1.5


## 15. Generic resumable provider runner

In [31]:
def serialize_provider_result(result: ProviderCallResult) -> dict[str, Any]:
    record = asdict(result)
    record['accepted_parameters_json'] = canonical_json(record.pop('accepted_parameters'))
    record['provider_metadata_json'] = canonical_json(record.pop('provider_metadata'))
    record['parsed_output_json'] = canonical_json(record.pop('parsed_output'))
    record['response_sha256'] = sha256_text(record['raw_response_text'])
    record['estimated_cost_usd'] = estimate_call_cost_usd(
        result.provider,
        result.input_tokens,
        result.output_tokens,
        result.cached_input_tokens,
        result.cache_creation_input_tokens,
    )
    return record


def condition_manifest(condition: ConditionSpec) -> dict[str, Any]:
    definition = DEFINITIONS[condition.definition_id]
    return {
        **asdict(condition),
        'definition_name': definition.name,
        'definition_status': definition.status,
        'definition_sha256': sha256_text(definition.operational_text),
    }


def phase_log_paths(phase: str, provider: str) -> dict[str, Path]:
    directory = provider_phase_directory(phase, provider)
    return {
        'directory': directory,
        'requests': directory / 'requests.jsonl',
        'results': directory / 'results.jsonl',
        'errors': directory / 'errors.jsonl',
        'snapshot': directory / 'latest_results.csv',
        'status': directory / 'provider_status.json',
    }


def provider_results_snapshot(path: Path) -> pd.DataFrame:
    latest = list(latest_records_by_run_key(path).values())
    return pd.DataFrame(latest)


def parsed_stage1_lookup_key(
    provider: str,
    condition_id: str,
    row_id: str,
    seed: int,
    repeat_id: str,
) -> tuple[str, str, str, int, str]:
    return provider, condition_id, str(row_id), int(seed), repeat_id


def run_provider_tasks(
    provider: str,
    tasks: Sequence[TaskSpec],
    conditions: Sequence[ConditionSpec],
    stage1_outputs: Optional[dict[tuple[str,str,str,int,str], dict[str,Any]]] = None,
) -> pd.DataFrame:
    if BENCHMARK is None:
        raise RuntimeError(f'Benchmark unavailable: {BENCHMARK_LOAD_ERROR}')
    if provider not in MODEL_CONFIGS:
        raise KeyError(f'Unknown provider: {provider}')
    if provider not in ENABLED_PROVIDERS:
        raise ProviderConfigurationError(f'{provider}: provider is disabled.')
    if not tasks:
        return pd.DataFrame()

    assert_api_execution_allowed(provider)
    assert_provider_smoke_passed(provider)

    config = MODEL_CONFIGS[provider]
    condition_map = conditions_by_id(conditions)
    row_map = {str(row.row_id): row for row in BENCHMARK.itertuples(index=False)}
    paths = phase_log_paths(tasks[0].phase, provider)
    success_keys = successful_run_keys(paths['results'])
    new_calls = 0
    stopped_early = False
    stop_reason = None
    started_at = datetime.now(timezone.utc).isoformat()
    consecutive_errors = 0
    error_fingerprint_counts: Counter[str] = Counter()
    last_error_fingerprint: Optional[str] = None

    ordered_tasks = sorted(tasks, key=lambda item: (item.seed, item.sequence))
    for task in tqdm(ordered_tasks, desc=f'{provider}:{tasks[0].phase}'):
        condition = condition_map[task.condition_id]
        row_tuple = row_map.get(str(task.row_id))
        if row_tuple is None:
            raise KeyError(f'Row not found: {task.row_id}')
        row = pd.Series(row_tuple._asdict())
        stage1_output = None
        if task.stage == 'target':
            lookup = parsed_stage1_lookup_key(
                provider, task.condition_id, task.row_id, task.seed, task.repeat_id
            )
            stage1_output = (stage1_outputs or {}).get(lookup)
            if not stage1_output:
                continue
        package = build_prompt_package(row, condition, stage=task.stage, stage1_output=stage1_output)
        run_key = make_run_key(config, task, condition, package)
        if run_key in success_keys:
            continue
        if new_calls >= MAX_NEW_CALLS_PER_PROVIDER_CELL:
            stopped_early = True
            stop_reason = f'MAX_NEW_CALLS_PER_PROVIDER_CELL={MAX_NEW_CALLS_PER_PROVIDER_CELL}'
            break

        request_record = {
            'run_key': run_key,
            'protocol_version': PROTOCOL_VERSION,
            'dataset_sha256': DATASET_SHA256,
            'provider': provider,
            'requested_model': config.model_id,
            'model_config_sha256': model_config_hash(config),
            **asdict(task),
            **condition_manifest(condition),
            'schema_id': package.schema_id,
            'schema_sha256': package.schema_sha256,
            'prompt_sha256': package.prompt_sha256,
            'cache_key': package.cache_key,
            'system_prompt_sha256': sha256_text(package.system_prompt),
            'user_prompt_sha256': sha256_text(package.user_prompt),
            'system_prompt': package.system_prompt,
            'user_prompt': package.user_prompt,
            'output_schema_json': canonical_json(package.schema_model.model_json_schema()),
            'source_blocks_json': canonical_json(package.source_blocks),
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
        }
        append_jsonl(paths['requests'], request_record)
        attempt_started = datetime.now(timezone.utc).isoformat()
        try:
            result = call_with_retry(
                lambda: dispatch_provider_call(config, package, task.seed),
                seed=stable_int_seed(task.seed, provider, task.row_id, task.condition_id),
            )
            validated = package.schema_model.model_validate(result.parsed_output).model_dump()
            result.parsed_output = validated
            result_record = {
                **request_record,
                **serialize_provider_result(result),
                'status': 'ok',
                'attempt_started_at_utc': attempt_started,
                'completed_at_utc': datetime.now(timezone.utc).isoformat(),
            }
            append_jsonl(paths['results'], result_record)
            success_keys.add(run_key)
            new_calls += 1
            consecutive_errors = 0
            if REQUEST_SPACING_SECONDS > 0 and not USE_MOCK_PROVIDER:
                time.sleep(REQUEST_SPACING_SECONDS)
        except Exception as exc:
            classification = error_classification(exc)
            fingerprint = error_fingerprint(exc, classification)
            last_error_fingerprint = fingerprint
            error_fingerprint_counts[fingerprint] += 1
            consecutive_errors += 1
            error_record = {
                **request_record,
                'status': 'error',
                'error_classification': classification,
                'error_fingerprint': fingerprint,
                'error_fingerprint_count': error_fingerprint_counts[fingerprint],
                'consecutive_error_count': consecutive_errors,
                'error_type': type(exc).__name__,
                'error_message': str(exc),
                'status_code': exception_status_code(exc),
                'traceback': traceback.format_exc(),
                'attempt_started_at_utc': attempt_started,
                'completed_at_utc': datetime.now(timezone.utc).isoformat(),
            }
            append_jsonl(paths['errors'], error_record)
            append_jsonl(paths['results'], error_record)
            new_calls += 1

            breaker_reason = provider_circuit_breaker_reason(
                classification,
                error_fingerprint_counts[fingerprint],
                consecutive_errors,
            )
            if breaker_reason:
                stopped_early = True
                stop_reason = (
                    f'{breaker_reason}: {type(exc).__name__}: {exc}; '
                    f'identical={error_fingerprint_counts[fingerprint]}, '
                    f'consecutive={consecutive_errors}'
                )
                break

    snapshot = provider_results_snapshot(paths['results'])
    atomic_write_dataframe_csv(snapshot, paths['snapshot'])
    smoke_passed, smoke_reason = smoke_status_is_current(provider)
    status = {
        'provider': provider,
        'phase': tasks[0].phase,
        'started_at_utc': started_at,
        'completed_at_utc': datetime.now(timezone.utc).isoformat(),
        'tasks_requested': len(tasks),
        'new_attempts': new_calls,
        'successful_run_keys_total': len(successful_run_keys(paths['results'])),
        'smoke_passed': smoke_passed,
        'smoke_reason': smoke_reason,
        'stopped_early': stopped_early,
        'stop_reason': stop_reason,
        'last_error_fingerprint': last_error_fingerprint,
        'max_consecutive_errors': MAX_CONSECUTIVE_ERRORS,
        'max_identical_errors': MAX_IDENTICAL_ERROR_FINGERPRINTS,
        'result_log': str(paths['results']),
        'error_log': str(paths['errors']),
    }
    atomic_write_text(paths['status'], json.dumps(status, indent=2, default=str))
    if stopped_early:
        print(f'{provider} stopped early: {stop_reason}')
    return snapshot


def safe_run_provider_cell(
    provider: str,
    tasks: Sequence[TaskSpec],
    conditions: Sequence[ConditionSpec],
    stage1_outputs: Optional[dict[tuple[str,str,str,int,str], dict[str,Any]]] = None,
) -> pd.DataFrame:
    try:
        return run_provider_tasks(provider, tasks, conditions, stage1_outputs)
    except Exception as exc:
        print(f'{provider} provider cell failed without affecting other providers: {type(exc).__name__}: {exc}')
        traceback.print_exc(limit=3)
        if RAISE_PROVIDER_CELL_EXCEPTIONS:
            raise
        return pd.DataFrame()

In [32]:
def load_phase_results(phase: str, providers: Sequence[str] = ('openai','anthropic','gemini')) -> pd.DataFrame:
    frames = []
    for provider in providers:
        path = phase_log_paths(phase, provider)['results']
        frame = provider_results_snapshot(path)
        if not frame.empty:
            frames.append(frame)
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()

def parse_output_column(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    data = frame.copy()
    parsed_records = []
    for value in data.get('parsed_output_json', pd.Series([None] * len(data))):
        if isinstance(value, dict):
            parsed_records.append(value)
        elif isinstance(value, str) and value.strip():
            with contextlib.suppress(json.JSONDecodeError):
                parsed_records.append(json.loads(value))
                continue
            parsed_records.append({})
        else:
            parsed_records.append({})
    parsed = pd.json_normalize(parsed_records, sep='__').add_prefix('out__')
    return pd.concat([data.reset_index(drop=True), parsed.reset_index(drop=True)], axis=1)

def successful_predictions(phase: str) -> pd.DataFrame:
    frame = load_phase_results(phase)
    if frame.empty:
        return frame
    frame = frame[frame['status'].eq('ok')].copy()
    frame = frame.sort_values('completed_at_utc').drop_duplicates('run_key', keep='last')
    return parse_output_column(frame)

## 16. Evidence validation and protocol-rule audit

In [33]:
def quote_matches_source(quote: Any, source_scope: Any, blocks: dict[str,str]) -> tuple[bool, Optional[str]]:
    quote_text = normalize_for_match(quote)
    scope = normalize_space(source_scope)
    if not quote_text:
        return scope == 'absent', None
    scope_to_key = {
        'target':'target','previous_1':'previous_1','previous_2':'previous_2',
        'next_1':'next_1','next_2':'next_2','metadata':'metadata',
    }
    key = scope_to_key.get(scope)
    if key is None:
        return False, None
    source = normalize_for_match(blocks.get(key, ''))
    if quote_text in source:
        return True, key
    # Exact quote is the rule, but normalized Unicode/whitespace matching avoids
    # false failures caused only by PDF typography.
    return False, key

def extract_evidence_items(parsed: dict[str,Any]) -> list[dict[str,Any]]:
    if isinstance(parsed.get('evidence'), list):
        return [dict(item) for item in parsed['evidence']]
    items = []
    for key in ['prior_state','inadequacy_or_failure','departure_or_reconfiguration']:
        element = parsed.get(key)
        if isinstance(element, dict):
            items.append({
                'element': key,
                'quote': element.get('quote'),
                'source_scope': element.get('source_scope'),
                'identified': element.get('identified'),
            })
    if 'target_evidence_quote' in parsed:
        items.append({
            'element':'other',
            'quote':parsed.get('target_evidence_quote'),
            'source_scope':parsed.get('target_evidence_scope'),
            'identified':bool(parsed.get('target_evidence_quote')),
        })
    return items

def audit_prediction_evidence(record: pd.Series) -> dict[str,Any]:
    if BENCHMARK is None:
        return {}
    row_matches = BENCHMARK[BENCHMARK['row_id'].astype(str).eq(str(record['row_id']))]
    if row_matches.empty:
        return {'evidence_valid':False,'evidence_error':'row not found'}
    row = row_matches.iloc[0]
    condition = ConditionSpec(
        condition_id=record['condition_id'], phase=record['phase'],
        definition_id=record['definition_id'], prompt_style=record['prompt_style'],
        context_id=record['context_id'], workflow=record['workflow'],
        description=record.get('description',''),
    )
    blocks = source_blocks_for_context(row, condition.context_id)
    try:
        parsed = json.loads(record['parsed_output_json'])
    except Exception:
        return {'evidence_valid':False,'schema_valid':False,'evidence_error':'parsed_output_json invalid'}
    schema_id = record['schema_id']
    schema_valid = True
    try:
        SCHEMA_MODELS[schema_id].model_validate(parsed)
    except Exception as exc:
        schema_valid = False
        schema_error = str(exc)
    else:
        schema_error = None

    evidence_items = extract_evidence_items(parsed)
    quote_checks = []
    for item in evidence_items:
        valid, matched_source = quote_matches_source(
            item.get('quote'), item.get('source_scope'), blocks
        )
        quote_checks.append({**item,'valid':valid,'matched_source':matched_source})
    all_quotes_valid = all(item['valid'] for item in quote_checks) if quote_checks else True
    positive = bool(parsed.get('unlearning_present', True if schema_id == 'stage2_target' else False))
    departure_items = [
        item for item in quote_checks
        if item.get('element') == 'departure_or_reconfiguration'
    ]
    target_departure_present = any(
        item.get('valid') and item.get('source_scope') == 'target' and normalize_space(item.get('quote'))
        for item in departure_items
    )
    if schema_id == 'stage2_target':
        target_departure_present = any(
            item.get('valid') and item.get('source_scope') == 'target' and normalize_space(item.get('quote'))
            for item in quote_checks
        )
    no_joint_target_violation = True
    if schema_id in {'simple_joint','checklist_joint'}:
        if positive:
            no_joint_target_violation = parsed.get('target_type') != 'none'
        else:
            no_joint_target_violation = (
                parsed.get('target_type') == 'none' and parsed.get('agency') is None
            )
    d3_additive_violation = (
        record['definition_id'] == 'D3_provisional_adaptive'
        and positive
        and parsed.get('change_type') == 'add_capacity_only'
    )
    checklist_consistent = True
    if schema_id in {'checklist_joint','checklist_binary'}:
        required = ['prior_state','inadequacy_or_failure','departure_or_reconfiguration']
        identified = {name: bool(parsed.get(name,{}).get('identified')) for name in required}
        expected_missing = sorted(name for name, value in identified.items() if not value)
        checklist_consistent = (
            sorted(parsed.get('missing_elements',[])) == expected_missing
            and bool(parsed.get('all_required_elements_present')) == all(identified.values())
        )
    evidence_valid = all([
        schema_valid,
        all_quotes_valid,
        (not positive or target_departure_present),
        no_joint_target_violation,
        not d3_additive_violation,
        checklist_consistent,
    ])
    return {
        'schema_valid': schema_valid,
        'schema_error': schema_error,
        'evidence_quote_count': len(quote_checks),
        'all_quotes_valid': all_quotes_valid,
        'target_departure_present': target_departure_present,
        'joint_target_null_rule_valid': no_joint_target_violation,
        'd3_additive_rule_valid': not d3_additive_violation,
        'checklist_consistent': checklist_consistent,
        'evidence_valid': evidence_valid,
        'evidence_audit_json': canonical_json(quote_checks),
    }

def add_evidence_audit(predictions: pd.DataFrame) -> pd.DataFrame:
    if predictions.empty:
        return predictions.copy()
    audit = predictions.apply(audit_prediction_evidence, axis=1, result_type='expand')
    return pd.concat([predictions.reset_index(drop=True), audit.reset_index(drop=True)], axis=1)

## 17. Gold-label policy and binary evaluation

In [34]:
def complete_gold_column(column: str) -> bool:
    return bool(
        BENCHMARK is not None
        and column in BENCHMARK.columns
        and BENCHMARK[column].notna().all()
    )

def aligned_gold_column(definition_id: str) -> tuple[str,str]:
    preferred = DEFINITION_GOLD_COLUMN.get(definition_id, 'gold_final_definition')
    if complete_gold_column(preferred):
        return preferred, 'definition-aligned adjudicated gold'
    return 'gold_historical', 'historical provisional gold'

def common_final_gold_column() -> tuple[str,str]:
    if complete_gold_column('gold_final_definition'):
        return 'gold_final_definition', 'common final adjudicated gold'
    return 'gold_historical', 'historical provisional gold'

def attach_gold_and_predictions(predictions: pd.DataFrame) -> pd.DataFrame:
    if predictions.empty or BENCHMARK is None:
        return predictions.copy()
    benchmark_columns = [
        'row_id','document_title','gold_historical','gold_old_definition',
        'gold_current_definition','gold_final_definition','gold_target','gold_agency'
    ]
    data = predictions.merge(
        BENCHMARK[benchmark_columns], on='row_id', how='left', validate='many_to_one'
    )
    aligned_values, aligned_basis = [], []
    for _, row in data.iterrows():
        column, basis = aligned_gold_column(row['definition_id'])
        aligned_values.append(row.get(column))
        aligned_basis.append(basis + f' ({column})')
    common_column, common_basis = common_final_gold_column()
    data['gold_definition_aligned'] = aligned_values
    data['gold_definition_aligned_basis'] = aligned_basis
    data['gold_common_final'] = data[common_column]
    data['gold_common_final_basis'] = common_basis + f' ({common_column})'
    data['pred_unlearning'] = data['out__unlearning_present'].map(
        lambda value: 'Yes' if value is True else ('No' if value is False else None)
    )
    data['pred_int'] = data['pred_unlearning'].map(label_to_int)
    data['gold_aligned_int'] = data['gold_definition_aligned'].map(label_to_int)
    data['gold_common_int'] = data['gold_common_final'].map(label_to_int)
    data['confidence'] = pd.to_numeric(data.get('out__confidence'), errors='coerce')
    # Convert confidence in the chosen class to a probability of Yes.
    data['probability_yes'] = np.where(
        data['pred_int'].eq(1), data['confidence'], 1 - data['confidence']
    )
    return data

def safe_rate(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else np.nan

def binary_metric_record(
    group: pd.DataFrame,
    gold_column: str = 'gold_common_int',
) -> dict[str,Any]:
    valid = group.dropna(subset=[gold_column,'pred_int']).copy()
    if valid.empty:
        return {'n_scored':0}
    y_true = valid[gold_column].astype(int).to_numpy()
    y_pred = valid['pred_int'].astype(int).to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    probability = pd.to_numeric(valid.get('probability_yes'), errors='coerce')
    brier = float(np.mean((probability - y_true) ** 2)) if probability.notna().all() else np.nan
    return {
        'n_scored':len(valid),
        'human_yes':int(y_true.sum()),
        'human_no':int((1-y_true).sum()),
        'pred_yes':int(y_pred.sum()),
        'pred_no':int((1-y_pred).sum()),
        'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn),
        'accuracy':accuracy_score(y_true,y_pred),
        'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),
        'precision_yes':precision_score(y_true,y_pred,zero_division=0),
        'recall_yes':recall_score(y_true,y_pred,zero_division=0),
        'specificity':safe_rate(tn,tn+fp),
        'f1_yes':f1_score(y_true,y_pred,zero_division=0),
        'mcc':matthews_corrcoef(y_true,y_pred) if len(set(y_true)) > 1 else np.nan,
        'cohen_kappa':cohen_kappa_score(y_true,y_pred) if len(set(y_true)) > 1 else np.nan,
        'npv':safe_rate(tn,tn+fn),
        'brier_score':brier,
    }

def summarize_binary(
    data: pd.DataFrame,
    group_columns: Sequence[str],
    gold_column: str = 'gold_common_int',
) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()
    rows = []
    grouper = group_columns[0] if len(group_columns) == 1 else list(group_columns)
    for keys, group in data.groupby(grouper, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_columns, keys))
        row.update(binary_metric_record(group, gold_column))
        row.update({
            'schema_valid_rate':pd.to_numeric(group.get('schema_valid'), errors='coerce').mean(),
            'evidence_valid_rate':pd.to_numeric(group.get('evidence_valid'), errors='coerce').mean(),
            'mean_confidence':pd.to_numeric(group.get('confidence'), errors='coerce').mean(),
            'input_tokens':pd.to_numeric(group.get('input_tokens'), errors='coerce').sum(min_count=1),
            'output_tokens':pd.to_numeric(group.get('output_tokens'), errors='coerce').sum(min_count=1),
            'reasoning_tokens':pd.to_numeric(group.get('reasoning_tokens'), errors='coerce').sum(min_count=1),
            'cached_input_tokens':pd.to_numeric(group.get('cached_input_tokens'), errors='coerce').sum(min_count=1),
            'cache_creation_input_tokens':pd.to_numeric(group.get('cache_creation_input_tokens'), errors='coerce').sum(min_count=1),
            'estimated_cost_usd':pd.to_numeric(group.get('estimated_cost_usd'), errors='coerce').sum(min_count=1),
            'mean_latency_seconds':pd.to_numeric(group.get('latency_seconds'), errors='coerce').mean(),
        })
        rows.append(row)
    return pd.DataFrame(rows)

def prepare_scored_predictions(phase: str) -> pd.DataFrame:
    data = successful_predictions(phase)
    if data.empty:
        return data
    data = add_evidence_audit(data)
    return attach_gold_and_predictions(data)

In [35]:
def condition_ranking(scored: pd.DataFrame) -> tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame]:
    if scored.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    provider_metrics = summarize_binary(
        scored, ['condition_id','provider','definition_id','prompt_style','context_id','workflow']
    )
    document_metrics = summarize_binary(
        scored, ['condition_id','provider','document_title']
    )
    document_positive = document_metrics[document_metrics['human_yes'].gt(0)].copy()
    worst_document = (
        document_positive.groupby('condition_id')['recall_yes']
        .min().rename('worst_provider_document_recall')
        if not document_positive.empty else pd.Series(dtype=float)
    )
    ranking = (
        provider_metrics.groupby(['condition_id','definition_id','prompt_style','context_id','workflow'], dropna=False)
        .agg(
            providers=('provider','nunique'),
            mean_accuracy=('accuracy','mean'),
            mean_balanced_accuracy=('balanced_accuracy','mean'),
            mean_precision_yes=('precision_yes','mean'),
            mean_recall_yes=('recall_yes','mean'),
            mean_specificity=('specificity','mean'),
            mean_f1_yes=('f1_yes','mean'),
            mean_mcc=('mcc','mean'),
            mean_kappa=('cohen_kappa','mean'),
            schema_valid_rate=('schema_valid_rate','mean'),
            evidence_valid_rate=('evidence_valid_rate','mean'),
            total_cost_usd=('estimated_cost_usd','sum'),
            mean_latency_seconds=('mean_latency_seconds','mean'),
        )
        .reset_index()
        .merge(worst_document, on='condition_id', how='left')
    )
    ranking['eligible'] = (
        ranking['providers'].eq(len(MODEL_CONFIGS))
        & ranking['schema_valid_rate'].ge(MIN_SCHEMA_VALID_RATE)
        & ranking['evidence_valid_rate'].ge(MIN_EVIDENCE_VALID_RATE)
        & ranking['mean_specificity'].ge(SPECIFICITY_FLOOR)
    )
    ranking['constraint_shortfall'] = (
        (len(MODEL_CONFIGS) - ranking['providers']).clip(lower=0)
        + (MIN_SCHEMA_VALID_RATE - ranking['schema_valid_rate']).clip(lower=0)
        + (MIN_EVIDENCE_VALID_RATE - ranking['evidence_valid_rate']).clip(lower=0)
        + (SPECIFICITY_FLOOR - ranking['mean_specificity']).clip(lower=0)
    )
    ranking = ranking.sort_values(
        [
            'eligible','constraint_shortfall','worst_provider_document_recall',
            'mean_f1_yes','mean_mcc','mean_balanced_accuracy',
            'mean_specificity','total_cost_usd','condition_id'
        ],
        ascending=[False,True,False,False,False,False,False,True,True],
        na_position='last',
    ).reset_index(drop=True)
    ranking['rank'] = np.arange(1, len(ranking)+1)
    return ranking, provider_metrics, document_metrics

def select_top_condition_ids(
    ranking: pd.DataFrame,
    n: int,
    override: Sequence[str] = (),
) -> list[str]:
    if override:
        unknown = set(override) - set(ranking['condition_id'])
        if unknown:
            raise ValueError(f'Unknown selection override(s): {sorted(unknown)}')
        return list(override)[:n]
    if ranking.empty:
        return []
    eligible = ranking[ranking['eligible']]
    source = eligible if len(eligible) >= n else ranking
    if eligible.empty and not ALLOW_PROVISIONAL_SELECTION:
        raise RuntimeError('No condition met preregistered constraints and provisional selection is disabled.')
    return source.head(n)['condition_id'].tolist()

### Paired tests and model-based inference

In [36]:
def mcnemar_exact_table(
    scored: pd.DataFrame,
    condition_ids: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    if scored.empty:
        return pd.DataFrame()
    use_conditions = list(condition_ids or sorted(scored['condition_id'].unique()))
    rows = []
    for provider in sorted(scored['provider'].unique()):
        provider_data = scored[scored['provider'].eq(provider)]
        for left_id, right_id in itertools.combinations(use_conditions, 2):
            left = provider_data[provider_data['condition_id'].eq(left_id)][
                ['row_id','seed','gold_common_int','pred_int']
            ].rename(columns={'pred_int':'left_pred'})
            right = provider_data[provider_data['condition_id'].eq(right_id)][
                ['row_id','seed','pred_int']
            ].rename(columns={'pred_int':'right_pred'})
            paired = left.merge(right, on=['row_id','seed'], how='inner').dropna()
            if paired.empty:
                continue
            left_correct = paired['left_pred'].astype(int).eq(paired['gold_common_int'].astype(int))
            right_correct = paired['right_pred'].astype(int).eq(paired['gold_common_int'].astype(int))
            b = int((left_correct & ~right_correct).sum())
            c = int((~left_correct & right_correct).sum())
            discordant = b + c
            p_value = binomtest(min(b,c), discordant, 0.5, alternative='two-sided').pvalue if discordant else 1.0
            rows.append({
                'provider':provider,'condition_left':left_id,'condition_right':right_id,
                'left_only_correct':b,'right_only_correct':c,
                'discordant_pairs':discordant,'mcnemar_exact_p':p_value,
            })
    result = pd.DataFrame(rows)
    if not result.empty:
        if multipletests is not None:
            result['holm_adjusted_p'] = multipletests(
                result['mcnemar_exact_p'], method='holm'
            )[1]
        else:
            result['holm_adjusted_p'] = np.minimum(
                1.0, result['mcnemar_exact_p'] * len(result)
            )
    return result

def fit_correctness_gee(scored: pd.DataFrame, phase: str) -> pd.DataFrame:
    if scored.empty or not RUN_GEE_MODELS or sm is None:
        return pd.DataFrame([{'phase':phase,'status':'skipped'}])
    data = scored.dropna(subset=['gold_common_int','pred_int']).copy()
    data['correct'] = data['gold_common_int'].astype(int).eq(data['pred_int'].astype(int)).astype(int)
    data['cluster_id'] = data['provider'].astype(str) + '::' + data['row_id'].astype(str)
    if data['correct'].nunique() < 2 or data['cluster_id'].nunique() < 5:
        return pd.DataFrame([{'phase':phase,'status':'insufficient variation'}])
    formula = 'correct ~ C(definition_id) + C(prompt_style) + C(context_id) + C(workflow) + C(provider)'
    try:
        model = sm.GEE.from_formula(
            formula, groups='cluster_id', data=data,
            family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable()
        )
        fit = model.fit()
        confidence = fit.conf_int()
        return pd.DataFrame({
            'phase':phase,
            'term':fit.params.index,
            'log_odds':fit.params.values,
            'odds_ratio':np.exp(fit.params.values),
            'std_error':fit.bse.values,
            'p_value':fit.pvalues.values,
            'ci_low_or':np.exp(confidence.iloc[:,0].values),
            'ci_high_or':np.exp(confidence.iloc[:,1].values),
            'status':'ok',
        })
    except Exception as exc:
        return pd.DataFrame([{'phase':phase,'status':'failed','error':f'{type(exc).__name__}: {exc}'}])

## 18. Phase 1 — definition × prompt-structure experiment

In [ ]:
PHASE1_TASKS: list[TaskSpec] = []
if BENCHMARK is not None and 'phase1' in ACTIVE_PHASES:
    PHASE1_TASKS = make_task_schedule(
        BENCHMARK, PHASE1_CONDITIONS, ACTIVE_SEEDS, phase='phase1', stage='joint'
    )
PHASE1_SCHEDULE = pd.DataFrame([asdict(task) for task in PHASE1_TASKS])
print({
    'phase':'phase1',
    'benchmark_rows':0 if BENCHMARK is None else len(BENCHMARK),
    'conditions':len(PHASE1_CONDITIONS),
    'seeds':ACTIVE_SEEDS,
    'tasks_per_provider':len(PHASE1_TASKS),
    'maximum_requests_all_providers':len(PHASE1_TASKS) * len(MODEL_CONFIGS),
})
display(PHASE1_SCHEDULE.head(15))

{'phase': 'phase1', 'benchmark_rows': 42, 'conditions': 7, 'seeds': (17,), 'tasks_per_provider': 294, 'maximum_requests_all_providers': 882}


,phase,condition_id,row_id,seed,repeat_id,stage,sequence
0,phase1,P2_D3_ADAPTIVE,P031,17,seed_17_r1,joint,0
1,phase1,P2_D1_OLD,P031,17,seed_17_r1,joint,1
2,phase1,P2_D2_CURRENT,P031,17,seed_17_r1,joint,2
3,phase1,P3_D1_OLD,P031,17,seed_17_r1,joint,3
4,phase1,P3_D2_CURRENT,P031,17,seed_17_r1,joint,4
5,phase1,P1_DIRECT,P031,17,seed_17_r1,joint,5
6,phase1,P3_D3_ADAPTIVE,P031,17,seed_17_r1,joint,6
7,phase1,P2_D3_ADAPTIVE,P027,17,seed_17_r1,joint,7
8,phase1,P3_D2_CURRENT,P027,17,seed_17_r1,joint,8
9,phase1,P2_D1_OLD,P027,17,seed_17_r1,joint,9


In [ ]:
# PHASE 1 — OPENAI ONLY
PHASE1_OPENAI_RAW = pd.DataFrame()
if PHASE1_TASKS and provider_execution_ready('openai'):
    PHASE1_OPENAI_RAW = safe_run_provider_cell('openai', PHASE1_TASKS, PHASE1_CONDITIONS)
    display(PHASE1_OPENAI_RAW.tail())
else:
    print('Phase 1 OpenAI execution skipped.')

openai:phase1:   0%|          | 0/294 [00:00<?, ?it/s]

,run_key,protocol_version,dataset_sha256,provider,requested_model,model_config_sha256,phase,condition_id,row_id,seed,repeat_id,stage,sequence,definition_id,prompt_style,context_id,workflow,description,definition_name,definition_status,definition_sha256,schema_id,schema_sha256,prompt_sha256,cache_key,system_prompt_sha256,user_prompt_sha256,system_prompt,user_prompt,output_schema_json,source_blocks_json,created_at_utc,returned_model,request_id,raw_response_text,input_tokens,output_tokens,total_tokens,reasoning_tokens,cached_input_tokens,cache_creation_input_tokens,stop_reason,latency_seconds,accepted_parameters_json,provider_metadata_json,parsed_output_json,response_sha256,estimated_cost_usd,status,attempt_started_at_utc,completed_at_utc
289,29d89ebca455578136f44e3a361ae108d41056d419029d36c4228d0a81ff3240,prelangchain_ab_v1_2_api_hardened,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,openai,gpt-5.6-terra,9f79900c8bed0b7477c913b17364bb219bc9284c1c9b1d9099c7ce1f2cc084bc,phase1,P2_D2_CURRENT,P041,17,seed_17_r1,joint,289,D2_current_strict,P2_simple_definition,C1_target,W1_one_stage,Current definition + simple structured classification.,Current strict operational definition,current treatment,258f6556333fc5521c03d9aea3d43626ee22e7480b522f5e4c6729122e9e85ad,simple_joint,b6fae6ca706adbcd29328a1015fe3b7e1788e2e54977bfe56b5b790981eb481e,9b4d7333a0987c5fcfa1416f8d68cfa1c6ac6e9ac0c46d74b87bbb93e5406f82,prelangchain_ab_v1_2_api_hardened:D2_current_strict:P2_simple_definition:joint:simple_joint,5046edd74ad323640739a7ecf6dcffedc0d032b8feb00a38c9a3cba87cc5bbdd,3828bc04c5dba1599f27ffd540699da5295cc9d1cf357230a6fa963ae0add8a1,You are an independent qualitative-coding model for a reproducible benchmark on organizational unlearning in government agencies.\n\nClassify one TARGET paragraph per request. Never infer or reproduce a human label. ...,"ROW ID: P041\n\n<TARGET>\nThe fourth and final pillar of a robust response system is alignment of all organizations to a central plan. There are not only too many cooks in the kitchen, as New Orleans Mayor Ray Nagin ...","{""$defs"":{""EvidenceQuote"":{""additionalProperties"":false,""properties"":{""element"":{""enum"":[""prior_state"",""inadequacy_or_failure"",""departure_or_reconfiguration"",""other""],""title"":""Element"",""type"":""string""},""quote"":{""anyO...","{""target"":""The fourth and final pillar of a robust response system is alignment of all organizations to a central plan. There are not only too many cooks in the kitchen, as New Orleans Mayor Ray Nagin complained the ...",2026-07-22T20:57:40.809265+00:00,gpt-5.6-terra,resp_0ec39468c1697934016a612ec4e6bc81a0b01dc37e14b173b3,"{""unlearning_present"":false,""unlearning_mode"":""none"",""change_type"":""diagnosis_only"",""evidence"":[{""element"":""inadequacy_or_failure"",""quote"":""they rarely blend together into a seamless whole"",""source_scope"":""target""},{...",1011,115,1126,0,0,0,completed,1.551077,"{""adapter_revision"":""api_hardened_2026_07_22_r1"",""experiment_seed_local_only"":17,""max_output_tokens"":1800,""parameter_firewall_removed"":{},""prompt_cache_key"":""ul-e5b3176615aa274d26ee698811f44828338af6c873dbdcdfeba2fab...","{""incomplete_details"":null,""sdk_version"":""2.47.0"",""usage"":""ResponseUsage(input_tokens=1011, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=115, output_tokens_details=Out...","{""agency"":null,""change_type"":""diagnosis_only"",""confidence"":0.86,""evidence"":[{""element"":""inadequacy_or_failure"",""quote"":""they rarely blend together into a seamless whole"",""source_scope"":""target""},{""element"":""prior_sta...",e341287ed9c80b64b91334056c39fd8a9a43e10ba52bf14705702375f8f31679,0.004253,ok,2026-07-22T20:57:40.810195+00:00,2026-07-22T20:57:42.363204+00:00
290,cf8557c1c5d38ccac91b98aafc479c45827fd3a4d8349eab8e65c871725acadb,prelangchain_ab_v1_2_api_hardened,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,openai,gpt-5.6-

In [ ]:
# PHASE 1 — ANTHROPIC ONLY
PHASE1_ANTHROPIC_RAW = pd.DataFrame()
if PHASE1_TASKS and provider_execution_ready('anthropic'):
    PHASE1_ANTHROPIC_RAW = safe_run_provider_cell('anthropic', PHASE1_TASKS, PHASE1_CONDITIONS)
    display(PHASE1_ANTHROPIC_RAW.tail())
else:
    print('Phase 1 Anthropic execution skipped.')

anthropic:phase1:   0%|          | 0/294 [00:00<?, ?it/s]

,run_key,protocol_version,dataset_sha256,provider,requested_model,model_config_sha256,phase,condition_id,row_id,seed,repeat_id,stage,sequence,definition_id,prompt_style,context_id,workflow,description,definition_name,definition_status,definition_sha256,schema_id,schema_sha256,prompt_sha256,cache_key,system_prompt_sha256,user_prompt_sha256,system_prompt,user_prompt,output_schema_json,source_blocks_json,created_at_utc,returned_model,request_id,raw_response_text,input_tokens,output_tokens,total_tokens,reasoning_tokens,cached_input_tokens,cache_creation_input_tokens,stop_reason,latency_seconds,accepted_parameters_json,provider_metadata_json,parsed_output_json,response_sha256,estimated_cost_usd,status,attempt_started_at_utc,completed_at_utc
289,7dd25f37a85eee5123d3ad637f19c668eab982ff93752e48b9de2d0a837d2aa6,prelangchain_ab_v1_2_api_hardened,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,anthropic,claude-haiku-4-5-20251001,8090d745697d7da125cd149fd7001c7978c6b4cafdc2156f22e73eec3857c1a2,phase1,P2_D2_CURRENT,P041,17,seed_17_r1,joint,289,D2_current_strict,P2_simple_definition,C1_target,W1_one_stage,Current definition + simple structured classification.,Current strict operational definition,current treatment,258f6556333fc5521c03d9aea3d43626ee22e7480b522f5e4c6729122e9e85ad,simple_joint,b6fae6ca706adbcd29328a1015fe3b7e1788e2e54977bfe56b5b790981eb481e,9b4d7333a0987c5fcfa1416f8d68cfa1c6ac6e9ac0c46d74b87bbb93e5406f82,prelangchain_ab_v1_2_api_hardened:D2_current_strict:P2_simple_definition:joint:simple_joint,5046edd74ad323640739a7ecf6dcffedc0d032b8feb00a38c9a3cba87cc5bbdd,3828bc04c5dba1599f27ffd540699da5295cc9d1cf357230a6fa963ae0add8a1,You are an independent qualitative-coding model for a reproducible benchmark on organizational unlearning in government agencies.\n\nClassify one TARGET paragraph per request. Never infer or reproduce a human label. ...,"ROW ID: P041\n\n<TARGET>\nThe fourth and final pillar of a robust response system is alignment of all organizations to a central plan. There are not only too many cooks in the kitchen, as New Orleans Mayor Ray Nagin ...","{""$defs"":{""EvidenceQuote"":{""additionalProperties"":false,""properties"":{""element"":{""enum"":[""prior_state"",""inadequacy_or_failure"",""departure_or_reconfiguration"",""other""],""title"":""Element"",""type"":""string""},""quote"":{""anyO...","{""target"":""The fourth and final pillar of a robust response system is alignment of all organizations to a central plan. There are not only too many cooks in the kitchen, as New Orleans Mayor Ray Nagin complained the ...",2026-07-22T21:09:11.475832+00:00,claude-haiku-4-5-20251001,msg_011CdHnEdjhMerg17XJVWmNY,"{""unlearning_present"": false, ""unlearning_mode"": ""none"", ""change_type"": ""no_change"", ""evidence"": [], ""target_type"": ""none"", ""agency"": null, ""confidence"": 0.95, ""needs_human_review"": false}",1704,66,1770,0,0,0,end_turn,1.768035,"{""adapter_revision"":""api_hardened_2026_07_22_r1"",""cache_control"":{""type"":""ephemeral""},""cache_control_location"":""system[0].cache_control"",""experiment_seed_local_only"":17,""max_tokens"":1800,""parameter_firewall_removed"":...","{""sdk_version"":""0.118.0"",""stop_sequence"":null,""usage"":""Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference...","{""agency"":null,""change_type"":""no_change"",""confidence"":0.95,""evidence"":[],""needs_human_review"":false,""target_type"":""none"",""unlearning_mode"":""none"",""unlearning_present"":false}",1a6fc07272fb56b3a8b8a1b61b091ca8582223a12b542b9407f6c411befbc815,0.002034,ok,2026-07-22T21:09:11.476491+00:00,2026-07-22T21:09:13.245759+00:00
290,427d25ca80e4334e4cbbc746e4e3bc61575355fc8a940a2471331a8ebecd176c,prelangchain_ab_v1_2_api_hardened,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,anthropic,claude-haiku-4-5-20251001,8090d745697d7da125cd149fd7001c7978c6b4cafdc2156f22e73eec

In [37]:
def retry_delay_from_exception(
    exc: Exception,
    default_seconds: float,
) -> float:
    """
    Extract provider-recommended delays such as:
    - 'Please retry in 47.32s'
    - "'retryDelay': '47s'"
    - 'retry after 30 seconds'
    """
    message = str(exc)

    patterns = [
        r"retry\s+in\s+(\d+(?:\.\d+)?)\s*s",
        r"retryDelay['\"]?\s*:\s*['\"](\d+(?:\.\d+)?)s",
        r"retry\s+after\s+(\d+(?:\.\d+)?)\s*seconds?",
    ]

    for pattern in patterns:
        match = re.search(pattern, message, flags=re.IGNORECASE)
        if match:
            # Add a small buffer so the next request does not arrive too early.
            return float(match.group(1)) + 1.0

    return default_seconds


def call_with_retry(
    function: Callable[[], ProviderCallResult],
    seed: int,
) -> ProviderCallResult:
    """
    Retry temporary server, rate-limit, and quota-window errors.

    Permanent daily quota or billing exhaustion will eventually raise after
    MAX_RETRY_ATTEMPTS instead of looping forever.
    """
    rng = random.Random(
        stable_int_seed(PROTOCOL_VERSION, seed, "retry")
    )

    retryable_classes = {"transient", "quota"}

    for attempt in range(1, MAX_RETRY_ATTEMPTS + 1):
        try:
            return function()

        except Exception as exc:
            classification = error_classification(exc)

            if (
                classification not in retryable_classes
                or attempt == MAX_RETRY_ATTEMPTS
            ):
                raise

            exponential_delay = (
                BASE_RETRY_SECONDS * (2 ** (attempt - 1))
            )

            provider_delay = retry_delay_from_exception(
                exc,
                default_seconds=exponential_delay,
            )

            # Respect the provider delay while adding modest deterministic jitter.
            jitter = rng.uniform(0.5, 2.0)
            delay = max(provider_delay, exponential_delay) + jitter

            print(
                f"{classification.title()} error "
                f"{attempt}/{MAX_RETRY_ATTEMPTS}. "
                f"Retrying in {delay:.1f} seconds.\n"
                f"{type(exc).__name__}: {exc}"
            )

            time.sleep(delay)

    raise RuntimeError("Retry loop ended unexpectedly.")

In [38]:
REQUEST_SPACING_SECONDS = 4.5

In [ ]:
%%script true
spacing_seconds = (
    4.5
    if config.provider == "gemini"
    else REQUEST_SPACING_SECONDS
)
time.sleep(spacing_seconds)

In [ ]:
# PHASE 1 — GEMINI ONLY
PHASE1_GEMINI_RAW = pd.DataFrame()
if PHASE1_TASKS and provider_execution_ready('gemini'):
    PHASE1_GEMINI_RAW = safe_run_provider_cell('gemini', PHASE1_TASKS, PHASE1_CONDITIONS)
    display(PHASE1_GEMINI_RAW.tail())
else:
    print('Phase 1 Gemini execution skipped.')

gemini:phase1:   0%|          | 0/294 [00:00<?, ?it/s]

,run_key,protocol_version,dataset_sha256,provider,requested_model,model_config_sha256,phase,condition_id,row_id,seed,repeat_id,stage,sequence,definition_id,prompt_style,context_id,workflow,description,definition_name,definition_status,definition_sha256,schema_id,schema_sha256,prompt_sha256,cache_key,system_prompt_sha256,user_prompt_sha256,system_prompt,user_prompt,output_schema_json,source_blocks_json,created_at_utc,returned_model,request_id,raw_response_text,input_tokens,output_tokens,total_tokens,reasoning_tokens,cached_input_tokens,cache_creation_input_tokens,stop_reason,latency_seconds,accepted_parameters_json,provider_metadata_json,parsed_output_json,response_sha256,estimated_cost_usd,status,attempt_started_at_utc,completed_at_utc
289,3becb0f1da82bbe5ae8cd9bdb19f2403fb91585f54dc80b48271294e62ea8293,prelangchain_ab_v1_2_api_hardened,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,gemini,gemini-3.1-flash-lite,1d141db9a3fbe2d79e7d076b6e712cc6af3f7d7478e85098f1aba9ff08c4deaa,phase1,P2_D2_CURRENT,P041,17,seed_17_r1,joint,289,D2_current_strict,P2_simple_definition,C1_target,W1_one_stage,Current definition + simple structured classification.,Current strict operational definition,current treatment,258f6556333fc5521c03d9aea3d43626ee22e7480b522f5e4c6729122e9e85ad,simple_joint,b6fae6ca706adbcd29328a1015fe3b7e1788e2e54977bfe56b5b790981eb481e,9b4d7333a0987c5fcfa1416f8d68cfa1c6ac6e9ac0c46d74b87bbb93e5406f82,prelangchain_ab_v1_2_api_hardened:D2_current_strict:P2_simple_definition:joint:simple_joint,5046edd74ad323640739a7ecf6dcffedc0d032b8feb00a38c9a3cba87cc5bbdd,3828bc04c5dba1599f27ffd540699da5295cc9d1cf357230a6fa963ae0add8a1,You are an independent qualitative-coding model for a reproducible benchmark on organizational unlearning in government agencies.\n\nClassify one TARGET paragraph per request. Never infer or reproduce a human label. ...,"ROW ID: P041\n\n<TARGET>\nThe fourth and final pillar of a robust response system is alignment of all organizations to a central plan. There are not only too many cooks in the kitchen, as New Orleans Mayor Ray Nagin ...","{""$defs"":{""EvidenceQuote"":{""additionalProperties"":false,""properties"":{""element"":{""enum"":[""prior_state"",""inadequacy_or_failure"",""departure_or_reconfiguration"",""other""],""title"":""Element"",""type"":""string""},""quote"":{""anyO...","{""target"":""The fourth and final pillar of a robust response system is alignment of all organizations to a central plan. There are not only too many cooks in the kitchen, as New Orleans Mayor Ray Nagin complained the ...",2026-07-22T21:39:46.722835+00:00,gemini-3.1-flash-lite,ojhhaoqAMKmIqtsP7pTBgQc,"{\n ""unlearning_present"": true,\n ""unlearning_mode"": ""adaptive_reconfiguration"",\n ""change_type"": ""replace"",\n ""evidence"": [\n {\n ""element"": ""inadequacy_or_failure"",\n ""quote"": ""too many cooks in th...",646,183,925,96,None,0,FinishReason.STOP,1.335495,"{""adapter_revision"":""api_hardened_2026_07_22_r1"",""experiment_seed_local_only"":null,""max_output_tokens"":1800,""provider_seed_sent"":true,""provider_seed_supported"":true,""provider_seed_value"":17,""response_json_schema_sha2...","{""prompt_feedback"":null,""sdk_version"":""2.13.0"",""usage_metadata"":""cache_tokens_details=None cached_content_token_count=None candidates_token_count=183 candidates_tokens_details=None prompt_token_count=646 prompt_token...","{""agency"":""New Orleans"",""change_type"":""replace"",""confidence"":0.9,""evidence"":[{""element"":""inadequacy_or_failure"",""quote"":""too many cooks in the kitchen... but too many recipes for action"",""source_scope"":""target""},{""el...",2fa180b3dfade0d8f981d4f58413f7302a1acf5daed0f0ddeedd01c3efb7c7bb,0.000436,ok,2026-07-22T21:39:46.723567+00:00,2026-07-22T21:39:48.064132+00:00
290,bda259b36dba6e72099d2afde6cd7c8d9ca37301ad8203de2f15b5b76d8d15d6,prelangchain_ab_v1_2_api_hardened,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,gemini,gemini-3.1-flash-lite,1d141db9a3fb

### Phase 1 analysis and selection

In [ ]:
PHASE1_SCORED = prepare_scored_predictions('phase1')
PHASE1_RANKING, PHASE1_PROVIDER_METRICS, PHASE1_DOCUMENT_METRICS = condition_ranking(PHASE1_SCORED)
PHASE1_MCNEMAR = mcnemar_exact_table(PHASE1_SCORED)
PHASE1_GEE = fit_correctness_gee(PHASE1_SCORED, 'phase1')

if not PHASE1_RANKING.empty:
    display(PHASE1_RANKING)
    display(PHASE1_PROVIDER_METRICS.sort_values(['condition_id','provider']))
    display(PHASE1_DOCUMENT_METRICS.sort_values(['condition_id','provider','document_title']))
else:
    print('No Phase 1 successful predictions available yet.')

/tmp/ipykernel_1654/689186196.py:68: RuntimeWarning: overflow encountered in exp
  'ci_high_or':np.exp(confidence.iloc[:,1].values),


,condition_id,definition_id,prompt_style,context_id,workflow,providers,mean_accuracy,mean_balanced_accuracy,mean_precision_yes,mean_recall_yes,mean_specificity,mean_f1_yes,mean_mcc,mean_kappa,schema_valid_rate,evidence_valid_rate,total_cost_usd,mean_latency_seconds,worst_provider_document_recall,eligible,constraint_shortfall,rank
0,P1_DIRECT,D0_none,P1_direct,C1_target,W1_one_stage,3,0.468254,0.616667,0.933333,0.333333,0.900000,0.485364,0.230461,0.135515,1.0,0.976190,0.300032,1.810204,0.166667,False,0.003810,1
1,P3_D2_CURRENT,D2_current_strict,P3_evidence_checklist,C1_target,W1_one_stage,3,0.507937,0.665625,0.979167,0.364583,0.966667,0.525794,0.316138,0.197925,1.0,0.944444,0.425127,2.307064,0.187500,False,0.035556,2
2,P3_D3_ADAPTIVE,D3_provisional_adaptive,P3_evidence_checklist,C1_target,W1_one_stage,3,0.380952,0.582292,0.972222,0.197917,0.966667,0.311146,0.194342,0.091408,1.0,0.944444,0.465131,2.398845,0.000000,False,0.035556,3
3,P2_D3_ADAPTIVE,D3_provisional_adaptive,P2_simple_definition,C1_target,W1_one_stage,3,0.404762,0.586458,0.952381,0.239583,0.933333,0.364389,0.194773,0.097348,1.0,0.944444,0.373258,1.940786,0.000000,False,0.035556,4
4,P2_D2_CURRENT,D2_current_strict,P2_simple_definition,C1_target,W1_one_stage,3,0.428571,0.613542,0.969697,0.260417,0.966667,0.408030,0.242339,0.126015,1.0,0.920635,0.325021,2.031538,0.062500,False,0.059365,5
5,P3_D1_OLD,D1_old_broad,P3_evidence_checklist,C1_target,W1_one_stage,3,0.555556,0.605208,0.847547,0.510417,0.700000,0.635454,0.180975,0.145493,1.0,0.944444,0.428266,2.448193,0.400000,False,0.135556,6
6,P2_D1_OLD,D1_old_broad,P2_simple_definition,C1_target,W1_one_stage,3,0.571429,0.604167,0.842029,0.541667,0.666667,0.652795,0.180926,0.151540,1.0,0.936508,0.343165,1.941406,0.333333,False,0.176825,7


,condition_id,provider,definition_id,prompt_style,context_id,workflow,n_scored,human_yes,human_no,pred_yes,pred_no,tp,tn,fp,fn,accuracy,balanced_accuracy,precision_yes,recall_yes,specificity,f1_yes,mcc,cohen_kappa,npv,brier_score,schema_valid_rate,evidence_valid_rate,mean_confidence,input_tokens,output_tokens,reasoning_tokens,cached_input_tokens,cache_creation_input_tokens,estimated_cost_usd,mean_latency_seconds
0,P1_DIRECT,anthropic,D0_none,P1_direct,C1_target,W1_one_stage,42,32,10,15,27,12,7,3,20,0.452381,0.537500,0.800000,0.37500,0.7,0.510638,0.066667,0.047337,0.259259,0.479186,1.0,0.952381,0.897619,66867,5541,0,0.0,0,0.094572,1.725245
1,P1_DIRECT,gemini,D0_none,P1_direct,C1_target,W1_one_stage,42,32,10,12,30,12,10,0,20,0.523810,0.687500,1.000000,0.37500,1.0,0.545455,0.353553,0.222222,0.333333,0.450476,1.0,0.976190,0.947619,22839,6934,4795,NaN,0,0.016111,1.487316
2,P1_DIRECT,openai,D0_none,P1_direct,C1_target,W1_one_stage,42,32,10,8,34,8,10,0,24,0.428571,0.625000,1.000000,0.25000,1.0,0.400000,0.271163,0.136986,0.294118,0.501474,1.0,1.000000,0.921190,37932,6158,1422,0.0,3438,0.189349,2.218051
3,P2_D1_OLD,anthropic,D1_old_broad,P2_simple_definition,C1_target,W1_one_stage,42,32,10,23,19,19,6,4,13,0.595238,0.596875,0.826087,0.59375,0.6,0.690909,0.165797,0.147971,0.315789,0.347252,1.0,0.928571,0.859524,72915,6376,0,0.0,0,0.104795,1.782411
4,P2_D1_OLD,gemini,D1_old_broad,P2_simple_definition,C1_target,W1_one_stage,42,32,10,15,27,13,8,2,19,0.500000,0.603125,0.866667,0.40625,0.8,0.553191,0.183333,0.130178,0.296296,0.473036,1.0,0.928571,0.939286,28383,6958,4176,NaN,0,0.017533,1.303851
5,P2_D1_OLD,openai,D1_old_broad,P2_simple_definition,C1_target,W1_one_stage,42,32,10,24,18,20,6,4,12,0.619048,0.612500,0.833333,0.62500,0.6,0.714286,0.193649,0.176471,0.333333,0.330026,1.0,0.952381,0.867381,43686,6614,1228,0.0,19859,0.220837,2.737955
6,P2_D2_CURRENT,anthropic,D2_current_strict,P2_simple_definition,C1_target,W1_one_stage,42,32,10,11,31,10,9,1,22,0.452381,0.606250,0.909091,0.31250,0.9,0.465116,0.205853,0.123412,0.290323,0.494762,1.0,0.928571,0.916190,73167,4983,0,0.0,0,0.098082,2.326719
7,P2_D2_CURRENT,gemini,D2_current_strict,P2_simple_definition,C1_target,W1_one_stage,42,32,10,8,34,8,10,0,24,0.428571,0.625000,1.000000,0.25000,1.0,0.400000,0.271163,0.136986,0.294118,0.539524,1.0,0.904762,0.957143,28551,7267,4329,NaN,0,0.018038,1.347548
8,P2_D2_CURRENT,openai,D2_current_strict,P2_simple_definition,C1_target,W1_one_stage,42,32,10,7,35,7,10,0,25,0.404762,0.609375,1.000000,0.21875,1.0,0.358974,0.250000,0.117647,0.285714,0.527510,1.0,0.928571,0.931429,43812,5795,756,0.0,19913,0.208901,2.420348
9,P2_D3_ADAPTIVE,anthropic,D3_provisional_adaptive,P2_simple_definition,C1_target,W1_one_stage,42,32,10,14,28,12,8,2,20,0.476190,0.587500,0.857143,0.37500,0.8,0.521739,0.158114,0.108108,0.285714,0.462214,1.0,0.952381,0.887619,77829,5328,0,0.0,0,0.104469,1.983826


,condition_id,provider,document_title,n_scored,human_yes,human_no,pred_yes,pred_no,tp,tn,fp,fn,accuracy,balanced_accuracy,precision_yes,recall_yes,specificity,f1_yes,mcc,cohen_kappa,npv,brier_score,schema_valid_rate,evidence_valid_rate,mean_confidence,input_tokens,output_tokens,reasoning_tokens,cached_input_tokens,cache_creation_input_tokens,estimated_cost_usd,mean_latency_seconds
0,P1_DIRECT,anthropic,"Hurricane Katrina: GAO’s Preliminary Observations Regarding Preparedness, Response, and Recovery",14,10,4,7,7,5,2,2,5,0.500000,0.500000,0.714286,0.500000,0.5,0.588235,0.000000,0.000000,0.285714,0.409171,1.0,1.000000,0.871429,22869,2045,0,0.0,0,0.033094,2.049719
1,P1_DIRECT,anthropic,Lessons Learned: EPA’s Response to Hurricane Katrina,21,16,5,6,15,5,4,1,11,0.428571,0.556250,0.833333,0.312500,0.8,0.454545,0.106066,0.066667,0.266667,0.510190,1.0,0.904762,0.910476,33141,2647,0,0.0,0,0.046376,1.566804
2,P1_DIRECT,anthropic,The Katrina Effect on American Preparedness — A report on the lessons Americans learned in watching the Katrina catastrophe unfold,7,6,1,2,5,2,1,0,4,0.428571,0.666667,1.000000,0.333333,1.0,0.500000,0.258199,0.125000,0.200000,0.526200,1.0,1.000000,0.911429,10857,849,0,0.0,0,0.015102,1.551618
3,P1_DIRECT,gemini,"Hurricane Katrina: GAO’s Preliminary Observations Regarding Preparedness, Response, and Recovery",14,10,4,4,10,4,4,0,6,0.571429,0.700000,1.000000,0.400000,1.0,0.571429,0.400000,0.275862,0.400000,0.397500,1.0,1.000000,0.950000,8151,2439,1932,NaN,0,0.005696,1.360727
4,P1_DIRECT,gemini,Lessons Learned: EPA’s Response to Hurricane Katrina,21,16,5,5,16,5,5,0,11,0.476190,0.656250,1.000000,0.312500,1.0,0.476190,0.312500,0.177936,0.312500,0.509167,1.0,0.952381,0.954762,11148,3349,2182,NaN,0,0.007810,1.636036
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,P3_D3_ADAPTIVE,gemini,Lessons Learned: EPA’s Response to Hurricane Katrina,21,16,5,5,16,5,5,0,11,0.476190,0.656250,1.000000,0.312500,1.0,0.476190,0.312500,0.177936,0.312500,0.508095,1.0,0.904762,0.966667,16566,5320,4353,NaN,0,0.012122,1.726925
59,P3_D3_ADAPTIVE,gemini,The Katrina Effect on American Preparedness — A report on the lessons Americans learned in watching the Katrina catastrophe unfold,7,6,1,1,6,1,1,0,5,0.285714,0.583333,1.000000,0.166667,1.0,0.285714,0.166667,0.054054,0.166667,0.688571,1.0,1.000000,0.971429,5346,1700,681,NaN,0,0.003886,1.449599
60,P3_D3_ADAPTIVE,openai,"Hurricane Katrina: GAO’s Preliminary Observations Regarding Preparedness, Response, and Recovery",14,10,4,0,14,0,4,0,10,0.285714,0.500000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.285714,0.636000,1.0,1.000000,0.951429,17699,3229,1258,0.0,17657,0.103718,2.807948
61,P3_D3_ADAPTIVE,openai,Lessons Learned: EPA’s Response to Hurricane Katrina,21,16,5,1,20,1,5,0,15,0.285714,0.531250,1.000000,0.062500,1.0,0.117647,0.125000,0.030769,0.250000,0.646367,1.0,1.000000,0.946190,25476,4947,1899,0.0,25413,0.153778,3.173246


In [ ]:
# Definition-aligned and common-final comparisons are shown side by side.
PHASE1_ALIGNED_METRICS = pd.DataFrame()
PHASE1_DEFINITION_TRADEOFF = pd.DataFrame()
PHASE1_EPA_DIAGNOSTICS = pd.DataFrame()
if not PHASE1_SCORED.empty:
    PHASE1_ALIGNED_METRICS = summarize_binary(
        PHASE1_SCORED,
        ['condition_id','provider','definition_id','prompt_style'],
        gold_column='gold_aligned_int',
    )
    common = summarize_binary(
        PHASE1_SCORED,
        ['condition_id','provider','definition_id','prompt_style'],
        gold_column='gold_common_int',
    )
    keep = ['condition_id','provider','definition_id','prompt_style','accuracy','recall_yes','specificity','f1_yes','mcc']
    PHASE1_DEFINITION_TRADEOFF = PHASE1_ALIGNED_METRICS[keep].merge(
        common[keep], on=['condition_id','provider','definition_id','prompt_style'],
        suffixes=('_aligned','_common_final')
    )
    PHASE1_EPA_DIAGNOSTICS = summarize_binary(
        PHASE1_SCORED[
            PHASE1_SCORED['document_title'].str.contains('EPA', case=False, na=False)
        ],
        ['condition_id','provider','definition_id','prompt_style'],
        gold_column='gold_common_int',
    )
    display(PHASE1_DEFINITION_TRADEOFF)
    display(PHASE1_EPA_DIAGNOSTICS)

,condition_id,provider,definition_id,prompt_style,accuracy_aligned,recall_yes_aligned,specificity_aligned,f1_yes_aligned,mcc_aligned,accuracy_common_final,recall_yes_common_final,specificity_common_final,f1_yes_common_final,mcc_common_final
0,P1_DIRECT,anthropic,D0_none,P1_direct,0.452381,0.37500,0.7,0.510638,0.066667,0.452381,0.37500,0.7,0.510638,0.066667
1,P1_DIRECT,gemini,D0_none,P1_direct,0.523810,0.37500,1.0,0.545455,0.353553,0.523810,0.37500,1.0,0.545455,0.353553
2,P1_DIRECT,openai,D0_none,P1_direct,0.428571,0.25000,1.0,0.400000,0.271163,0.428571,0.25000,1.0,0.400000,0.271163
3,P2_D1_OLD,anthropic,D1_old_broad,P2_simple_definition,0.595238,0.59375,0.6,0.690909,0.165797,0.595238,0.59375,0.6,0.690909,0.165797
4,P2_D1_OLD,gemini,D1_old_broad,P2_simple_definition,0.500000,0.40625,0.8,0.553191,0.183333,0.500000,0.40625,0.8,0.553191,0.183333
5,P2_D1_OLD,openai,D1_old_broad,P2_simple_definition,0.619048,0.62500,0.6,0.714286,0.193649,0.619048,0.62500,0.6,0.714286,0.193649
6,P2_D2_CURRENT,anthropic,D2_current_strict,P2_simple_definition,0.452381,0.31250,0.9,0.465116,0.205853,0.452381,0.31250,0.9,0.465116,0.205853
7,P2_D2_CURRENT,gemini,D2_current_strict,P2_simple_definition,0.428571,0.25000,1.0,0.400000,0.271163,0.428571,0.25000,1.0,0.400000,0.271163
8,P2_D2_CURRENT,openai,D2_current_strict,P2_simple_definition,0.404762,0.21875,1.0,0.358974,0.250000,0.404762,0.21875,1.0,0.358974,0.250000
9,P2_D3_ADAPTIVE,anthropic,D3_provisional_adaptive,P2_simple_definition,0.476190,0.37500,0.8,0.521739,0.158114,0.476190,0.37500,0.8,0.521739,0.158114


,condition_id,provider,definition_id,prompt_style,n_scored,human_yes,human_no,pred_yes,pred_no,tp,tn,fp,fn,accuracy,balanced_accuracy,precision_yes,recall_yes,specificity,f1_yes,mcc,cohen_kappa,npv,brier_score,schema_valid_rate,evidence_valid_rate,mean_confidence,input_tokens,output_tokens,reasoning_tokens,cached_input_tokens,cache_creation_input_tokens,estimated_cost_usd,mean_latency_seconds
0,P1_DIRECT,anthropic,D0_none,P1_direct,42,32,10,15,27,12,7,3,20,0.452381,0.537500,0.800000,0.37500,0.7,0.510638,0.066667,0.047337,0.259259,0.479186,1.0,0.952381,0.897619,66867,5541,0,0.0,0,0.094572,1.725245
1,P1_DIRECT,gemini,D0_none,P1_direct,42,32,10,12,30,12,10,0,20,0.523810,0.687500,1.000000,0.37500,1.0,0.545455,0.353553,0.222222,0.333333,0.450476,1.0,0.976190,0.947619,22839,6934,4795,NaN,0,0.016111,1.487316
2,P1_DIRECT,openai,D0_none,P1_direct,42,32,10,8,34,8,10,0,24,0.428571,0.625000,1.000000,0.25000,1.0,0.400000,0.271163,0.136986,0.294118,0.501474,1.0,1.000000,0.921190,37932,6158,1422,0.0,3438,0.189349,2.218051
3,P2_D1_OLD,anthropic,D1_old_broad,P2_simple_definition,42,32,10,23,19,19,6,4,13,0.595238,0.596875,0.826087,0.59375,0.6,0.690909,0.165797,0.147971,0.315789,0.347252,1.0,0.928571,0.859524,72915,6376,0,0.0,0,0.104795,1.782411
4,P2_D1_OLD,gemini,D1_old_broad,P2_simple_definition,42,32,10,15,27,13,8,2,19,0.500000,0.603125,0.866667,0.40625,0.8,0.553191,0.183333,0.130178,0.296296,0.473036,1.0,0.928571,0.939286,28383,6958,4176,NaN,0,0.017533,1.303851
5,P2_D1_OLD,openai,D1_old_broad,P2_simple_definition,42,32,10,24,18,20,6,4,12,0.619048,0.612500,0.833333,0.62500,0.6,0.714286,0.193649,0.176471,0.333333,0.330026,1.0,0.952381,0.867381,43686,6614,1228,0.0,19859,0.220837,2.737955
6,P2_D2_CURRENT,anthropic,D2_current_strict,P2_simple_definition,42,32,10,11,31,10,9,1,22,0.452381,0.606250,0.909091,0.31250,0.9,0.465116,0.205853,0.123412,0.290323,0.494762,1.0,0.928571,0.916190,73167,4983,0,0.0,0,0.098082,2.326719
7,P2_D2_CURRENT,gemini,D2_current_strict,P2_simple_definition,42,32,10,8,34,8,10,0,24,0.428571,0.625000,1.000000,0.25000,1.0,0.400000,0.271163,0.136986,0.294118,0.539524,1.0,0.904762,0.957143,28551,7267,4329,NaN,0,0.018038,1.347548
8,P2_D2_CURRENT,openai,D2_current_strict,P2_simple_definition,42,32,10,7,35,7,10,0,25,0.404762,0.609375,1.000000,0.21875,1.0,0.358974,0.250000,0.117647,0.285714,0.527510,1.0,0.928571,0.931429,43812,5795,756,0.0,19913,0.208901,2.420348
9,P2_D3_ADAPTIVE,anthropic,D3_provisional_adaptive,P2_simple_definition,42,32,10,14,28,12,8,2,20,0.476190,0.587500,0.857143,0.37500,0.8,0.521739,0.158114,0.108108,0.285714,0.462214,1.0,0.952381,0.887619,77829,5328,0,0.0,0,0.104469,1.983826


In [ ]:
PHASE1_SELECTED_IDS = select_top_condition_ids(
    PHASE1_RANKING, n=2, override=PHASE1_SELECTION_OVERRIDE
) if not PHASE1_RANKING.empty else list(PHASE1_SELECTION_OVERRIDE)
PHASE1_SELECTION = {
    'selected_condition_ids': PHASE1_SELECTED_IDS,
    'selection_basis': (
        'override' if PHASE1_SELECTION_OVERRIDE
        else 'preregistered constrained ranking against common-final gold'
    ),
    'provisional_gold': not complete_gold_column('gold_final_definition'),
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'phase1_ranking_sha256': sha256_text(
        PHASE1_RANKING.fillna('').to_csv(index=False) if not PHASE1_RANKING.empty else ''
    ),
}
atomic_write_text(
    CONFIG_ROOT / 'phase1_selection.json',
    json.dumps(PHASE1_SELECTION, indent=2, default=str)
)
print(PHASE1_SELECTION)

{'selected_condition_ids': ['P1_DIRECT', 'P3_D2_CURRENT'], 'selection_basis': 'preregistered constrained ranking against common-final gold', 'provisional_gold': True, 'created_at_utc': '2026-07-22T21:40:58.573684+00:00', 'phase1_ranking_sha256': '52f8a5ecf0e60ed480f97c1f9bbf06612247be7051f39555879f2fb46bf3f107'}


## 19. Phase 2 — matched context A/B test

In [ ]:
ACTIVE_PHASES = ("phase2",)

print("ACTIVE_PHASES:", ACTIVE_PHASES)

ACTIVE_PHASES: ('phase2',)


In [ ]:
def load_selection_json(name: str) -> dict[str,Any]:
    path = CONFIG_ROOT / name
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}

def resolve_phase1_selected_ids() -> list[str]:
    if PHASE1_SELECTION_OVERRIDE:
        return list(PHASE1_SELECTION_OVERRIDE)
    if 'PHASE1_SELECTED_IDS' in globals() and PHASE1_SELECTED_IDS:
        return list(PHASE1_SELECTED_IDS)
    return list(load_selection_json('phase1_selection.json').get('selected_condition_ids', []))

def phase2_conditions_from_phase1(selected_ids: Sequence[str]) -> list[ConditionSpec]:
    base_map = conditions_by_id(PHASE1_CONDITIONS)
    contexts: list[ContextLevel] = [
        'C1_target','C2_metadata','C3_plusminus1','C4_plusminus2'
    ]
    conditions = []
    for base_id in selected_ids:
        if base_id not in base_map:
            raise ValueError(f'Phase 1 selection not registered: {base_id}')
        base = base_map[base_id]
        for context_id in contexts:
            conditions.append(ConditionSpec(
                condition_id=f'{base_id}__{context_id}',
                phase='phase2', definition_id=base.definition_id,
                prompt_style=base.prompt_style, context_id=context_id,
                workflow='W1_one_stage',
                description=f'{base.description} Context treatment {context_id}.',
            ))
    return conditions

PHASE1_SELECTED_FOR_CONTEXT = resolve_phase1_selected_ids()
PHASE2_CONDITIONS = phase2_conditions_from_phase1(PHASE1_SELECTED_FOR_CONTEXT) if PHASE1_SELECTED_FOR_CONTEXT else []
# C1 is reused from Phase 1 to avoid paying for an identical prompt. Only C2–C4
# generate new provider calls.
PHASE2_NEW_CONDITIONS = [c for c in PHASE2_CONDITIONS if c.context_id != 'C1_target']
PHASE2_TASKS: list[TaskSpec] = []
if BENCHMARK is not None and PHASE2_NEW_CONDITIONS and 'phase2' in ACTIVE_PHASES:
    PHASE2_TASKS = make_task_schedule(
        BENCHMARK, PHASE2_NEW_CONDITIONS, ACTIVE_SEEDS, phase='phase2', stage='joint'
    )

display(pd.DataFrame([asdict(c) for c in PHASE2_CONDITIONS]))
print({
    'selected_phase1':PHASE1_SELECTED_FOR_CONTEXT,
    'new_tasks_per_provider':len(PHASE2_TASKS),
    'C1_reused':True,
})

,condition_id,phase,definition_id,prompt_style,context_id,workflow,description
0,P1_DIRECT__C1_target,phase2,D0_none,P1_direct,C1_target,W1_one_stage,Direct task; no supplied definition. Context treatment C1_target.
1,P1_DIRECT__C2_metadata,phase2,D0_none,P1_direct,C2_metadata,W1_one_stage,Direct task; no supplied definition. Context treatment C2_metadata.
2,P1_DIRECT__C3_plusminus1,phase2,D0_none,P1_direct,C3_plusminus1,W1_one_stage,Direct task; no supplied definition. Context treatment C3_plusminus1.
3,P1_DIRECT__C4_plusminus2,phase2,D0_none,P1_direct,C4_plusminus2,W1_one_stage,Direct task; no supplied definition. Context treatment C4_plusminus2.
4,P3_D2_CURRENT__C1_target,phase2,D2_current_strict,P3_evidence_checklist,C1_target,W1_one_stage,Current definition + explicit evidence checklist. Context treatment C1_target.
5,P3_D2_CURRENT__C2_metadata,phase2,D2_current_strict,P3_evidence_checklist,C2_metadata,W1_one_stage,Current definition + explicit evidence checklist. Context treatment C2_metadata.
6,P3_D2_CURRENT__C3_plusminus1,phase2,D2_current_strict,P3_evidence_checklist,C3_plusminus1,W1_one_stage,Current definition + explicit evidence checklist. Context treatment C3_plusminus1.
7,P3_D2_CURRENT__C4_plusminus2,phase2,D2_current_strict,P3_evidence_checklist,C4_plusminus2,W1_one_stage,Current definition + explicit evidence checklist. Context treatment C4_plusminus2.


{'selected_phase1': ['P1_DIRECT', 'P3_D2_CURRENT'], 'new_tasks_per_provider': 252, 'C1_reused': True}


In [ ]:
# PHASE 2 — OPENAI ONLY
PHASE2_OPENAI_RAW = pd.DataFrame()
if PHASE2_TASKS and provider_execution_ready('openai'):
    PHASE2_OPENAI_RAW = safe_run_provider_cell('openai', PHASE2_TASKS, PHASE2_NEW_CONDITIONS)
    display(PHASE2_OPENAI_RAW.tail())
else:
    print('Phase 2 OpenAI execution skipped.')

openai:phase2:   0%|          | 0/252 [00:00<?, ?it/s]

,run_key,protocol_version,dataset_sha256,provider,requested_model,model_config_sha256,phase,condition_id,row_id,seed,repeat_id,stage,sequence,definition_id,prompt_style,context_id,workflow,description,definition_name,definition_status,definition_sha256,schema_id,schema_sha256,prompt_sha256,cache_key,system_prompt_sha256,user_prompt_sha256,system_prompt,user_prompt,output_schema_json,source_blocks_json,created_at_utc,returned_model,request_id,raw_response_text,input_tokens,output_tokens,total_tokens,reasoning_tokens,cached_input_tokens,cache_creation_input_tokens,stop_reason,latency_seconds,accepted_parameters_json,provider_metadata_json,parsed_output_json,response_sha256,estimated_cost_usd,status,attempt_started_at_utc,completed_at_utc
247,4aff932a29b3a88d4cc7bc7a664eb11f598b5f28ddd530f7872868ea50979be3,prelangchain_ab_v1_2_api_hardened,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,openai,gpt-5.6-terra,9f79900c8bed0b7477c913b17364bb219bc9284c1c9b1d9099c7ce1f2cc084bc,phase2,P1_DIRECT__C4_plusminus2,P041,17,seed_17_r1,joint,247,D0_none,P1_direct,C4_plusminus2,W1_one_stage,Direct task; no supplied definition. Context treatment C4_plusminus2.,No supplied definition,control,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855,simple_joint,b6fae6ca706adbcd29328a1015fe3b7e1788e2e54977bfe56b5b790981eb481e,7a94486ea9ca8f5937de8c3e81b300a5f29a6bf825271ed0ce5c8b82a9d17576,prelangchain_ab_v1_2_api_hardened:D0_none:P1_direct:joint:simple_joint,964284c470c6697c61ba4b976eef1bdf9bfe9567eaf201d7bd2e2ec81a07c246,9a5bf6f53d0b29066ed7896f653e795bb21c7656e202f275886c98e89ba9d6ac,You are an independent qualitative-coding model for a reproducible benchmark on organizational unlearning in government agencies.\n\nClassify one TARGET paragraph per request. Never infer or reproduce a human label. ...,ROW ID: P041\n\n<METADATA>\nDocument title: The Katrina Effect on American Preparedness — A report on the lessons Americans learned in watching the Katrina catastrophe unfold\nSection heading: Conclusion\nPDF page: 1...,"{""$defs"":{""EvidenceQuote"":{""additionalProperties"":false,""properties"":{""element"":{""enum"":[""prior_state"",""inadequacy_or_failure"",""departure_or_reconfiguration"",""other""],""title"":""Element"",""type"":""string""},""quote"":{""anyO...","{""metadata"":""Document title: The Katrina Effect on American Preparedness — A report on the lessons Americans learned in watching the Katrina catastrophe unfold\nSection heading: Conclusion\nPDF page: 10"",""next_1"":""Cr...",2026-07-22T22:30:06.383284+00:00,gpt-5.6-terra,resp_0452c1b6a6331bd1016a61446e7450819db8d0588a49ca1d0e,"{""unlearning_present"":false,""unlearning_mode"":""none"",""change_type"":""diagnosis_only"",""evidence"":[{""element"":""prior_state"",""quote"":""too many recipes for action"",""source_scope"":""target""},{""element"":""inadequacy_or_failur...",1253,184,1437,67,0,1250,completed,2.074870,"{""adapter_revision"":""api_hardened_2026_07_22_r1"",""experiment_seed_local_only"":17,""max_output_tokens"":1800,""parameter_firewall_removed"":{},""prompt_cache_key"":""ul-7aaa7537e783b914f593386fde3f5e5f1cfc7509bf26c060fd36ecc...","{""incomplete_details"":null,""sdk_version"":""2.47.0"",""usage"":""ResponseUsage(input_tokens=1253, input_tokens_details=InputTokensDetails(cache_write_tokens=1250, cached_tokens=0), output_tokens=184, output_tokens_details=...","{""agency"":null,""change_type"":""diagnosis_only"",""confidence"":0.94,""evidence"":[{""element"":""prior_state"",""quote"":""too many recipes for action"",""source_scope"":""target""},{""element"":""inadequacy_or_failure"",""quote"":""they rar...",092945b4d6a69d359d103380f9ccfeb89a0324469a1e27d36e98461c27963ada,0.006674,ok,2026-07-22T22:30:06.383808+00:00,2026-07-22T22:30:08.462434+00:00
248,82ff8879b8f5d081b0e59c4f93105ada16cabc9e6550c3102fc7725de852c53b,prelangchain_ab_v1_2_api_hardened,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,openai,gpt-5.6-terra,9f79900c8bed0b7477

In [ ]:
# PHASE 2 — ANTHROPIC ONLY
PHASE2_ANTHROPIC_RAW = pd.DataFrame()
if PHASE2_TASKS and provider_execution_ready('anthropic'):
    PHASE2_ANTHROPIC_RAW = safe_run_provider_cell('anthropic', PHASE2_TASKS, PHASE2_NEW_CONDITIONS)
    display(PHASE2_ANTHROPIC_RAW.tail())
else:
    print('Phase 2 Anthropic execution skipped.')

anthropic:phase2:   0%|          | 0/252 [00:00<?, ?it/s]

,run_key,protocol_version,dataset_sha256,provider,requested_model,model_config_sha256,phase,condition_id,row_id,seed,repeat_id,stage,sequence,definition_id,prompt_style,context_id,workflow,description,definition_name,definition_status,definition_sha256,schema_id,schema_sha256,prompt_sha256,cache_key,system_prompt_sha256,user_prompt_sha256,system_prompt,user_prompt,output_schema_json,source_blocks_json,created_at_utc,returned_model,request_id,raw_response_text,input_tokens,output_tokens,total_tokens,reasoning_tokens,cached_input_tokens,cache_creation_input_tokens,stop_reason,latency_seconds,accepted_parameters_json,provider_metadata_json,parsed_output_json,response_sha256,estimated_cost_usd,status,attempt_started_at_utc,completed_at_utc
247,7d7ef133bd56fd1efc2b9c3c80df17dd1db0647f100046726779be6cfdfd9f6b,prelangchain_ab_v1_2_api_hardened,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,anthropic,claude-haiku-4-5-20251001,8090d745697d7da125cd149fd7001c7978c6b4cafdc2156f22e73eec3857c1a2,phase2,P1_DIRECT__C4_plusminus2,P041,17,seed_17_r1,joint,247,D0_none,P1_direct,C4_plusminus2,W1_one_stage,Direct task; no supplied definition. Context treatment C4_plusminus2.,No supplied definition,control,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855,simple_joint,b6fae6ca706adbcd29328a1015fe3b7e1788e2e54977bfe56b5b790981eb481e,7a94486ea9ca8f5937de8c3e81b300a5f29a6bf825271ed0ce5c8b82a9d17576,prelangchain_ab_v1_2_api_hardened:D0_none:P1_direct:joint:simple_joint,964284c470c6697c61ba4b976eef1bdf9bfe9567eaf201d7bd2e2ec81a07c246,9a5bf6f53d0b29066ed7896f653e795bb21c7656e202f275886c98e89ba9d6ac,You are an independent qualitative-coding model for a reproducible benchmark on organizational unlearning in government agencies.\n\nClassify one TARGET paragraph per request. Never infer or reproduce a human label. ...,ROW ID: P041\n\n<METADATA>\nDocument title: The Katrina Effect on American Preparedness — A report on the lessons Americans learned in watching the Katrina catastrophe unfold\nSection heading: Conclusion\nPDF page: 1...,"{""$defs"":{""EvidenceQuote"":{""additionalProperties"":false,""properties"":{""element"":{""enum"":[""prior_state"",""inadequacy_or_failure"",""departure_or_reconfiguration"",""other""],""title"":""Element"",""type"":""string""},""quote"":{""anyO...","{""metadata"":""Document title: The Katrina Effect on American Preparedness — A report on the lessons Americans learned in watching the Katrina catastrophe unfold\nSection heading: Conclusion\nPDF page: 10"",""next_1"":""Cr...",2026-07-22T23:00:02.180752+00:00,claude-haiku-4-5-20251001,msg_011CdHvgqUYEY1T67EmxAzXn,"{""unlearning_present"": true, ""unlearning_mode"": ""adaptive_reconfiguration"", ""change_type"": ""revise_protocol_or_standard"", ""evidence"": [{""element"": ""prior_state"", ""quote"": ""too many cooks in the kitchen, as New Orlean...",1989,239,2228,0,0,0,end_turn,2.299158,"{""adapter_revision"":""api_hardened_2026_07_22_r1"",""cache_control"":{""type"":""ephemeral""},""cache_control_location"":""system[0].cache_control"",""experiment_seed_local_only"":17,""max_tokens"":1800,""parameter_firewall_removed"":...","{""sdk_version"":""0.118.0"",""stop_sequence"":null,""usage"":""Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference...","{""agency"":""government organizations (federal, state, local)"",""change_type"":""revise_protocol_or_standard"",""confidence"":0.85,""evidence"":[{""element"":""prior_state"",""quote"":""too many cooks in the kitchen, as New Orleans M...",f73d76d27f0ead37878b83ebdc36537ff4d00b526ef829d94df8930c38353bb6,0.003184,ok,2026-07-22T23:00:02.183033+00:00,2026-07-22T23:00:04.483556+00:00
248,7ba3a2129136e01b8ffc0f0fcad4f3efc1350c7ad3dd4853878890c07f80be93,prelangchain_ab_v1_2_api_hardened,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,anthropic,claude-haiku-4-5-20251001,8090d745697d7da125c

In [ ]:
# PHASE 2 — GEMINI ONLY
PHASE2_GEMINI_RAW = pd.DataFrame()
if PHASE2_TASKS and provider_execution_ready('gemini'):
    PHASE2_GEMINI_RAW = safe_run_provider_cell('gemini', PHASE2_TASKS, PHASE2_NEW_CONDITIONS)
    display(PHASE2_GEMINI_RAW.tail())
else:
    print('Phase 2 Gemini execution skipped.')

gemini:phase2:   0%|          | 0/252 [00:00<?, ?it/s]

Quota error 1/8. Retrying in 48.8 seconds.
ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite\nPlease retry in 46.786431442s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'m

KeyboardInterrupt: 

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ================================================================
# COLAB CHECKPOINT — COPY PHASE 1 + PHASE 2 PROGRESS TO GOOGLE DRIVE
#
# Safe to rerun. Existing identical files are skipped.
# Changed files are copied to a temporary path, hash-verified,
# and then promoted to the final destination.
#
# Run this AFTER the currently executing provider cell returns.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
from typing import Any
import hashlib
import json
import os
import shutil

import pandas as pd


# ----------------------------------------------------------------
# 1. Persistent Drive workspace
# ----------------------------------------------------------------

DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Unlearning_Project/"
    "prelangchain_ab_v1_2_workspace"
).resolve()

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

PROTOCOL = str(
    globals().get(
        "PROTOCOL_VERSION",
        "prelangchain_ab_v1_2_api_hardened",
    )
)


def notebook_path_variable(
    variable_name: str,
    fallback: Path,
) -> Path:
    """Read an existing notebook Path variable or use a safe fallback."""
    value = globals().get(variable_name)
    return Path(value).expanduser().resolve() if value is not None else fallback.resolve()


SOURCE_PROJECT_ROOT = notebook_path_variable(
    "PROJECT_ROOT",
    Path.cwd(),
)

SOURCE_RUNS_ROOT = notebook_path_variable(
    "RUNS_ROOT",
    SOURCE_PROJECT_ROOT / "runs" / PROTOCOL,
)

SOURCE_ARTIFACTS_ROOT = notebook_path_variable(
    "ARTIFACTS_ROOT",
    SOURCE_PROJECT_ROOT / "artifacts" / PROTOCOL,
)

SOURCE_REPORTS_ROOT = notebook_path_variable(
    "REPORTS_ROOT",
    SOURCE_PROJECT_ROOT / "reports" / PROTOCOL,
)

SOURCE_CONFIG_ROOT = notebook_path_variable(
    "CONFIG_ROOT",
    SOURCE_PROJECT_ROOT / "configs" / PROTOCOL,
)


DESTINATION_ROOTS = {
    "runs": DRIVE_PROJECT_ROOT / "runs" / PROTOCOL,
    "artifacts": DRIVE_PROJECT_ROOT / "artifacts" / PROTOCOL,
    "reports": DRIVE_PROJECT_ROOT / "reports" / PROTOCOL,
    "configs": DRIVE_PROJECT_ROOT / "configs" / PROTOCOL,
}

SOURCE_ROOTS = {
    "runs": SOURCE_RUNS_ROOT,
    "artifacts": SOURCE_ARTIFACTS_ROOT,
    "reports": SOURCE_REPORTS_ROOT,
    "configs": SOURCE_CONFIG_ROOT,
}

for destination in DESTINATION_ROOTS.values():
    destination.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------
# 2. Hashing and verified-copy utilities
# ----------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def verified_atomic_copy(
    source: Path,
    destination: Path,
) -> dict[str, Any]:
    """
    Copy one file and verify its SHA-256 digest.

    Returns a manifest record. Identical destination files are skipped.
    """
    source = source.resolve()
    destination = destination.resolve()

    if source == destination:
        return {
            "status": "already_on_drive",
            "source": str(source),
            "destination": str(destination),
            "size_bytes": source.stat().st_size,
            "sha256": sha256_file(source),
        }

    destination.parent.mkdir(parents=True, exist_ok=True)

    source_hash_before = sha256_file(source)
    source_size_before = source.stat().st_size

    if destination.exists():
        try:
            if (
                destination.stat().st_size == source_size_before
                and sha256_file(destination) == source_hash_before
            ):
                return {
                    "status": "unchanged",
                    "source": str(source),
                    "destination": str(destination),
                    "size_bytes": source_size_before,
                    "sha256": source_hash_before,
                }
        except OSError:
            # A damaged destination is simply replaced below.
            pass

    temporary_destination = destination.with_name(
        destination.name + f".partial-{os.getpid()}"
    )

    if temporary_destination.exists():
        temporary_destination.unlink()

    shutil.copy2(source, temporary_destination)

    temporary_hash = sha256_file(temporary_destination)

    if temporary_hash != source_hash_before:
        temporary_destination.unlink(missing_ok=True)
        raise IOError(
            "Backup verification failed before promotion:\n"
            f"source={source}\n"
            f"destination={destination}\n"
            f"source_sha256={source_hash_before}\n"
            f"temporary_sha256={temporary_hash}"
        )

    # Recheck the source in case it changed while being copied.
    source_hash_after = sha256_file(source)

    if source_hash_after != source_hash_before:
        temporary_destination.unlink(missing_ok=True)
        raise RuntimeError(
            f"{source} changed while it was being copied. "
            "Run the checkpoint cell again after the provider cell is idle."
        )

    os.replace(temporary_destination, destination)

    destination_hash = sha256_file(destination)

    if destination_hash != source_hash_before:
        raise IOError(
            "Backup verification failed after promotion:\n"
            f"source={source}\n"
            f"destination={destination}"
        )

    return {
        "status": "copied",
        "source": str(source),
        "destination": str(destination),
        "size_bytes": source_size_before,
        "sha256": source_hash_before,
    }


def copy_directory_tree(
    source_root: Path,
    destination_root: Path,
    category: str,
) -> list[dict[str, Any]]:
    """Copy every regular file below a source directory."""
    records: list[dict[str, Any]] = []

    if not source_root.exists():
        records.append({
            "category": category,
            "status": "source_missing",
            "source": str(source_root),
            "destination": str(destination_root),
            "size_bytes": None,
            "sha256": None,
        })
        return records

    files = sorted(
        path
        for path in source_root.rglob("*")
        if (
            path.is_file()
            and ".partial-" not in path.name
            and not path.name.endswith(".tmp")
        )
    )

    for source_file in files:
        relative_path = source_file.relative_to(source_root)
        destination_file = destination_root / relative_path

        record = verified_atomic_copy(
            source_file,
            destination_file,
        )
        record["category"] = category
        record["relative_path"] = str(relative_path)
        records.append(record)

    return records


# ----------------------------------------------------------------
# 3. Copy runs, configs, artifacts, and reports
# ----------------------------------------------------------------

backup_records: list[dict[str, Any]] = []

for category, source_root in SOURCE_ROOTS.items():
    backup_records.extend(
        copy_directory_tree(
            source_root=source_root,
            destination_root=DESTINATION_ROOTS[category],
            category=category,
        )
    )


# ----------------------------------------------------------------
# 4. Copy the frozen input workbook
# ----------------------------------------------------------------

INPUT_WORKBOOK_VALUE = globals().get("INPUT_WORKBOOK")
drive_input_workbook = None

if INPUT_WORKBOOK_VALUE is not None:
    source_input_workbook = Path(
        INPUT_WORKBOOK_VALUE
    ).expanduser().resolve()

    if source_input_workbook.is_file():
        drive_input_workbook = (
            DRIVE_PROJECT_ROOT
            / "data"
            / source_input_workbook.name
        )

        input_record = verified_atomic_copy(
            source_input_workbook,
            drive_input_workbook,
        )
        input_record["category"] = "input_workbook"
        input_record["relative_path"] = str(
            Path("data") / source_input_workbook.name
        )
        backup_records.append(input_record)
    else:
        print(
            "WARNING: INPUT_WORKBOOK was defined but was not found:",
            source_input_workbook,
        )
else:
    print(
        "WARNING: INPUT_WORKBOOK is not currently defined. "
        "The run logs will still be backed up."
    )


# ----------------------------------------------------------------
# 5. Save currently available Phase 1/2 DataFrames
#
# Raw JSONL remains the authoritative source. These compressed CSVs
# are convenience checkpoints for inspection and emergency recovery.
# ----------------------------------------------------------------

checkpoint_table_root = (
    DRIVE_PROJECT_ROOT
    / "checkpoints"
    / PROTOCOL
    / "tables"
)
checkpoint_table_root.mkdir(parents=True, exist_ok=True)

saved_dataframe_names: list[str] = []

for variable_name, value in list(globals().items()):
    is_phase_table = variable_name.startswith(
        ("PHASE1_", "PHASE2_")
    )

    if is_phase_table and isinstance(value, pd.DataFrame):
        output_path = (
            checkpoint_table_root
            / f"{variable_name.lower()}.csv.gz"
        )
        temporary_path = output_path.with_name(
            output_path.name + f".partial-{os.getpid()}"
        )

        value.to_csv(
            temporary_path,
            index=False,
            compression="gzip",
        )

        os.replace(temporary_path, output_path)

        backup_records.append({
            "category": "in_memory_dataframe",
            "status": "exported",
            "source": variable_name,
            "destination": str(output_path),
            "relative_path": str(
                output_path.relative_to(DRIVE_PROJECT_ROOT)
            ),
            "size_bytes": output_path.stat().st_size,
            "sha256": sha256_file(output_path),
            "rows": len(value),
            "columns": len(value.columns),
        })

        saved_dataframe_names.append(variable_name)


# ----------------------------------------------------------------
# 6. Save in-memory selection dictionaries when available
# ----------------------------------------------------------------

checkpoint_state_root = (
    DRIVE_PROJECT_ROOT
    / "checkpoints"
    / PROTOCOL
    / "state"
)
checkpoint_state_root.mkdir(parents=True, exist_ok=True)


def atomic_write_json(
    destination: Path,
    payload: Any,
) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)

    temporary = destination.with_name(
        destination.name + f".partial-{os.getpid()}"
    )

    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

    os.replace(temporary, destination)


selection_variables = [
    "PHASE1_SELECTION",
    "PHASE2_SELECTION",
]

saved_selection_variables = []

for variable_name in selection_variables:
    value = globals().get(variable_name)

    if isinstance(value, dict):
        state_path = (
            checkpoint_state_root
            / f"{variable_name.lower()}.json"
        )
        atomic_write_json(state_path, value)
        saved_selection_variables.append(variable_name)


# ----------------------------------------------------------------
# 7. Build a complete checkpoint manifest
# ----------------------------------------------------------------

checkpoint_time = datetime.now(timezone.utc)
checkpoint_stamp = checkpoint_time.strftime(
    "%Y%m%dT%H%M%SZ"
)

backup_metadata_root = (
    DRIVE_PROJECT_ROOT
    / "backups"
    / PROTOCOL
)
backup_metadata_root.mkdir(parents=True, exist_ok=True)

manifest_df = pd.DataFrame(backup_records)

manifest_path = (
    backup_metadata_root
    / f"checkpoint_manifest_{checkpoint_stamp}.csv"
)

manifest_df.to_csv(
    manifest_path,
    index=False,
)

phase1_selection_file = (
    DESTINATION_ROOTS["configs"]
    / "phase1_selection.json"
)

phase2_selection_file = (
    DESTINATION_ROOTS["configs"]
    / "phase2_selection.json"
)

metadata = {
    "checkpoint_created_at_utc": checkpoint_time.isoformat(),
    "protocol_version": PROTOCOL,
    "adapter_revision": globals().get("ADAPTER_REVISION"),
    "source_project_root": str(SOURCE_PROJECT_ROOT),
    "drive_project_root": str(DRIVE_PROJECT_ROOT),
    "source_roots": {
        name: str(path)
        for name, path in SOURCE_ROOTS.items()
    },
    "destination_roots": {
        name: str(path)
        for name, path in DESTINATION_ROOTS.items()
    },
    "input_workbook_on_drive": (
        str(drive_input_workbook)
        if drive_input_workbook is not None
        else None
    ),
    "dataset_sha256": globals().get("DATASET_SHA256"),
    "run_replicate_id": globals().get(
        "RUN_REPLICATE_ID",
        "r1",
    ),
    "active_phases": list(
        globals().get("ACTIVE_PHASES", ())
    ),
    "active_seeds": list(
        globals().get("ACTIVE_SEEDS", ())
    ),
    "stability_seeds": list(
        globals().get("STABILITY_SEEDS", ())
    ),
    "enabled_providers": list(
        globals().get("ENABLED_PROVIDERS", ())
    ),
    "phase1_selection_file_exists": (
        phase1_selection_file.exists()
    ),
    "phase2_selection_file_exists": (
        phase2_selection_file.exists()
    ),
    "saved_dataframe_names": sorted(
        saved_dataframe_names
    ),
    "saved_selection_variables": sorted(
        saved_selection_variables
    ),
    "manifest_path": str(manifest_path),
    "files_copied": int(
        manifest_df["status"].eq("copied").sum()
    ) if not manifest_df.empty else 0,
    "files_unchanged": int(
        manifest_df["status"].eq("unchanged").sum()
    ) if not manifest_df.empty else 0,
    "files_exported": int(
        manifest_df["status"].eq("exported").sum()
    ) if not manifest_df.empty else 0,
    "total_manifest_records": len(manifest_df),
}

checkpoint_metadata_path = (
    backup_metadata_root
    / f"checkpoint_{checkpoint_stamp}.json"
)

latest_checkpoint_path = (
    backup_metadata_root
    / "LATEST_CHECKPOINT.json"
)

atomic_write_json(
    checkpoint_metadata_path,
    metadata,
)

atomic_write_json(
    latest_checkpoint_path,
    metadata,
)


# ----------------------------------------------------------------
# 8. Final verification and readable summary
# ----------------------------------------------------------------

print("\n" + "=" * 72)
print("GOOGLE DRIVE CHECKPOINT COMPLETED")
print("=" * 72)

print("Drive project root:")
print(DRIVE_PROJECT_ROOT)

print("\nProtocol:")
print(PROTOCOL)

print("\nManifest:")
print(manifest_path)

print("\nCopied files:", metadata["files_copied"])
print("Unchanged files:", metadata["files_unchanged"])
print("Exported DataFrames:", metadata["files_exported"])

print("\nPhase 1 selection backed up:")
print(phase1_selection_file.exists())

print("\nPhase 2 selection backed up:")
print(phase2_selection_file.exists())

if not phase1_selection_file.exists():
    print(
        "\nWARNING: phase1_selection.json is missing. "
        "Run the Phase 1 selection cell and checkpoint again."
    )

if not phase2_selection_file.exists():
    print(
        "\nNOTICE: phase2_selection.json does not exist yet. "
        "This is expected while Phase 2 execution is still underway. "
        "After Phase 2 analysis and selection finish, run this "
        "checkpoint cell one more time."
    )

print("\nLatest-checkpoint metadata:")
print(latest_checkpoint_path)

print("\nBackup verification status:")
if not manifest_df.empty:
    display(
        manifest_df[
            [
                column
                for column in [
                    "category",
                    "status",
                    "relative_path",
                    "size_bytes",
                    "sha256",
                ]
                if column in manifest_df.columns
            ]
        ]
    )


GOOGLE DRIVE CHECKPOINT COMPLETED
Drive project root:
/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace

Protocol:
prelangchain_ab_v1_2_api_hardened

Manifest:
/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/backups/prelangchain_ab_v1_2_api_hardened/checkpoint_manifest_20260722T235530Z.csv

Copied files: 41
Unchanged files: 0
Exported DataFrames: 17

Phase 1 selection backed up:
True

Phase 2 selection backed up:
False

NOTICE: phase2_selection.json does not exist yet. This is expected while Phase 2 execution is still underway. After Phase 2 analysis and selection finish, run this checkpoint cell one more time.

Latest-checkpoint metadata:
/content/drive/MyDrive/Unlearning_Project/prelangchain_ab_v1_2_workspace/backups/prelangchain_ab_v1_2_api_hardened/LATEST_CHECKPOINT.json

Backup verification status:


,category,status,relative_path,size_bytes,sha256
0,runs,copied,phase1/anthropic/latest_results.csv,2952345,4d3b3be785a9d31fa357166fb8eb452b423555d33602294c0899c82a48201ad2
1,runs,copied,phase1/anthropic/provider_status.json,657,ea92b6698c62ab323389a1b73068ffeffabb6256b74c9235226c027fa70bec1c
2,runs,copied,phase1/anthropic/requests.jsonl,2422674,fd1b45c97382d032fb67877acf6d3715ef3eec4ecfa9b0e0a6f65478f9e5a880
3,runs,copied,phase1/anthropic/results.jsonl,3261508,b35e5a646fa62b4f2c5f87340f6ea2be4036e246c068e0c0fdadde82ab622667
4,runs,copied,phase1/gemini/errors.jsonl,258095,4598b2f0d896f87cce761d93b73c4cd6b97aa5f7f444a883efed9b3f0465bd6d
5,runs,copied,phase1/gemini/latest_results.csv,2949066,16de352459265dfb1ea5e4119a8466991daba484a18a7f5b49cfd1d10d0bb473
6,runs,copied,phase1/gemini/provider_status.json,648,e30305a1195fb7682f0a68b8a13ccb89fbf4140dc52c92b182960729c1743f2b
7,runs,copied,phase1/gemini/requests.jsonl,2570255,23eabb8da119a544abf1bdb220b165bf627dd3f02f0a3c702cb12cbff5c68ced
8,runs,copied,phase1/gemini/results.jsonl,3524824,0623498f896bd9c7f473e985b1adb876a8a485b048aa68248979505d4ac6583f
9,runs,copied,phase1/openai/latest_results.csv,2986555,3c7c17e3c44d78879d39906fe278b5f99d8be0f30a1ef4d21ebd54ce547e1faa


In [39]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Phase 2 assembly, analysis, and selection

In [40]:
def reused_phase1_c1_rows() -> pd.DataFrame:
    if 'PHASE1_SCORED' not in globals() or PHASE1_SCORED.empty:
        return pd.DataFrame()
    mapping = {
        base_id: f'{base_id}__C1_target'
        for base_id in PHASE1_SELECTED_FOR_CONTEXT
    }
    rows = PHASE1_SCORED[PHASE1_SCORED['condition_id'].isin(mapping)].copy()
    if rows.empty:
        return rows
    rows['source_phase'] = 'phase1'
    rows['phase'] = 'phase2'
    rows['condition_id'] = rows['condition_id'].map(mapping)
    rows['context_id'] = 'C1_target'
    rows['reused_from_phase1'] = True
    return rows

def prepare_phase2_scored() -> pd.DataFrame:
    new = prepare_scored_predictions('phase2')
    if not new.empty:
        new['reused_from_phase1'] = False
        new['source_phase'] = 'phase2'
    reused = reused_phase1_c1_rows()
    frames = [frame for frame in [reused,new] if not frame.empty]
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()

PHASE2_SCORED = prepare_phase2_scored()
PHASE2_RANKING, PHASE2_PROVIDER_METRICS, PHASE2_DOCUMENT_METRICS = condition_ranking(PHASE2_SCORED)
PHASE2_MCNEMAR = mcnemar_exact_table(PHASE2_SCORED)
PHASE2_GEE = fit_correctness_gee(PHASE2_SCORED, 'phase2')
if not PHASE2_RANKING.empty:
    display(PHASE2_RANKING)
    display(PHASE2_DOCUMENT_METRICS.sort_values(['condition_id','provider','document_title']))
else:
    print('No Phase 2 successful predictions available yet.')

No Phase 2 successful predictions available yet.


In [41]:
if PHASE2_SELECTION_OVERRIDE:
    if PHASE2_SELECTION_OVERRIDE not in set(PHASE2_RANKING.get('condition_id', [])):
        raise ValueError(f'Unknown PHASE2_SELECTION_OVERRIDE={PHASE2_SELECTION_OVERRIDE}')
    PHASE2_SELECTED_ID = PHASE2_SELECTION_OVERRIDE
else:
    selected = select_top_condition_ids(PHASE2_RANKING, n=1)
    PHASE2_SELECTED_ID = selected[0] if selected else ''
PHASE2_SELECTION = {
    'selected_condition_id':PHASE2_SELECTED_ID,
    'selection_basis':'override' if PHASE2_SELECTION_OVERRIDE else 'preregistered constrained ranking',
    'provisional_gold':not complete_gold_column('gold_final_definition'),
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
}
atomic_write_text(
    CONFIG_ROOT / 'phase2_selection.json',
    json.dumps(PHASE2_SELECTION, indent=2, default=str)
)
print(PHASE2_SELECTION)

{'selected_condition_id': '', 'selection_basis': 'preregistered constrained ranking', 'provisional_gold': True, 'created_at_utc': '2026-07-23T21:47:20.176447+00:00'}


## 20. Phase 3 — one-stage joint vs binary-first hierarchical workflow

In [ ]:
ACTIVE_PHASES = ("phase3",)

print("ACTIVE_PHASES:", ACTIVE_PHASES)

In [ ]:
def resolve_phase2_selected_id() -> str:
    if PHASE2_SELECTION_OVERRIDE:
        return PHASE2_SELECTION_OVERRIDE
    if 'PHASE2_SELECTED_ID' in globals() and PHASE2_SELECTED_ID:
        return PHASE2_SELECTED_ID
    return str(load_selection_json('phase2_selection.json').get('selected_condition_id',''))

def selected_phase2_condition() -> Optional[ConditionSpec]:
    selected_id = resolve_phase2_selected_id()
    mapping = conditions_by_id(PHASE2_CONDITIONS) if PHASE2_CONDITIONS else {}
    return mapping.get(selected_id)

PHASE2_WINNER = selected_phase2_condition()
PHASE3_CONDITIONS: list[ConditionSpec] = []
if PHASE2_WINNER is not None:
    common = dict(
        phase='phase3', definition_id=PHASE2_WINNER.definition_id,
        prompt_style=PHASE2_WINNER.prompt_style,
        context_id=PHASE2_WINNER.context_id,
    )
    PHASE3_CONDITIONS = [
        ConditionSpec(
            condition_id='W1_ONE_STAGE', workflow='W1_one_stage',
            description='One request jointly returns binary decision, structured evidence, target, and agency.',
            **common,
        ),
        ConditionSpec(
            condition_id='W2_BINARY_FIRST', workflow='W2_binary_first',
            description='Stage 1 returns binary/evidence; Stage 2 assigns target/agency only after a positive.',
            **common,
        ),
    ]
PHASE3_W1 = [c for c in PHASE3_CONDITIONS if c.workflow == 'W1_one_stage']
PHASE3_W2 = [c for c in PHASE3_CONDITIONS if c.workflow == 'W2_binary_first']
PHASE3_W1_TASKS = make_task_schedule(BENCHMARK, PHASE3_W1, ACTIVE_SEEDS, 'phase3', 'joint') if BENCHMARK is not None and PHASE3_W1 and 'phase3' in ACTIVE_PHASES else []
PHASE3_W2_BINARY_TASKS = make_task_schedule(BENCHMARK, PHASE3_W2, ACTIVE_SEEDS, 'phase3', 'binary') if BENCHMARK is not None and PHASE3_W2 and 'phase3' in ACTIVE_PHASES else []
PHASE3_W2_TARGET_TASKS = make_task_schedule(BENCHMARK, PHASE3_W2, ACTIVE_SEEDS, 'phase3', 'target') if BENCHMARK is not None and PHASE3_W2 and 'phase3' in ACTIVE_PHASES else []
display(pd.DataFrame([asdict(c) for c in PHASE3_CONDITIONS]))
print({
    'phase2_winner':None if PHASE2_WINNER is None else PHASE2_WINNER.condition_id,
    'W1_tasks_per_provider':len(PHASE3_W1_TASKS),
    'W2_stage1_tasks_per_provider':len(PHASE3_W2_BINARY_TASKS),
    'W2_stage2_ceiling_per_provider':len(PHASE3_W2_TARGET_TASKS),
})

### Phase 3A — isolated one-stage provider cells

In [ ]:
# PHASE 3 W1 — OPENAI ONLY
PHASE3_W1_OPENAI_RAW = pd.DataFrame()
if PHASE3_W1_TASKS and provider_execution_ready('openai'):
    PHASE3_W1_OPENAI_RAW = safe_run_provider_cell('openai', PHASE3_W1_TASKS, PHASE3_W1)
    display(PHASE3_W1_OPENAI_RAW.tail())
else:
    print('Phase 3 W1 OpenAI execution skipped.')

In [ ]:
# PHASE 3 W1 — ANTHROPIC ONLY
PHASE3_W1_ANTHROPIC_RAW = pd.DataFrame()
if PHASE3_W1_TASKS and provider_execution_ready('anthropic'):
    PHASE3_W1_ANTHROPIC_RAW = safe_run_provider_cell('anthropic', PHASE3_W1_TASKS, PHASE3_W1)
    display(PHASE3_W1_ANTHROPIC_RAW.tail())
else:
    print('Phase 3 W1 Anthropic execution skipped.')

In [ ]:
# PHASE 3 W1 — GEMINI ONLY
PHASE3_W1_GEMINI_RAW = pd.DataFrame()
if PHASE3_W1_TASKS and provider_execution_ready('gemini'):
    PHASE3_W1_GEMINI_RAW = safe_run_provider_cell('gemini', PHASE3_W1_TASKS, PHASE3_W1)
    display(PHASE3_W1_GEMINI_RAW.tail())
else:
    print('Phase 3 W1 Gemini execution skipped.')

### Phase 3B — isolated binary-first Stage-1 provider cells

In [ ]:
# PHASE 3 W2 STAGE 1 — OPENAI ONLY
PHASE3_W2_BINARY_OPENAI_RAW = pd.DataFrame()
if PHASE3_W2_BINARY_TASKS and provider_execution_ready('openai'):
    PHASE3_W2_BINARY_OPENAI_RAW = safe_run_provider_cell('openai', PHASE3_W2_BINARY_TASKS, PHASE3_W2)
    display(PHASE3_W2_BINARY_OPENAI_RAW.tail())
else:
    print('Phase 3 W2 Stage 1 OpenAI execution skipped.')

In [ ]:
# PHASE 3 W2 STAGE 1 — ANTHROPIC ONLY
PHASE3_W2_BINARY_ANTHROPIC_RAW = pd.DataFrame()
if PHASE3_W2_BINARY_TASKS and provider_execution_ready('anthropic'):
    PHASE3_W2_BINARY_ANTHROPIC_RAW = safe_run_provider_cell('anthropic', PHASE3_W2_BINARY_TASKS, PHASE3_W2)
    display(PHASE3_W2_BINARY_ANTHROPIC_RAW.tail())
else:
    print('Phase 3 W2 Stage 1 Anthropic execution skipped.')

In [ ]:
# PHASE 3 W2 STAGE 1 — GEMINI ONLY
PHASE3_W2_BINARY_GEMINI_RAW = pd.DataFrame()
if PHASE3_W2_BINARY_TASKS and provider_execution_ready('gemini'):
    PHASE3_W2_BINARY_GEMINI_RAW = safe_run_provider_cell('gemini', PHASE3_W2_BINARY_TASKS, PHASE3_W2)
    display(PHASE3_W2_BINARY_GEMINI_RAW.tail())
else:
    print('Phase 3 W2 Stage 1 Gemini execution skipped.')

In [ ]:
def positive_stage1_lookup(phase: str, provider: str, condition_id: str) -> dict[tuple[str,str,str,int,str],dict[str,Any]]:
    predictions = successful_predictions(phase)
    if predictions.empty:
        return {}
    subset = predictions[
        predictions['provider'].eq(provider)
        & predictions['condition_id'].eq(condition_id)
        & predictions['stage'].eq('binary')
        & predictions['out__unlearning_present'].eq(True)
    ]
    lookup = {}
    for _, record in subset.iterrows():
        lookup[parsed_stage1_lookup_key(
            provider, condition_id, record['row_id'], int(record['seed']), record['repeat_id']
        )] = json.loads(record['parsed_output_json'])
    return lookup

PHASE3_STAGE1_LOOKUPS = {
    provider: positive_stage1_lookup('phase3', provider, 'W2_BINARY_FIRST')
    for provider in MODEL_CONFIGS
}
print({provider:len(lookup) for provider,lookup in PHASE3_STAGE1_LOOKUPS.items()})

### Phase 3C — isolated Stage-2 target/agency provider cells

In [ ]:
# PHASE 3 W2 STAGE 2 — OPENAI ONLY
PHASE3_W2_TARGET_OPENAI_RAW = pd.DataFrame()
if PHASE3_W2_TARGET_TASKS and PHASE3_STAGE1_LOOKUPS.get('openai') and provider_execution_ready('openai'):
    PHASE3_W2_TARGET_OPENAI_RAW = safe_run_provider_cell(
        'openai', PHASE3_W2_TARGET_TASKS, PHASE3_W2, PHASE3_STAGE1_LOOKUPS['openai']
    )
    display(PHASE3_W2_TARGET_OPENAI_RAW.tail())
else:
    print('Phase 3 W2 Stage 2 OpenAI execution skipped or no positive Stage-1 rows.')

In [ ]:
# PHASE 3 W2 STAGE 2 — ANTHROPIC ONLY
PHASE3_W2_TARGET_ANTHROPIC_RAW = pd.DataFrame()
if PHASE3_W2_TARGET_TASKS and PHASE3_STAGE1_LOOKUPS.get('anthropic') and provider_execution_ready('anthropic'):
    PHASE3_W2_TARGET_ANTHROPIC_RAW = safe_run_provider_cell(
        'anthropic', PHASE3_W2_TARGET_TASKS, PHASE3_W2, PHASE3_STAGE1_LOOKUPS['anthropic']
    )
    display(PHASE3_W2_TARGET_ANTHROPIC_RAW.tail())
else:
    print('Phase 3 W2 Stage 2 Anthropic execution skipped or no positive Stage-1 rows.')

In [ ]:
# PHASE 3 W2 STAGE 2 — GEMINI ONLY
PHASE3_W2_TARGET_GEMINI_RAW = pd.DataFrame()
if PHASE3_W2_TARGET_TASKS and PHASE3_STAGE1_LOOKUPS.get('gemini') and provider_execution_ready('gemini'):
    PHASE3_W2_TARGET_GEMINI_RAW = safe_run_provider_cell(
        'gemini', PHASE3_W2_TARGET_TASKS, PHASE3_W2, PHASE3_STAGE1_LOOKUPS['gemini']
    )
    display(PHASE3_W2_TARGET_GEMINI_RAW.tail())
else:
    print('Phase 3 W2 Stage 2 Gemini execution skipped or no positive Stage-1 rows.')

### Phase 3 workflow assembly and evaluation

In [ ]:
MERGE_KEYS = ['provider','condition_id','row_id','seed','repeat_id']
USAGE_COLUMNS = [
    'input_tokens','output_tokens','total_tokens','reasoning_tokens',
    'cached_input_tokens','cache_creation_input_tokens','estimated_cost_usd','latency_seconds'
]

def assemble_workflow_predictions(phase: str) -> pd.DataFrame:
    raw = successful_predictions(phase)
    if raw.empty:
        return raw
    joint = raw[raw['stage'].eq('joint')].copy()
    binary = raw[raw['stage'].eq('binary')].copy()
    target = raw[raw['stage'].eq('target')].copy()
    if not joint.empty:
        joint['workflow_complete'] = True
        joint['stage2_called'] = False
    if not binary.empty:
        target_fields = MERGE_KEYS + [
            'out__target_type','out__agency','out__target_evidence_quote',
            'out__target_evidence_scope','out__confidence','out__needs_human_review'
        ] + [column for column in USAGE_COLUMNS if column in target.columns]
        target_keep = target[[c for c in target_fields if c in target.columns]].copy()
        rename = {
            'out__target_type':'stage2__target_type',
            'out__agency':'stage2__agency',
            'out__target_evidence_quote':'stage2__target_evidence_quote',
            'out__target_evidence_scope':'stage2__target_evidence_scope',
            'out__confidence':'stage2__confidence',
            'out__needs_human_review':'stage2__needs_human_review',
            **{column:f'stage2__{column}' for column in USAGE_COLUMNS if column in target_keep.columns},
        }
        target_keep = target_keep.rename(columns=rename)
        binary = binary.merge(target_keep, on=MERGE_KEYS, how='left', validate='one_to_one')
        positive = binary['out__unlearning_present'].eq(True)
        binary['out__target_type'] = np.where(
            positive, binary.get('stage2__target_type'), 'none'
        )
        binary['out__agency'] = np.where(
            positive, binary.get('stage2__agency'), None
        )
        binary['stage2_called'] = positive & binary.get('stage2__target_type', pd.Series(index=binary.index,dtype=object)).notna()
        binary['workflow_complete'] = (~positive) | binary['stage2_called']
        for column in USAGE_COLUMNS:
            left = pd.to_numeric(binary.get(column), errors='coerce').fillna(0)
            right = pd.to_numeric(binary.get(f'stage2__{column}'), errors='coerce').fillna(0)
            binary[column] = left + right
    frames = [frame for frame in [joint,binary] if not frame.empty]
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()

PHASE3_UNIFIED = assemble_workflow_predictions('phase3')
PHASE3_SCORED = attach_gold_and_predictions(add_evidence_audit(PHASE3_UNIFIED)) if not PHASE3_UNIFIED.empty else pd.DataFrame()
PHASE3_RANKING, PHASE3_PROVIDER_METRICS, PHASE3_DOCUMENT_METRICS = condition_ranking(PHASE3_SCORED)
PHASE3_MCNEMAR = mcnemar_exact_table(PHASE3_SCORED)
PHASE3_GEE = fit_correctness_gee(PHASE3_SCORED, 'phase3')
if not PHASE3_RANKING.empty:
    display(PHASE3_RANKING)
    display(PHASE3_PROVIDER_METRICS)
else:
    print('No Phase 3 successful workflow predictions available yet.')

In [ ]:
TARGET_NORMALIZATION = {
    'leadership':'leadership',
    'laws plans and policies':'laws_plans_policies',
    'laws plans policies':'laws_plans_policies',
    'laws_plans_policies':'laws_plans_policies',
    'capabilities':'capabilities',
    'funds and resources':'funds_resources',
    'funds resources':'funds_resources',
    'funds_resources':'funds_resources',
    'misc organizational':'misc_organizational',
    'miscellaneous organizational':'misc_organizational',
    'misc_organizational':'misc_organizational',
    'none':'none',
}

def normalize_target(value: Any) -> Optional[str]:
    text = normalize_for_match(value).replace('/',' ').replace('_',' ')
    text = re.sub(r'[^a-z0-9 ]+',' ',text)
    text = normalize_space(text)
    return TARGET_NORMALIZATION.get(text, text.replace(' ','_') if text else None)

def agency_token_set(value: Any) -> set[str]:
    return {
        token for token in re.findall(r'[a-z0-9]+', normalize_for_match(value))
        if token not in {'the','of','and','department','office','agency','government'}
    }

def target_agency_metrics(data: pd.DataFrame) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()
    scored = data.copy()
    scored['gold_target_norm'] = scored['gold_target'].map(normalize_target)
    scored['pred_target_norm'] = scored['out__target_type'].map(normalize_target)
    gold_positive = scored['gold_common_int'].eq(1)
    scored['target_exact'] = np.where(
        gold_positive,
        scored['gold_target_norm'].eq(scored['pred_target_norm']),
        np.nan,
    )
    scored['binary_target_exact'] = (
        scored['gold_common_int'].eq(scored['pred_int'])
        & np.where(gold_positive, scored['gold_target_norm'].eq(scored['pred_target_norm']), True)
    )
    scored['agency_overlap'] = scored.apply(
        lambda row: (
            np.nan if not bool(row['gold_common_int']) or not normalize_space(row.get('gold_agency'))
            else bool(agency_token_set(row.get('gold_agency')) & agency_token_set(row.get('out__agency')))
        ), axis=1
    )
    return (
        scored.groupby(['condition_id','provider','workflow'], dropna=False)
        .agg(
            rows=('row_id','count'),
            target_accuracy_on_gold_positive=('target_exact','mean'),
            binary_plus_target_exact=('binary_target_exact','mean'),
            agency_overlap_accuracy=('agency_overlap','mean'),
            workflow_complete_rate=('workflow_complete','mean'),
            stage2_call_rate=('stage2_called','mean'),
        )
        .reset_index()
    )

PHASE3_TARGET_METRICS = target_agency_metrics(PHASE3_SCORED)
display(PHASE3_TARGET_METRICS)

In [ ]:
if PHASE3_WORKFLOW_OVERRIDE:
    workflow_matches = PHASE3_RANKING[
        PHASE3_RANKING['workflow'].eq(PHASE3_WORKFLOW_OVERRIDE)
    ]
    if workflow_matches.empty:
        raise ValueError(f'Unknown PHASE3_WORKFLOW_OVERRIDE={PHASE3_WORKFLOW_OVERRIDE}')
    PHASE3_SELECTED_ID = workflow_matches.iloc[0]['condition_id']
else:
    selected = select_top_condition_ids(PHASE3_RANKING, n=1)
    PHASE3_SELECTED_ID = selected[0] if selected else ''
PHASE3_SELECTED_CONDITION = conditions_by_id(PHASE3_CONDITIONS).get(PHASE3_SELECTED_ID) if PHASE3_CONDITIONS else None
PHASE3_SELECTION = {
    'selected_condition_id':PHASE3_SELECTED_ID,
    'selected_workflow':None if PHASE3_SELECTED_CONDITION is None else PHASE3_SELECTED_CONDITION.workflow,
    'selection_basis':'override' if PHASE3_WORKFLOW_OVERRIDE else 'preregistered constrained ranking',
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
}
atomic_write_text(
    CONFIG_ROOT / 'phase3_selection.json',
    json.dumps(PHASE3_SELECTION, indent=2, default=str)
)
print(PHASE3_SELECTION)

## 21. Finalist stability experiment across multiple seeds

In [ ]:
def resolve_phase3_selected_condition() -> Optional[ConditionSpec]:
    if 'PHASE3_SELECTED_CONDITION' in globals() and PHASE3_SELECTED_CONDITION is not None:
        return PHASE3_SELECTED_CONDITION
    saved = load_selection_json('phase3_selection.json')
    selected_id = saved.get('selected_condition_id','')
    mapping = conditions_by_id(PHASE3_CONDITIONS) if PHASE3_CONDITIONS else {}
    return mapping.get(selected_id)

PHASE3_WINNER = resolve_phase3_selected_condition()
STABILITY_CONDITIONS: list[ConditionSpec] = []
if PHASE3_WINNER is not None:
    STABILITY_CONDITIONS = [replace(
        PHASE3_WINNER,
        condition_id='FINALIST_STABILITY',
        phase='stability',
        description='Selected pre-LangChain finalist repeated across registered seeds.',
    )]
STABILITY_WORKFLOW = STABILITY_CONDITIONS[0].workflow if STABILITY_CONDITIONS else None
STABILITY_JOINT_TASKS = (
    make_task_schedule(BENCHMARK, STABILITY_CONDITIONS, STABILITY_SEEDS, 'stability', 'joint')
    if BENCHMARK is not None and STABILITY_CONDITIONS and STABILITY_WORKFLOW == 'W1_one_stage' and 'stability' in ACTIVE_PHASES
    else []
)
STABILITY_BINARY_TASKS = (
    make_task_schedule(BENCHMARK, STABILITY_CONDITIONS, STABILITY_SEEDS, 'stability', 'binary')
    if BENCHMARK is not None and STABILITY_CONDITIONS and STABILITY_WORKFLOW == 'W2_binary_first' and 'stability' in ACTIVE_PHASES
    else []
)
STABILITY_TARGET_TASKS = (
    make_task_schedule(BENCHMARK, STABILITY_CONDITIONS, STABILITY_SEEDS, 'stability', 'target')
    if BENCHMARK is not None and STABILITY_CONDITIONS and STABILITY_WORKFLOW == 'W2_binary_first' and 'stability' in ACTIVE_PHASES
    else []
)
display(pd.DataFrame([asdict(c) for c in STABILITY_CONDITIONS]))
print({
    'workflow':STABILITY_WORKFLOW,
    'seeds':STABILITY_SEEDS,
    'joint_tasks_per_provider':len(STABILITY_JOINT_TASKS),
    'binary_tasks_per_provider':len(STABILITY_BINARY_TASKS),
    'target_task_ceiling_per_provider':len(STABILITY_TARGET_TASKS),
})

### Stability Stage 1 / one-stage provider cells

In [ ]:
# STABILITY — OPENAI JOINT OR BINARY ONLY
STABILITY_OPENAI_STAGE1_RAW = pd.DataFrame()
STABILITY_OPENAI_STAGE1_TASKS = STABILITY_JOINT_TASKS or STABILITY_BINARY_TASKS
if STABILITY_OPENAI_STAGE1_TASKS and provider_execution_ready('openai'):
    STABILITY_OPENAI_STAGE1_RAW = safe_run_provider_cell(
        'openai', STABILITY_OPENAI_STAGE1_TASKS, STABILITY_CONDITIONS
    )
    display(STABILITY_OPENAI_STAGE1_RAW.tail())
else:
    print('Stability OpenAI Stage 1/joint execution skipped.')

In [ ]:
# STABILITY — ANTHROPIC JOINT OR BINARY ONLY
STABILITY_ANTHROPIC_STAGE1_RAW = pd.DataFrame()
STABILITY_ANTHROPIC_STAGE1_TASKS = STABILITY_JOINT_TASKS or STABILITY_BINARY_TASKS
if STABILITY_ANTHROPIC_STAGE1_TASKS and provider_execution_ready('anthropic'):
    STABILITY_ANTHROPIC_STAGE1_RAW = safe_run_provider_cell(
        'anthropic', STABILITY_ANTHROPIC_STAGE1_TASKS, STABILITY_CONDITIONS
    )
    display(STABILITY_ANTHROPIC_STAGE1_RAW.tail())
else:
    print('Stability Anthropic Stage 1/joint execution skipped.')

In [ ]:
# STABILITY — GEMINI JOINT OR BINARY ONLY
STABILITY_GEMINI_STAGE1_RAW = pd.DataFrame()
STABILITY_GEMINI_STAGE1_TASKS = STABILITY_JOINT_TASKS or STABILITY_BINARY_TASKS
if STABILITY_GEMINI_STAGE1_TASKS and provider_execution_ready('gemini'):
    STABILITY_GEMINI_STAGE1_RAW = safe_run_provider_cell(
        'gemini', STABILITY_GEMINI_STAGE1_TASKS, STABILITY_CONDITIONS
    )
    display(STABILITY_GEMINI_STAGE1_RAW.tail())
else:
    print('Stability Gemini Stage 1/joint execution skipped.')

In [ ]:
STABILITY_STAGE1_LOOKUPS = {
    provider: positive_stage1_lookup('stability', provider, 'FINALIST_STABILITY')
    for provider in MODEL_CONFIGS
} if STABILITY_WORKFLOW == 'W2_binary_first' else {provider:{} for provider in MODEL_CONFIGS}
print({provider:len(lookup) for provider,lookup in STABILITY_STAGE1_LOOKUPS.items()})

### Stability hierarchical Stage-2 provider cells

In [ ]:
# STABILITY W2 STAGE 2 — OPENAI ONLY
STABILITY_OPENAI_TARGET_RAW = pd.DataFrame()
if STABILITY_TARGET_TASKS and STABILITY_STAGE1_LOOKUPS.get('openai') and provider_execution_ready('openai'):
    STABILITY_OPENAI_TARGET_RAW = safe_run_provider_cell(
        'openai', STABILITY_TARGET_TASKS, STABILITY_CONDITIONS,
        STABILITY_STAGE1_LOOKUPS['openai']
    )
    display(STABILITY_OPENAI_TARGET_RAW.tail())
else:
    print('Stability OpenAI Stage 2 skipped or not applicable.')

In [ ]:
# STABILITY W2 STAGE 2 — ANTHROPIC ONLY
STABILITY_ANTHROPIC_TARGET_RAW = pd.DataFrame()
if STABILITY_TARGET_TASKS and STABILITY_STAGE1_LOOKUPS.get('anthropic') and provider_execution_ready('anthropic'):
    STABILITY_ANTHROPIC_TARGET_RAW = safe_run_provider_cell(
        'anthropic', STABILITY_TARGET_TASKS, STABILITY_CONDITIONS,
        STABILITY_STAGE1_LOOKUPS['anthropic']
    )
    display(STABILITY_ANTHROPIC_TARGET_RAW.tail())
else:
    print('Stability Anthropic Stage 2 skipped or not applicable.')

In [ ]:
# STABILITY W2 STAGE 2 — GEMINI ONLY
STABILITY_GEMINI_TARGET_RAW = pd.DataFrame()
if STABILITY_TARGET_TASKS and STABILITY_STAGE1_LOOKUPS.get('gemini') and provider_execution_ready('gemini'):
    STABILITY_GEMINI_TARGET_RAW = safe_run_provider_cell(
        'gemini', STABILITY_TARGET_TASKS, STABILITY_CONDITIONS,
        STABILITY_STAGE1_LOOKUPS['gemini']
    )
    display(STABILITY_GEMINI_TARGET_RAW.tail())
else:
    print('Stability Gemini Stage 2 skipped or not applicable.')

### Stability metrics, repeat agreement, and tiered review

In [ ]:
STABILITY_UNIFIED = assemble_workflow_predictions('stability')
STABILITY_SCORED = attach_gold_and_predictions(add_evidence_audit(STABILITY_UNIFIED)) if not STABILITY_UNIFIED.empty else pd.DataFrame()
STABILITY_RUN_METRICS = summarize_binary(
    STABILITY_SCORED,
    ['provider','seed','repeat_id','condition_id','definition_id','prompt_style','context_id','workflow']
) if not STABILITY_SCORED.empty else pd.DataFrame()
STABILITY_METRIC_DISTRIBUTION = pd.DataFrame()
if not STABILITY_RUN_METRICS.empty:
    metric_columns = [
        'accuracy','balanced_accuracy','precision_yes','recall_yes','specificity',
        'f1_yes','mcc','cohen_kappa','brier_score','schema_valid_rate',
        'evidence_valid_rate','estimated_cost_usd','mean_latency_seconds'
    ]
    rows = []
    for provider, group in STABILITY_RUN_METRICS.groupby('provider'):
        for metric in metric_columns:
            values = pd.to_numeric(group[metric], errors='coerce').dropna()
            rows.append({
                'provider':provider,'metric':metric,'n_runs':len(values),
                'mean':values.mean(),'std':values.std(ddof=1) if len(values)>1 else 0.0,
                'min':values.min() if len(values) else np.nan,
                'max':values.max() if len(values) else np.nan,
            })
    STABILITY_METRIC_DISTRIBUTION = pd.DataFrame(rows)
    display(STABILITY_RUN_METRICS)
    display(STABILITY_METRIC_DISTRIBUTION)
else:
    print('No stability predictions available yet.')

In [ ]:
def provider_repeat_stability(data: pd.DataFrame) -> tuple[pd.DataFrame,pd.DataFrame]:
    if data.empty:
        return pd.DataFrame(), pd.DataFrame()
    rows = []
    for (provider,row_id), group in data.groupby(['provider','row_id']):
        values = group.sort_values('seed')['pred_int'].dropna().astype(int).tolist()
        counts = Counter(values)
        modal_count = max(counts.values()) if counts else 0
        rows.append({
            'provider':provider,'row_id':row_id,'n_repeats':len(values),
            'yes_repeats':counts.get(1,0),'no_repeats':counts.get(0,0),
            'modal_prediction':max(counts, key=counts.get) if counts else np.nan,
            'repeat_agreement_share':safe_rate(modal_count,len(values)),
            'changed_across_repeats':len(counts)>1,
        })
    row_stability = pd.DataFrame(rows)
    pairwise = []
    for provider, provider_data in data.groupby('provider'):
        pivot = provider_data.pivot_table(index='row_id', columns='seed', values='pred_int', aggfunc='first')
        for left_seed,right_seed in itertools.combinations(pivot.columns,2):
            paired = pivot[[left_seed,right_seed]].dropna()
            if paired.empty:
                continue
            pairwise.append({
                'provider':provider,'seed_left':left_seed,'seed_right':right_seed,
                'n_rows':len(paired),
                'percent_agreement':paired[left_seed].eq(paired[right_seed]).mean(),
                'cohen_kappa':cohen_kappa_score(paired[left_seed],paired[right_seed])
                if paired[left_seed].nunique()>1 or paired[right_seed].nunique()>1 else np.nan,
            })
    return row_stability, pd.DataFrame(pairwise)

PROVIDER_ROW_STABILITY, PROVIDER_SEED_PAIRWISE = provider_repeat_stability(STABILITY_SCORED)
display(PROVIDER_ROW_STABILITY.head(30))
display(PROVIDER_SEED_PAIRWISE)

In [ ]:
def tier_from_yes_votes(yes_votes: int, n_votes: int) -> str:
    if n_votes != 3:
        return 'INCOMPLETE'
    return {3:'Tier 1',2:'Tier 2',1:'Tier 3',0:'Tier 4'}[int(yes_votes)]

def build_tiered_review(data: pd.DataFrame) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()
    rows = []
    for (row_id,seed), group in data.groupby(['row_id','seed']):
        valid = group.dropna(subset=['pred_int'])
        yes_votes = int(valid['pred_int'].sum())
        n_votes = valid['provider'].nunique()
        tier = tier_from_yes_votes(yes_votes,n_votes)
        rows.append({
            'row_id':row_id,'seed':seed,'n_provider_votes':n_votes,
            'yes_votes':yes_votes,'no_votes':n_votes-yes_votes,
            'tier':tier,
            'majority_prediction':'Yes' if yes_votes>=2 else 'No',
            'providers_yes':' | '.join(sorted(valid.loc[valid['pred_int'].eq(1),'provider'].astype(str))),
            'providers_no':' | '.join(sorted(valid.loc[valid['pred_int'].eq(0),'provider'].astype(str))),
            'mean_confidence':valid['confidence'].mean(),
            'any_evidence_invalid':not bool(valid['evidence_valid'].fillna(False).all()),
        })
    return pd.DataFrame(rows).merge(
        BENCHMARK[['row_id','document_title','target_text','gold_historical','gold_final_definition']],
        on='row_id', how='left', validate='many_to_one'
    )

TIERED_REVIEW_BY_SEED = build_tiered_review(STABILITY_SCORED)
TIER_STABILITY = pd.DataFrame()
if not TIERED_REVIEW_BY_SEED.empty:
    tier_order = {'Tier 1':1,'Tier 2':2,'Tier 3':3,'Tier 4':4,'INCOMPLETE':9}
    rows=[]
    for row_id, group in TIERED_REVIEW_BY_SEED.groupby('row_id'):
        tiers=group.sort_values('seed')['tier'].tolist(); counts=Counter(tiers)
        rows.append({
            'row_id':row_id,'n_repeats':len(tiers),'tiers_by_seed':' | '.join(tiers),
            'distinct_tiers':len(counts),'modal_tier':counts.most_common(1)[0][0],
            'tier_stable':len(counts)==1,
            'tier_range':max(tier_order.get(t,9) for t in tiers)-min(tier_order.get(t,9) for t in tiers),
        })
    TIER_STABILITY = pd.DataFrame(rows).merge(
        BENCHMARK[['row_id','document_title','target_text']], on='row_id', how='left'
    )
    display(TIERED_REVIEW_BY_SEED.head(30))
    display(TIER_STABILITY.sort_values(['tier_stable','tier_range'], ascending=[True,False]))

In [ ]:
def fleiss_kappa_binary(vote_matrix: np.ndarray) -> float:
    matrix = np.asarray(vote_matrix, dtype=float)
    n_items, n_categories = matrix.shape
    n_raters = matrix.sum(axis=1)
    if n_items == 0 or np.any(n_raters < 2) or not np.allclose(n_raters,n_raters[0]):
        return np.nan
    n = n_raters[0]
    p = matrix.sum(axis=0) / (n_items*n)
    p_bar_e = np.sum(p**2)
    p_i = (np.sum(matrix**2,axis=1)-n)/(n*(n-1))
    p_bar = p_i.mean()
    return float((p_bar-p_bar_e)/(1-p_bar_e)) if p_bar_e < 1 else np.nan

def cross_provider_reliability(data: pd.DataFrame) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()
    rows=[]
    try:
        import krippendorff as kd
    except ImportError:
        kd=None
    for seed, group in data.groupby('seed'):
        pivot=group.pivot_table(index='row_id',columns='provider',values='pred_int',aggfunc='first')
        complete=pivot.dropna()
        if complete.empty:
            continue
        counts=np.column_stack([(complete==0).sum(axis=1),(complete==1).sum(axis=1)])
        alpha=np.nan
        if kd is not None:
            with contextlib.suppress(Exception):
                alpha=float(kd.alpha(reliability_data=complete.to_numpy().T,level_of_measurement='nominal'))
        rows.append({
            'seed':seed,'rows_complete':len(complete),'providers':len(complete.columns),
            'unanimous_agreement':complete.nunique(axis=1).eq(1).mean(),
            'fleiss_kappa':fleiss_kappa_binary(counts),
            'krippendorff_alpha':alpha,
        })
    return pd.DataFrame(rows)

CROSS_PROVIDER_RELIABILITY = cross_provider_reliability(STABILITY_SCORED)
display(CROSS_PROVIDER_RELIABILITY)

### Document-stratified bootstrap uncertainty

In [ ]:
def document_stratified_bootstrap(
    data: pd.DataFrame,
    group_columns: Sequence[str] = ('provider',),
    metrics: Sequence[str] = ('accuracy','recall_yes','specificity','f1_yes','mcc'),
    n_bootstrap: int = N_BOOTSTRAP,
    seed: int = 20260722,
) -> pd.DataFrame:
    if data.empty or n_bootstrap <= 0:
        return pd.DataFrame()
    rng=np.random.default_rng(seed)
    results=[]
    grouper=group_columns[0] if len(group_columns)==1 else list(group_columns)
    for keys,group in data.groupby(grouper,dropna=False):
        keys=keys if isinstance(keys,tuple) else (keys,)
        document_groups={doc:doc_frame for doc,doc_frame in group.groupby('document_title')}
        estimates={metric:[] for metric in metrics}
        for _ in range(n_bootstrap):
            pieces=[]
            for document,doc_frame in document_groups.items():
                indices=rng.integers(0,len(doc_frame),size=len(doc_frame))
                pieces.append(doc_frame.iloc[indices])
            sample=pd.concat(pieces,ignore_index=True)
            record=binary_metric_record(sample,'gold_common_int')
            for metric in metrics:
                estimates[metric].append(record.get(metric,np.nan))
        point=binary_metric_record(group,'gold_common_int')
        for metric,values in estimates.items():
            values=np.asarray(values,dtype=float); values=values[~np.isnan(values)]
            results.append({
                **dict(zip(group_columns,keys)),'metric':metric,
                'point_estimate':point.get(metric,np.nan),'n_bootstrap':len(values),
                'ci_2_5':np.quantile(values,0.025) if len(values) else np.nan,
                'ci_97_5':np.quantile(values,0.975) if len(values) else np.nan,
            })
    return pd.DataFrame(results)

STABILITY_BOOTSTRAP_CI = document_stratified_bootstrap(STABILITY_SCORED)
display(STABILITY_BOOTSTRAP_CI)

In [ ]:
def final_review_queue() -> pd.DataFrame:
    if TIERED_REVIEW_BY_SEED.empty:
        return pd.DataFrame()
    tier_summary=TIER_STABILITY.copy()
    per_row=(
        TIERED_REVIEW_BY_SEED.groupby('row_id')
        .agg(
            seeds=('seed','nunique'),
            any_tier1=('tier',lambda x:any(v=='Tier 1' for v in x)),
            any_tier2=('tier',lambda x:any(v=='Tier 2' for v in x)),
            any_tier3=('tier',lambda x:any(v=='Tier 3' for v in x)),
            any_incomplete=('tier',lambda x:any(v=='INCOMPLETE' for v in x)),
            evidence_invalid_any=('any_evidence_invalid','max'),
            mean_confidence=('mean_confidence','mean'),
        ).reset_index()
    )
    provider_instability=(
        PROVIDER_ROW_STABILITY.groupby('row_id')
        .agg(
            unstable_providers=('changed_across_repeats','sum'),
            minimum_repeat_agreement=('repeat_agreement_share','min'),
        ).reset_index()
        if not PROVIDER_ROW_STABILITY.empty else pd.DataFrame({'row_id':BENCHMARK['row_id']})
    )
    queue=tier_summary.merge(per_row,on='row_id',how='left').merge(provider_instability,on='row_id',how='left')
    queue['review_priority_score']=(
        (~queue['tier_stable']).astype(int)*4
        + queue['tier_range'].fillna(0).clip(upper=4)
        + queue['any_tier2'].astype(int)*2
        + queue['any_tier3'].astype(int)*3
        + queue['evidence_invalid_any'].fillna(False).astype(int)*4
        + queue['unstable_providers'].fillna(0)
    )
    return queue.sort_values(
        ['review_priority_score','tier_range','minimum_repeat_agreement'],
        ascending=[False,False,True]
    )

FINAL_REVIEW_QUEUE = final_review_queue()
display(FINAL_REVIEW_QUEUE.head(50))

## 22. Definition-impact diagnostics with special attention to EPA

In [ ]:
def definition_impact_table(phase1_scored: pd.DataFrame) -> pd.DataFrame:
    if phase1_scored.empty:
        return pd.DataFrame()
    subset=phase1_scored[
        phase1_scored['definition_id'].isin(['D1_old_broad','D2_current_strict','D3_provisional_adaptive'])
    ].copy()
    rows=[]
    for (provider,prompt_style,document),group in subset.groupby(['provider','prompt_style','document_title']):
        pivot=group.pivot_table(
            index=['row_id','seed'],columns='definition_id',values='pred_int',aggfunc='first'
        )
        for left,right in [('D1_old_broad','D2_current_strict'),('D2_current_strict','D3_provisional_adaptive'),('D1_old_broad','D3_provisional_adaptive')]:
            if left not in pivot or right not in pivot:
                continue
            paired=pivot[[left,right]].dropna()
            rows.append({
                'provider':provider,'prompt_style':prompt_style,'document_title':document,
                'contrast':f'{left} -> {right}','n_paired':len(paired),
                'no_to_yes':int(((paired[left]==0)&(paired[right]==1)).sum()),
                'yes_to_no':int(((paired[left]==1)&(paired[right]==0)).sum()),
                'unchanged_yes':int(((paired[left]==1)&(paired[right]==1)).sum()),
                'unchanged_no':int(((paired[left]==0)&(paired[right]==0)).sum()),
            })
    return pd.DataFrame(rows)

DEFINITION_IMPACT = definition_impact_table(PHASE1_SCORED)
D3_BOUNDARY_CASES = pd.DataFrame()
if not PHASE1_SCORED.empty:
    d2=PHASE1_SCORED[PHASE1_SCORED['definition_id'].eq('D2_current_strict')][
        ['provider','prompt_style','row_id','seed','pred_int','out__change_type','out__unlearning_mode']
    ].rename(columns={
        'pred_int':'d2_pred','out__change_type':'d2_change_type','out__unlearning_mode':'d2_mode'
    })
    d3=PHASE1_SCORED[PHASE1_SCORED['definition_id'].eq('D3_provisional_adaptive')][
        ['provider','prompt_style','row_id','seed','pred_int','out__change_type','out__unlearning_mode','evidence_valid']
    ].rename(columns={
        'pred_int':'d3_pred','out__change_type':'d3_change_type','out__unlearning_mode':'d3_mode'
    })
    D3_BOUNDARY_CASES=d2.merge(d3,on=['provider','prompt_style','row_id','seed'],how='inner')
    D3_BOUNDARY_CASES=D3_BOUNDARY_CASES[
        D3_BOUNDARY_CASES['d2_pred'].ne(D3_BOUNDARY_CASES['d3_pred'])
    ].merge(
        BENCHMARK[['row_id','document_title','target_text','gold_historical']],
        on='row_id',how='left'
    )
    D3_BOUNDARY_CASES['is_epa']=D3_BOUNDARY_CASES['document_title'].str.contains('EPA',case=False,na=False)
    display(DEFINITION_IMPACT)
    display(D3_BOUNDARY_CASES.sort_values(['is_epa','row_id'],ascending=[False,True]))

**Interpretation rule:** D3 is promising only when it converts theoretically plausible durable reconfiguration cases while preserving non-EPA specificity and rejecting capacity-only additions. Higher agreement with legacy labels is not itself evidence that D3 is the correct construct.

## 23. Run completeness, manifests, and final configuration

In [ ]:
def run_completeness_table() -> pd.DataFrame:
    expected_rows=[]
    phase_specs={
        'phase1':PHASE1_TASKS,
        'phase2':PHASE2_TASKS,
        'phase3':PHASE3_W1_TASKS+PHASE3_W2_BINARY_TASKS+PHASE3_W2_TARGET_TASKS,
        'stability':STABILITY_JOINT_TASKS+STABILITY_BINARY_TASKS+STABILITY_TARGET_TASKS,
    }
    for phase,tasks in phase_specs.items():
        for provider in MODEL_CONFIGS:
            latest=provider_results_snapshot(phase_log_paths(phase,provider)['results'])
            successful=latest[latest.get('status',pd.Series(dtype=str)).eq('ok')] if not latest.empty else pd.DataFrame()
            for stage in sorted(set(task.stage for task in tasks) or {'none'}):
                expected=sum(task.stage==stage for task in tasks)
                observed=int(successful.get('stage',pd.Series(dtype=str)).eq(stage).sum()) if not successful.empty else 0
                # Target-stage expected is a ceiling because negative Stage 1 rows are intentionally skipped.
                expected_kind='ceiling' if stage=='target' else 'exact'
                expected_rows.append({
                    'phase':phase,'provider':provider,'stage':stage,
                    'expected_tasks':expected,'expected_kind':expected_kind,
                    'successful_records':observed,
                    'complete':observed==expected if expected_kind=='exact' else observed<=expected,
                })
    return pd.DataFrame(expected_rows)

RUN_COMPLETENESS = run_completeness_table()
display(RUN_COMPLETENESS)

In [ ]:
FINAL_CONDITION = STABILITY_CONDITIONS[0] if STABILITY_CONDITIONS else PHASE3_SELECTED_CONDITION
FINAL_CONFIGURATION = {
    'protocol_version':PROTOCOL_VERSION,
    'adapter_revision':ADAPTER_REVISION,
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'dataset_sha256':DATASET_SHA256,
    'benchmark_rows':None if BENCHMARK is None else len(BENCHMARK),
    'gold_common_final_basis':common_final_gold_column()[1],
    'definition_is_provisional':bool(FINAL_CONDITION and FINAL_CONDITION.definition_id=='D3_provisional_adaptive'),
    'condition':None if FINAL_CONDITION is None else asdict(FINAL_CONDITION),
    'models':{provider:asdict(config) for provider,config in MODEL_CONFIGS.items()},
    'temperature_requested':{p:c.temperature_requested for p,c in MODEL_CONFIGS.items()},
    'temperature_sent':{p:c.temperature_sent for p,c in MODEL_CONFIGS.items()},
    'provider_temperature_transport':{p:c.temperature_transport for p,c in MODEL_CONFIGS.items()},
    'provider_native_seed_supported':{p:c.native_seed_supported for p,c in MODEL_CONFIGS.items()},
    'provider_native_seed_enabled':{p:c.native_seed_enabled for p,c in MODEL_CONFIGS.items()},
    'provider_native_seed_transport':{p:c.native_seed_transport for p,c in MODEL_CONFIGS.items()},
    'reproducibility_interpretation':(
        'OpenAI and Claude experiment seeds are local scheduling/repeat identifiers, not '
        'provider decoding seeds. Gemini receives its registered native decoding seed. '
        'All providers are evaluated through repeated identical calls and complete provenance.'
    ),
    'stability_seeds':STABILITY_SEEDS,
    'one_target_paragraph_per_request':True,
    'few_shot_examples_used':False,
    'structured_evidence_only':True,
    'tiered_review_policy':{
        'Tier 1':'3 of 3 providers Yes',
        'Tier 2':'2 of 3 providers Yes',
        'Tier 3':'1 of 3 providers Yes',
        'Tier 4':'0 of 3 providers Yes',
    },
    'selection_constraints':{
        'minimum_schema_valid_rate':MIN_SCHEMA_VALID_RATE,
        'minimum_evidence_valid_rate':MIN_EVIDENCE_VALID_RATE,
        'specificity_floor':SPECIFICITY_FLOOR,
    },
    'warning':'Do not treat D3 as final until human adjudication is completed.',
}
atomic_write_text(
    CONFIG_ROOT / 'final_configuration.json',
    json.dumps(FINAL_CONFIGURATION,indent=2,default=str)
)
print(json.dumps(FINAL_CONFIGURATION,indent=2,default=str))

## 24. Export reproducible tables and a formatted analysis workbook

In [ ]:
def exportable_tables() -> dict[str,pd.DataFrame]:
    candidates={
        'Config Summary':pd.DataFrame([CONFIG_SUMMARY]),
        'Models':MODEL_PROTOCOL,
        'Definitions':DEFINITION_REGISTRY,
        'Gold Availability':GOLD_AVAILABILITY,
        'Benchmark Audit':BENCHMARK_AUDIT,
        'Context Availability':CONTEXT_AVAILABILITY,
        'Prompt Audit':PROMPT_AUDIT,
        'Phase1 Ranking':PHASE1_RANKING,
        'Phase1 Provider':PHASE1_PROVIDER_METRICS,
        'Phase1 Documents':PHASE1_DOCUMENT_METRICS,
        'Phase1 Aligned':PHASE1_ALIGNED_METRICS,
        'Phase1 Tradeoff':PHASE1_DEFINITION_TRADEOFF,
        'Phase1 EPA':PHASE1_EPA_DIAGNOSTICS,
        'Phase1 McNemar':PHASE1_MCNEMAR,
        'Phase1 GEE':PHASE1_GEE,
        'Definition Impact':DEFINITION_IMPACT,
        'D3 Boundary Cases':D3_BOUNDARY_CASES,
        'Phase2 Ranking':PHASE2_RANKING,
        'Phase2 Provider':PHASE2_PROVIDER_METRICS,
        'Phase2 Documents':PHASE2_DOCUMENT_METRICS,
        'Phase2 McNemar':PHASE2_MCNEMAR,
        'Phase2 GEE':PHASE2_GEE,
        'Phase3 Ranking':PHASE3_RANKING,
        'Phase3 Provider':PHASE3_PROVIDER_METRICS,
        'Phase3 Documents':PHASE3_DOCUMENT_METRICS,
        'Phase3 Targets':PHASE3_TARGET_METRICS,
        'Phase3 McNemar':PHASE3_MCNEMAR,
        'Phase3 GEE':PHASE3_GEE,
        'Stability Runs':STABILITY_RUN_METRICS,
        'Stability Distribution':STABILITY_METRIC_DISTRIBUTION,
        'Provider Row Stability':PROVIDER_ROW_STABILITY,
        'Seed Pairwise':PROVIDER_SEED_PAIRWISE,
        'Tiered Review':TIERED_REVIEW_BY_SEED,
        'Tier Stability':TIER_STABILITY,
        'Reliability':CROSS_PROVIDER_RELIABILITY,
        'Bootstrap CI':STABILITY_BOOTSTRAP_CI,
        'Review Queue':FINAL_REVIEW_QUEUE,
        'Completeness':RUN_COMPLETENESS,
    }
    return {name:frame for name,frame in candidates.items() if isinstance(frame,pd.DataFrame) and not frame.empty}

EXPORT_TABLES=exportable_tables()
TABLES_DIR=REPORTS_ROOT/'tables'
TABLES_DIR.mkdir(parents=True,exist_ok=True)
TABLE_MANIFEST=[]
for name,frame in EXPORT_TABLES.items():
    filename=re.sub(r'[^A-Za-z0-9_-]+','_',name).strip('_').lower()+'.csv'
    path=TABLES_DIR/filename
    frame.to_csv(path,index=False)
    TABLE_MANIFEST.append({
        'table':name,'filename':filename,'rows':len(frame),'columns':len(frame.columns),
        'sha256':sha256_file(path),
    })
TABLE_MANIFEST_DF=pd.DataFrame(TABLE_MANIFEST)
TABLE_MANIFEST_DF.to_csv(REPORTS_ROOT/'table_manifest.csv',index=False)
display(TABLE_MANIFEST_DF)

In [ ]:
def unique_sheet_name(name: str, used: set[str]) -> str:
    base=re.sub(r'[\\/*?:\[\]]','_',name)[:31] or 'Sheet'
    candidate=base; counter=1
    while candidate in used:
        suffix=f'_{counter}'
        candidate=base[:31-len(suffix)]+suffix
        counter+=1
    used.add(candidate)
    return candidate

def format_excel_worksheet(worksheet) -> None:
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
    worksheet.freeze_panes='A2'
    worksheet.auto_filter.ref=worksheet.dimensions
    header_fill=PatternFill('solid',fgColor='1F4E78')
    header_font=Font(color='FFFFFF',bold=True)
    for cell in worksheet[1]:
        cell.fill=header_fill; cell.font=header_font
        cell.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
    for column_cells in worksheet.columns:
        letter=get_column_letter(column_cells[0].column)
        max_length=max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells[:200])
        worksheet.column_dimensions[letter].width=min(max(max_length+2,10),60)
    for row in worksheet.iter_rows(min_row=2):
        for cell in row:
            cell.alignment=Alignment(vertical='top',wrap_text=True)

REPORT_WORKBOOK=REPORTS_ROOT/'prelangchain_ab_results.xlsx'
if EXPORT_TABLES:
    with pd.ExcelWriter(REPORT_WORKBOOK,engine='openpyxl') as writer:
        used=set()
        for name,frame in EXPORT_TABLES.items():
            sheet=unique_sheet_name(name,used)
            frame.to_excel(writer,sheet_name=sheet,index=False)
        for worksheet in writer.book.worksheets:
            format_excel_worksheet(worksheet)
    print('Saved:',REPORT_WORKBOOK)
else:
    print('No result tables available; workbook not written.')

In [ ]:
def artifact_manifest(root: Path) -> pd.DataFrame:
    rows=[]
    for path in sorted(root.rglob('*')):
        if path.is_file():
            rows.append({
                'relative_path':str(path.relative_to(root)),
                'size_bytes':path.stat().st_size,
                'modified_utc':datetime.fromtimestamp(path.stat().st_mtime,tz=timezone.utc).isoformat(),
                'sha256':sha256_file(path),
            })
    return pd.DataFrame(rows)

ARTIFACT_MANIFEST=artifact_manifest(PROJECT_ROOT)
ARTIFACT_MANIFEST.to_csv(REPORTS_ROOT/'artifact_manifest.csv',index=False)
print('Manifested files:',len(ARTIFACT_MANIFEST))

## 25. Protocol self-tests

In [ ]:
def protocol_self_tests() -> pd.DataFrame:
    checks=[]

    def add(name: str, passed: bool, observed: Any, requirement: str, severity: str='error'):
        checks.append({
            'check':name,'passed':bool(passed),'observed':observed,
            'requirement':requirement,'severity':severity
        })

    add('all requested model temperatures are zero',
        all(c.temperature_requested == 0.0 for c in MODEL_CONFIGS.values()),
        [c.temperature_requested for c in MODEL_CONFIGS.values()], 'all 0.0')
    add('provider temperature transport matches capability contract',
        MODEL_CONFIGS['openai'].temperature_sent is None
        and MODEL_CONFIGS['anthropic'].temperature_sent == 0.0
        and MODEL_CONFIGS['gemini'].temperature_sent == 0.0,
        {p:c.temperature_sent for p,c in MODEL_CONFIGS.items()},
        "{'openai': None, 'anthropic': 0.0, 'gemini': 0.0}")
    add('native decoding seed enabled only for Gemini',
        not MODEL_CONFIGS['openai'].native_seed_enabled
        and not MODEL_CONFIGS['anthropic'].native_seed_enabled
        and MODEL_CONFIGS['gemini'].native_seed_enabled == ENABLE_GEMINI_NATIVE_SEED,
        {p:c.native_seed_enabled for p,c in MODEL_CONFIGS.items()},
        'OpenAI=False, Anthropic=False, Gemini=registered setting')
    add('three distinct providers', set(MODEL_CONFIGS) == {'openai','anthropic','gemini'},
        sorted(MODEL_CONFIGS), 'openai, anthropic, gemini')
    add('no rationale schema fields', not SCHEMA_AUDIT['has_rationale_field'].any(),
        SCHEMA_AUDIT['has_rationale_field'].sum(), '0')
    add('no few-shot condition',
        not any('example' in c.condition_id.casefold() or 'example' in c.description.casefold() for c in PHASE1_CONDITIONS),
        [c.condition_id for c in PHASE1_CONDITIONS], 'none')
    add('phase1 condition IDs unique', PHASE1_REGISTRY['condition_id'].is_unique,
        PHASE1_REGISTRY['condition_id'].nunique(), len(PHASE1_REGISTRY))
    add('target appears once in audited prompts',
        PROMPT_AUDIT.empty or PROMPT_AUDIT['target_occurrences_normalized'].eq(1).all(),
        None if PROMPT_AUDIT.empty else PROMPT_AUDIT['target_occurrences_normalized'].tolist(), 'all 1')
    add('no gold leakage in audited prompts',
        PROMPT_AUDIT.empty or not PROMPT_AUDIT['contains_forbidden_term'].any(),
        None if PROMPT_AUDIT.empty else PROMPT_AUDIT['contains_forbidden_term'].sum(), '0')
    add('dataset hash recorded', bool(DATASET_SHA256), DATASET_SHA256, 'nonempty SHA-256')
    add('definition hashes unique', DEFINITION_REGISTRY['sha256'].nunique() == len(DEFINITION_REGISTRY),
        DEFINITION_REGISTRY['sha256'].nunique(), len(DEFINITION_REGISTRY))
    add('provider directories isolated',
        len({str(provider_phase_directory('phase1',p)) for p in MODEL_CONFIGS}) == 3,
        [str(provider_phase_directory('phase1',p)) for p in MODEL_CONFIGS], '3 unique paths')
    add('all live adapters use current revision',
        all(getattr(fn, '__adapter_revision__', None) == ADAPTER_REVISION for fn in PROVIDER_CALLS.values()),
        {p:getattr(fn, '__adapter_revision__', None) for p,fn in PROVIDER_CALLS.items()},
        ADAPTER_REVISION)
    add('pure parameter preflight passed', PROVIDER_PARAMETER_PREFLIGHT['passed'].all(),
        PROVIDER_PARAMETER_PREFLIGHT[['provider','passed']].to_dict('records'), 'all passed')

    fixture = canonicalize_benchmark(synthetic_benchmark()).iloc[2]
    condition = next(c for c in PHASE1_CONDITIONS if c.condition_id == 'P3_D3_ADAPTIVE')
    package = build_prompt_package(fixture, condition, stage='joint')
    openai_parameters = build_openai_request_parameters_v12(
        MODEL_CONFIGS['openai'], package, DISCOVERY_SEED
    )
    anthropic_parameters = build_anthropic_request_parameters_v12(
        MODEL_CONFIGS['anthropic'], package, DISCOVERY_SEED
    )
    gemini_manifest = build_gemini_request_manifest_v12(
        MODEL_CONFIGS['gemini'], package, DISCOVERY_SEED
    )

    add('OpenAI cache key <= 64 characters',
        len(openai_parameters['prompt_cache_key']) <= OPENAI_PROMPT_CACHE_KEY_MAX_CHARS,
        len(openai_parameters['prompt_cache_key']), f'<={OPENAI_PROMPT_CACHE_KEY_MAX_CHARS}')
    add('OpenAI request omits unsupported temperature',
        'temperature' not in openai_parameters, sorted(openai_parameters), 'temperature absent')
    add('OpenAI request omits unsupported provider seed',
        'seed' not in openai_parameters, sorted(openai_parameters), 'seed absent')
    add('OpenAI request uses only allowlisted fields',
        not (set(openai_parameters) - OPENAI_PARSE_ALLOWED_PARAMETERS),
        sorted(openai_parameters), sorted(OPENAI_PARSE_ALLOWED_PARAMETERS))
    injected = {**openai_parameters, 'temperature':0.0, 'seed':DISCOVERY_SEED}
    sanitized, removed = sanitize_openai_request_parameters_v12(injected)
    add('OpenAI runtime firewall strips stale temperature and seed',
        set(removed) == {'temperature','seed'} and 'temperature' not in sanitized and 'seed' not in sanitized,
        {'removed':sorted(removed),'remaining':sorted(sanitized)},
        'temperature and seed removed before network boundary')

    add('Anthropic temperature zero is transmitted',
        anthropic_parameters.get('temperature') == 0.0,
        anthropic_parameters.get('temperature'), 0.0)
    add('Anthropic request omits top-level cache_control, seed, and thinking',
        all(key not in anthropic_parameters for key in ['cache_control','seed','thinking']),
        sorted(anthropic_parameters), 'all absent')
    anthropic_system = anthropic_parameters.get('system', [])
    block_cache = bool(
        isinstance(anthropic_system, list) and anthropic_system and
        anthropic_system[0].get('cache_control') == {'type':'ephemeral'}
    )
    add('Anthropic explicit system-block cache breakpoint present', block_cache,
        anthropic_system[0].get('cache_control') if anthropic_system else None,
        "{'type':'ephemeral'}")

    add('Gemini temperature zero is transmitted',
        gemini_manifest['temperature'] == 0.0,
        gemini_manifest['temperature'], 0.0)
    add('Gemini low thinking level registered',
        gemini_manifest['thinking_level'] == 'low',
        gemini_manifest['thinking_level'], 'low')
    add('Gemini native seed follows experiment seed when enabled',
        (not MODEL_CONFIGS['gemini'].native_seed_enabled and gemini_manifest['seed'] is None)
        or gemini_manifest['seed'] == DISCOVERY_SEED,
        gemini_manifest['seed'], DISCOVERY_SEED if MODEL_CONFIGS['gemini'].native_seed_enabled else None)
    add('Gemini structured JSON contract registered',
        gemini_manifest['response_mime_type'] == 'application/json'
        and gemini_manifest['response_json_schema_sha256'] == package.schema_sha256,
        gemini_manifest, 'application/json and matching schema hash')

    add('unexpected keyword TypeError classified as configuration',
        error_classification(TypeError("unexpected keyword argument 'cache_control'")) == 'configuration',
        error_classification(TypeError("unexpected keyword argument 'cache_control'")), 'configuration')
    add('quota-exhausted error is non-transient',
        error_classification(RuntimeError('429 RESOURCE_EXHAUSTED quota exceeded')) == 'quota',
        error_classification(RuntimeError('429 RESOURCE_EXHAUSTED quota exceeded')), 'quota')
    add('configuration error trips circuit breaker immediately',
        provider_circuit_breaker_reason('configuration', 1, 1) == 'configuration error',
        provider_circuit_breaker_reason('configuration', 1, 1), 'configuration error')
    add('unknown error trips after identical-error limit',
        provider_circuit_breaker_reason('unknown', MAX_IDENTICAL_ERROR_FINGERPRINTS, 1) is not None,
        provider_circuit_breaker_reason('unknown', MAX_IDENTICAL_ERROR_FINGERPRINTS, 1), 'nonempty reason')
    boundary = parse_context_blocks('[PREVIOUS 1]\nBefore.\n[TARGET]\nTarget.\n[NEXT 1]\nAfter.')
    add('boundary context without ±2 remains structurally valid',
        boundary['target_marker_count'] == 1 and boundary['duplicate_marker_count'] == 0,
        boundary, 'one TARGET and no duplicate markers')

    if USE_MOCK_PROVIDER or RUN_API_CALLS:
        for provider in ENABLED_PROVIDERS:
            current, reason = smoke_status_is_current(provider)
            add(f'{provider} smoke signature is current', current, reason, 'current smoke test passed')

    if not BENCHMARK_AUDIT.empty:
        fatal_audit_failures = BENCHMARK_AUDIT[
            ~BENCHMARK_AUDIT['passed'] & BENCHMARK_AUDIT['severity'].eq('error')
        ]
        add('benchmark has no fatal integrity failures', fatal_audit_failures.empty,
            fatal_audit_failures['check'].tolist(), 'none')

    if USE_MOCK_PROVIDER and not PHASE1_SCORED.empty:
        duplicates = PHASE1_SCORED.groupby(
            ['provider','condition_id','row_id','seed']
        )['pred_int'].nunique().max()
        add('mock deterministic within run key', duplicates <= 1, duplicates,
            '<=1 unique prediction', severity='error')

    if not STABILITY_SCORED.empty:
        add('registered stability seeds observed',
            set(STABILITY_SCORED['seed']) == set(STABILITY_SEEDS),
            sorted(STABILITY_SCORED['seed'].unique()), sorted(STABILITY_SEEDS))
        add('all three enabled providers in stability',
            set(STABILITY_SCORED['provider']) == set(ENABLED_PROVIDERS),
            sorted(STABILITY_SCORED['provider'].unique()), sorted(ENABLED_PROVIDERS))

    return pd.DataFrame(checks)


PROTOCOL_SELF_TESTS = protocol_self_tests()
display(PROTOCOL_SELF_TESTS)
failed_errors = PROTOCOL_SELF_TESTS[
    ~PROTOCOL_SELF_TESTS['passed'] & PROTOCOL_SELF_TESTS['severity'].eq('error')
]
if not failed_errors.empty:
    raise AssertionError('Protocol self-tests failed:\n' + failed_errors.to_string(index=False))

In [ ]:
EXECUTION_SUMMARY={
    'protocol_version':PROTOCOL_VERSION,
    'adapter_revision':ADAPTER_REVISION,
    'completed_at_utc':datetime.now(timezone.utc).isoformat(),
    'synthetic_fixture':USE_SYNTHETIC_DATA,
    'mock_provider':USE_MOCK_PROVIDER,
    'paid_api_calls_enabled':RUN_API_CALLS,
    'dataset_sha256':DATASET_SHA256,
    'phase1_unified_predictions':len(PHASE1_SCORED),
    'phase2_unified_predictions':len(PHASE2_SCORED),
    'phase3_unified_predictions':len(PHASE3_SCORED),
    'stability_unified_predictions':len(STABILITY_SCORED),
    'phase1_selection':PHASE1_SELECTION,
    'phase2_selection':PHASE2_SELECTION,
    'phase3_selection':PHASE3_SELECTION,
    'final_configuration_path':str(CONFIG_ROOT/'final_configuration.json'),
    'report_workbook':str(REPORT_WORKBOOK) if REPORT_WORKBOOK.exists() else None,
    'protocol_self_tests_passed':bool(PROTOCOL_SELF_TESTS['passed'].all()),
}
atomic_write_text(
    REPORTS_ROOT/'execution_summary.json',
    json.dumps(EXECUTION_SUMMARY,indent=2,default=str)
)
print(json.dumps(EXECUTION_SUMMARY,indent=2,default=str))

## 26. How to run the API-hardened protocol on the real benchmark

1. **Restart the kernel.** Do not continue in a kernel that executed v1 or v1.1.
2. Place `Unlearning_Codebook_Local_Context_Test_Set.xlsx` in the project root, `data/`, or `/content/`; alternatively set `UNLEARNING_INPUT_WORKBOOK`.
3. Run once with APIs disabled. Confirm that:
   - the benchmark has no fatal integrity failures;
   - `PROVIDER_PARAMETER_PREFLIGHT` shows `passed=True` for all providers;
   - the OpenAI transport keys contain neither `temperature` nor `seed`.
4. Upgrade the three provider SDKs using the installation cell, restart the kernel again, and rerun setup.
5. Store API keys in environment variables or Colab Secrets. Never paste or print keys.
6. Enable API access and run **only the three isolated smoke-test cells**, one at a time. Each smoke test performs a no-network SDK preflight followed by at most one request.
7. Enable only Phase 1:

```bash
export UNLEARNING_RUN_API_CALLS=true
export UNLEARNING_API_CONFIRMATION=RUN_PRELANGCHAIN_AB_V1_2
export UNLEARNING_ACTIVE_PHASES=phase1
export UNLEARNING_SEEDS=17
export UNLEARNING_ENABLE_GEMINI_NATIVE_SEED=true
```

8. Run OpenAI, Anthropic, and Gemini Phase-1 cells independently. Each provider has its own logs and circuit breaker.
9. Inspect `provider_status.json`, `errors.jsonl`, completeness, evidence validity, and cost before enabling `phase2`, then `phase3`, then `stability` in later clean-kernel runs.
10. Successful run keys are resumable; raw JSONL is append-only. To create an independent notebook-level replication, change `UNLEARNING_REPLICATE_ID` while retaining frozen inputs and registered seeds.
11. Human adjudication should later add complete `Gold Old Definition`, `Gold Current Definition`, and `Gold Final Definition` columns. Existing raw outputs remain available for analyses whose prompts are unchanged.

### Interpreting the seed column

- **OpenAI:** seed is local experiment/repeat metadata; no provider decoding seed is sent.
- **Anthropic:** seed is local experiment/repeat metadata; no provider decoding seed is sent.
- **Gemini:** the registered seed is also sent through `GenerationConfig.seed`.

The primary reproducibility evidence is therefore the combination of fixed inputs, hashes, pinned/stable model IDs, exact transport manifests, raw responses, returned model identifiers, SDK versions, and repeated-run stability. Temperature zero is not treated as a guarantee of identical hosted-model output.

### Recommended replication IDs

```text
r1_discovery
r2_clean_kernel
r3_independent_day
```

Do not pool provider votes and repeated runs as if all observations were independent. Tiered review is calculated within each seed; repeat stability is reported separately.

## 27. Deferred LangChain phase

This notebook intentionally contains no retrieved or within-sheet examples. The later LangChain experiment should reuse the frozen benchmark, definition registry, provider adapters, evidence schemas, logging, evaluation, and tiered-review functions. Only example retrieval and prompt assembly should be added.

For each target, exclude the target row, exact/near-duplicate cluster, evaluation rows, and preferably the entire source document from the example pool. That later experiment becomes a new protocol version rather than an edit to this one.